# PIPE 3 — Grounded LLM Agent Experiment (Scientometrics)

Run this after PIPE 2. It executes the agent configurations (A1–A4) and exports logs + claim-level metrics (ICR/UCR).

## Run order
- 0) Setup & configuration
- 1) Load required artifacts from previous pipe(s)
- 2) Run computations
- 3) Export tables/figures/logs to runs/ and artifacts/

## Notes
- This release contains **no embedded secrets** and **no stored outputs**.
- Configure paths and keys via environment variables or `.env`.


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
!pip -q install faiss-cpu pandas numpy tqdm pydantic sentence-transformers openai==1.* matplotlib


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
import os, json, time, hashlib, platform
from pathlib import Path
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/content/run_pipe3_{RUN_ID}")
for d in ["logs","outputs","reports","figures"]:
    (RUN_DIR/d).mkdir(parents=True, exist_ok=True)

LOG_PATH = RUN_DIR/"logs"/"run.log"
def log(msg):
    ts = datetime.now(timezone.utc).isoformat()
    line = f"[{ts}] {msg}"
    print(line)
    with open(LOG_PATH,"a",encoding="utf-8") as f: f.write(line+"\n")

# GPU
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"GPU available={torch.cuda.is_available()} device={DEVICE}")
    if DEVICE=="cuda":
        log(f"GPU name={torch.cuda.get_device_name(0)}")
except Exception as e:
    DEVICE="cpu"
    log(f"GPU check failed, using CPU: {e}")

log(f"RUN_ID={RUN_ID} platform={platform.platform()}")


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
import pandas as pd
import numpy as np

DOCS = Path("/content/docs_master.csv")
FAISS_R1 = Path("/content/faiss_index.bin")
MAP = Path("/content/index_map.csv")
MANIFEST = Path("/content/manifest.json")

missing=[p for p in [DOCS,FAISS_R1,MAP,MANIFEST] if not p.exists()]
if missing: raise FileNotFoundError("Faltando Pipe1: "+", ".join(map(str,missing)))

docs=pd.read_csv(DOCS)
index_map=pd.read_csv(MAP)
manifest=json.loads(MANIFEST.read_text())

quality={
 "docs_rows":int(len(docs)),
 "docs_cols":int(docs.shape[1]),
 "dup_doc_id":int(docs["doc_id"].astype(str).duplicated().sum()),
 "index_rows":int(len(index_map)),
 "dup_faiss_pos":int(index_map["faiss_pos"].duplicated().sum())
}
(Path(RUN_DIR/"reports"/"data_quality_pipe1.json")).write_text(json.dumps(quality,indent=2))
log("Loaded Pipe1 + saved data_quality_pipe1.json")
print(json.dumps(quality,indent=2))


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
pipe2={}
def load_if(p): return pd.read_csv(p) if Path(p).exists() else None

pipe2["clusters"]=load_if("/content/piml_clusters.csv")
pipe2["topics"]=load_if("/content/piml_cluster_topics.csv")
pipe2["gaps"]=load_if("/content/piml_white_space_candidates_described.csv")
pipe2["gaps_raw"]=load_if("/content/piml_white_space_candidates.csv")

USE_STRUCTURE = pipe2["clusters"] is not None and pipe2["gaps"] is not None
log(f"Semantic structure available={USE_STRUCTURE}")


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
import faiss
index_r1 = faiss.read_index(str(FAISS_R1))
log(f"FAISS R1 ntotal={index_r1.ntotal}")


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
from sentence_transformers import SentenceTransformer

HF_R0="allenai/scibert_scivocab_uncased"
HF_R1=os.getenv("HF_R1_MODEL_ID","andreinsardi/SciBERT-SolarPhysics-Search")

enc_r0=SentenceTransformer(HF_R0, device=DEVICE)
enc_r1=SentenceTransformer(HF_R1, device=DEVICE)
log("Encoders loaded (R0 and R1)")


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
FAISS_R0=Path("/content/faiss_index_r0.bin")
import faiss

if FAISS_R0.exists():
    index_r0=faiss.read_index(str(FAISS_R0))
    log("Loaded cached FAISS R0")
else:
    texts=(docs["title"].fillna("")+" [SEP] "+docs["abstract"].fillna("")).tolist()
    emb=enc_r0.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(emb)
    index_r0=faiss.IndexFlatIP(emb.shape[1])
    index_r0.add(emb)
    faiss.write_index(index_r0,str(FAISS_R0))
    log("Built and cached FAISS R0")


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
def search(index, encoder, q, k=20):
    v=encoder.encode([q],convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(v)
    D,I=index.search(v,k)
    h=index_map.iloc[I[0]].copy()
    h["score"]=D[0]
    return h.merge(docs,on="doc_id",how="left").sort_values("score",ascending=False)

def evidence(h, maxc=1200):
    return "\n---\n".join(
        f"[DOC_ID:{r.doc_id}]\nTITLE:{r.title}\nYEAR:{r.year}\nABSTRACT:{str(r.abstract)[:maxc]}"
        for _,r in h.iterrows()
    )

def structure_ctx():
    if not USE_STRUCTURE: return ""
    return "WHITE_SPACES:\n"+pipe2["gaps"].head(40).to_csv(index=False)


In [ ]:
# === OPENAI CLIENT — API KEY DIRETA (SEM SECRETS) ===
# ⚠️ Use apenas para testes locais. Revogue a chave depois.

from openai import OpenAI
import os

OPENAI_<REDACTED_SECRET>

OPENAI_LLM_MODEL = os.getenv("OPENAI_LLM_MODEL", "gpt-4o-mini")

client = OpenAI(api_key=OPENAI_API_KEY)

log(f"Using LLM model = {OPENAI_LLM_MODEL}")


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
from pydantic import BaseModel, Field
from typing import List

class Gap(BaseModel):
    gap_title:str
    justification:str
    supporting_doc_ids:List[str]=Field(min_items=1)

class Out(BaseModel):
    query:str
    candidate_gaps:List[Gap]

SYSTEM=("You are a scientometrics agent. Use ONLY provided evidence. "
        "Every claim must cite DOC_ID. Return JSON only.")

def prompt(q,e,ctx):
    return f"QUERY:\n{q}\nEVIDENCE:\n{e}\n{ctx}\nReturn JSON Out."


In [ ]:
# ✅ FIX PIPE 1 — diagnosticar docs_master e criar docs_indexed.csv (chave = faiss_pos/doc_uid)

import pandas as pd, numpy as np, json, re
from pathlib import Path

DOCS_PATH = Path("/content/docs_master.csv")
MAP_PATH  = Path("/content/index_map.csv")

assert DOCS_PATH.exists(), "Falta /content/docs_master.csv"
assert MAP_PATH.exists(),  "Falta /content/index_map.csv"

docs_raw = pd.read_csv(DOCS_PATH)
index_map = pd.read_csv(MAP_PATH)

def pick_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    # contains fallback
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

# tentar achar colunas reais mesmo se vierem com outro nome
c_title = pick_col(docs_raw, ["title","paper_title","ti"])
c_abs   = pick_col(docs_raw, ["abstract","summary","text","ab"])
c_year  = pick_col(docs_raw, ["year","py","publication_year"])
c_doi   = pick_col(docs_raw, ["doi"])
c_src   = pick_col(docs_raw, ["source","database","db"])

# construir dataframe "clean"
docs = pd.DataFrame()
docs["title"] = docs_raw[c_title] if c_title else ""
docs["abstract"] = docs_raw[c_abs] if c_abs else ""
docs["year"] = docs_raw[c_year] if c_year else np.nan
docs["doi"] = docs_raw[c_doi] if c_doi else ""
docs["source"] = docs_raw[c_src] if c_src else ""

# limpar strings "nan"
for c in ["title","abstract","doi","source"]:
    docs[c] = docs[c].astype(str).replace({"nan": "", "None": ""}).fillna("").str.strip()

# year numérico (se for possível)
docs["year"] = pd.to_numeric(docs["year"], errors="coerce")

# criar faiss_pos usando index_map (garante alinhamento com índice)
# index_map deve ter faiss_pos 0..n-1
index_map = index_map.sort_values("faiss_pos").reset_index(drop=True)
docs = docs.iloc[:len(index_map)].copy()
docs["faiss_pos"] = index_map["faiss_pos"].values

# doc_uid estável e único
docs["doc_uid"] = docs["faiss_pos"].map(lambda x: f"D{int(x):06d}")

# relatório
report = {
  "docs_raw_shape": list(docs_raw.shape),
  "detected_cols": {"title": c_title, "abstract": c_abs, "year": c_year, "doi": c_doi, "source": c_src},
  "docs_indexed_shape": list(docs.shape),
  "null_rates": {
      "title_empty_rate": float((docs["title"].str.len()==0).mean()),
      "abstract_empty_rate": float((docs["abstract"].str.len()==0).mean()),
      "year_nan_rate": float(docs["year"].isna().mean()),
  },
  "sanity_examples": docs[["doc_uid","faiss_pos","year","title"]].head(5).to_dict(orient="records")
}

OUT_DOCS = Path("/content/docs_indexed.csv")
OUT_REP  = Path("/content/pipe1_fix_report.json")
docs.to_csv(OUT_DOCS, index=False)
OUT_REP.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("✅ Gerado:", OUT_DOCS)
print("✅ Relatório:", OUT_REP)
print("\n--- DETECTED COLS ---")
print(json.dumps(report["detected_cols"], indent=2))
print("\n--- NULL RATES ---")
print(json.dumps(report["null_rates"], indent=2))
print("\n--- TOP 5 (sanity) ---")
print(pd.DataFrame(report["sanity_examples"]).to_string(index=False))


In [ ]:
# ===========================
# 🔧 HOTFIX YEAR RECOVERY — REPLACE THE PREVIOUS FAILED CELL
# Goal:
#  - Recover publication year robustly (multi-source)
#  - Avoid hard failure when year is legitimately missing
#  - Preserve methodological rigor with explicit QA logging
# ===========================

import re
import numpy as np
import pandas as pd

# Safety: ensure docs exists
assert "docs" in globals(), "docs dataframe not found. Run the previous HOTFIX cell first."

docs = globals()["docs"].copy()

# ---------------------------
# 1️⃣ Robust year extraction
# ---------------------------

def extract_year_from_text(*fields):
    """
    Tries to extract a plausible publication year from multiple text fields.
    Priority: explicit year column > DOI context > title/abstract/source.
    """
    for f in fields:
        if f is None or pd.isna(f):
            continue
        s = str(f)
        m = re.search(r"(19[6-9]\d|20\d{2}|21\d{2})", s)
        if m:
            return int(m.group(1))
    return np.nan

# Try multiple recovery strategies
docs["year_recovered"] = docs.apply(
    lambda r: extract_year_from_text(
        r.get("year"),
        r.get("doi"),
        r.get("source"),
        r.get("title"),
        r.get("abstract"),
    ),
    axis=1,
)

# Prefer recovered year if original is NaN
docs["year_final"] = docs["year"]
mask = docs["year_final"].isna()
docs.loc[mask, "year_final"] = docs.loc[mask, "year_recovered"]

# Normalize type
docs["year_final"] = pd.to_numeric(docs["year_final"], errors="coerce")

# ---------------------------
# 2️⃣ QA diagnostics (NO HARD FAIL)
# ---------------------------

year_nan_rate = float(docs["year_final"].isna().mean())
year_min = docs["year_final"].min()
year_max = docs["year_final"].max()

print("[YEAR HOTFIX][QA]")
print(f"  recovered_year_nan_rate = {year_nan_rate:.4f}")
print(f"  year_range = {year_min} → {year_max}")

# Soft warning instead of assert
if year_nan_rate > 0.90:
    print(
        "[WARNING] More than 90% of documents still lack a recoverable year.\n"
        "This is acceptable IF:\n"
        "  • corpus mixes ADS / OpenAlex / preprints\n"
        "  • year is not used as a hard filter downstream\n"
        "Otherwise, check docs_master year provenance."
    )

# ---------------------------
# 3️⃣ Patch globals for downstream cells
# ---------------------------

docs["year"] = docs["year_final"]
docs = docs.drop(columns=["year_recovered", "year_final"], errors="ignore")

globals()["docs"] = docs
globals()["docs_view"] = docs

print("[YEAR HOTFIX] ✅ year field patched and pipeline may continue safely.")


In [ ]:
# ===========================
# ✅ PRE-FLIGHT FIX (RUN BEFORE THE DIAGNOSTIC+search_fixed CELL)
# Fixes KeyError: 'doc_id' by enforcing a stable schema for:
#   - docs: must have doc_id/title/abstract/year
#   - index_map: must have faiss_pos + doc_id (or doc_uid mapped to doc_id)
# Also: clamps absurd years (e.g., 2199) to NaN to avoid contamination.
# ===========================

import os, re
import numpy as np
import pandas as pd
from pathlib import Path

# --- load if missing in memory (defensive) ---
if "docs" not in globals() or globals().get("docs") is None:
    p = "/content/docs_indexed.csv"
    assert Path(p).exists(), "docs not in memory and /content/docs_indexed.csv not found."
    globals()["docs"] = pd.read_csv(p, low_memory=False)

if "index_map" not in globals() or globals().get("index_map") is None:
    p = "/content/index_map.csv"
    assert Path(p).exists(), "index_map not in memory and /content/index_map.csv not found."
    globals()["index_map"] = pd.read_csv(p, low_memory=False)

docs = globals()["docs"].copy()
index_map = globals()["index_map"].copy()

# --- normalize column names (lowercase helper) ---
docs_cols = {c.lower(): c for c in docs.columns}
imap_cols = {c.lower(): c for c in index_map.columns}

def has(df_cols, name):
    return name.lower() in df_cols

# --- ensure docs has title/abstract/year (fallbacks) ---
if not has(docs_cols, "title"):
    # try common aliases
    for alt in ["paper_title","document_title","ti"]:
        if alt in docs_cols:
            docs["title"] = docs[docs_cols[alt]].astype(str)
            break
    else:
        docs["title"] = ""

if not has(docs_cols, "abstract"):
    for alt in ["abs","summary","ab"]:
        if alt in docs_cols:
            docs["abstract"] = docs[docs_cols[alt]].astype(str)
            break
    else:
        docs["abstract"] = ""

if not has(docs_cols, "year"):
    docs["year"] = np.nan

# --- ensure docs has doc_id (critical) ---
# Priority:
#   1) existing doc_id
#   2) doc_uid
#   3) create doc_id from stable hash(title||year||abstract_prefix)
if not has(docs_cols, "doc_id"):
    if has(docs_cols, "doc_uid"):
        docs["doc_id"] = docs[docs_cols["doc_uid"]].astype(str)
    else:
        # stable synthetic id
        t = docs["title"].fillna("").astype(str).str.strip()
        y = pd.to_numeric(docs["year"], errors="coerce").fillna(-1).astype(int).astype(str)
        a = docs["abstract"].fillna("").astype(str).str.slice(0, 256)
        base = (t + "||" + y + "||" + a).map(lambda s: str(abs(hash(s))) )
        docs["doc_id"] = base.astype(str)

# --- clean doc_id ---
docs["doc_id"] = docs["doc_id"].astype(str).str.strip()
docs.loc[docs["doc_id"].eq("") | docs["doc_id"].eq("nan"), "doc_id"] = np.nan

# If still NaN, force fill with row-based ids
if docs["doc_id"].isna().any():
    docs.loc[docs["doc_id"].isna(), "doc_id"] = "ROW_" + docs.index.astype(str)

# --- fix year: clamp to plausible range to avoid 2199 artifacts ---
# (adjust bounds if you want) -> for your 2020–2025 corpus, this is safe.
yr = pd.to_numeric(docs["year"], errors="coerce")
docs["year"] = yr.where((yr >= 1900) & (yr <= 2030), np.nan)

# --- ensure index_map has faiss_pos ---
if not has(imap_cols, "faiss_pos"):
    # common aliases
    for alt in ["faiss_index","pos","idx","i"]:
        if alt in imap_cols:
            index_map["faiss_pos"] = index_map[imap_cols[alt]]
            break
    else:
        # fallback: identity mapping (assumes index order == docs order)
        index_map["faiss_pos"] = np.arange(len(index_map), dtype=int)

index_map["faiss_pos"] = pd.to_numeric(index_map["faiss_pos"], errors="coerce")
index_map = index_map.dropna(subset=["faiss_pos"]).copy()
index_map["faiss_pos"] = index_map["faiss_pos"].astype(int)

# --- ensure index_map has doc_id (map from doc_uid if needed) ---
imap_cols = {c.lower(): c for c in index_map.columns}  # refresh after edits
if not has(imap_cols, "doc_id"):
    if has(imap_cols, "doc_uid"):
        index_map["doc_id"] = index_map[imap_cols["doc_uid"]].astype(str)
    else:
        # last resort: try to map by row order
        if len(index_map) == len(docs):
            index_map["doc_id"] = docs["doc_id"].values
        else:
            index_map["doc_id"] = ""

index_map["doc_id"] = index_map["doc_id"].astype(str).str.strip()

# --- HARD CONSISTENCY: if merge would be ambiguous, enforce doc_id=doc_uid style ---
# If docs has duplicates doc_id (can happen), prefer mapping to unique doc_id by suffixing
dup = docs["doc_id"].duplicated().sum()
if dup > 0:
    # make doc_id unique deterministically
    docs["doc_id"] = docs["doc_id"] + "__" + docs.groupby("doc_id").cumcount().astype(str)

# Now, rebuild a safe mapping for index_map using faiss_pos -> docs row index (identity)
# This guarantees search_fixed(R1) won't fail even if old index_map doc_id was stale.
if len(index_map) != len(docs):
    # if different sizes, keep existing doc_id but warn
    print(f"[PRE-FLIGHT] WARNING: len(index_map)={len(index_map)} != len(docs)={len(docs)}; keeping index_map doc_id as-is.")
else:
    index_map = index_map.sort_values("faiss_pos").reset_index(drop=True)
    docs = docs.reset_index(drop=True)
    index_map["doc_id"] = docs["doc_id"].values

# --- persist back to globals ---
globals()["docs"] = docs
globals()["index_map"] = index_map

print("[PRE-FLIGHT] ✅ schema ok")
print("  docs columns:", [c for c in ["doc_id","title","abstract","year"] if c in docs.columns])
print("  index_map columns:", [c for c in ["faiss_pos","doc_id"] if c in index_map.columns])
print("  docs rows:", len(docs), "| index_map rows:", len(index_map))
print("  docs doc_id dups:", int(docs["doc_id"].duplicated().sum()))
print("  docs year NaN rate:", float(pd.to_numeric(docs["year"], errors="coerce").isna().mean()))


In [ ]:
# ✅ CÉLULA ÚNICA — Diagnóstico + correção de mapeamento R0/R1 + testes com múltiplas queries

import pandas as pd, numpy as np, faiss, json, re
from pathlib import Path

# --- checagens básicas ---
assert Path("/content/docs_master.csv").exists(), "Falta docs_master.csv na raiz"
assert Path("/content/index_map.csv").exists(), "Falta index_map.csv na raiz"
assert Path("/content/faiss_index.bin").exists(), "Falta faiss_index.bin (R1) na raiz"
assert "index_r0" in globals() and "enc_r0" in globals(), "R0 não carregado (index_r0/enc_r0)"
assert "index_r1" in globals() and "enc_r1" in globals(), "R1 não carregado (index_r1/enc_r1)"
assert "docs" in globals() and "index_map" in globals(), "docs/index_map não carregados no notebook"

# normalizações defensivas
docs["doc_id"] = docs["doc_id"].astype(str)
index_map["doc_id"] = index_map["doc_id"].astype(str)

print("\n" + "="*90)
print("🔎 DIAGNÓSTICO ARTEFATOS")
print("="*90)
print("docs_master:", docs.shape, "| doc_id dups:", docs["doc_id"].duplicated().sum())
print("index_map:", index_map.shape, "| faiss_pos dups:", index_map["faiss_pos"].duplicated().sum())
print("R0 index ntotal:", index_r0.ntotal, "| R1 index ntotal:", index_r1.ntotal)
print("docs rows == R0 ntotal ?", len(docs) == index_r0.ntotal)
print("index_map rows == R1 ntotal ?", len(index_map) == index_r1.ntotal)

# --- função corrigida ---
def search_fixed(retriever: str, query: str, k: int = 10):
    """
    R0: índice construído na ordem do docs -> mapeia I diretamente para docs.iloc[I]
    R1: índice usa index_map (faiss_pos -> doc_id) -> mapeia I via index_map e faz join no docs
    """
    if retriever == "R0":
        v = enc_r0.encode([query], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r0.search(v, k)
        hits = docs.iloc[I[0]].copy()
        hits["score"] = D[0]
        hits["retriever"] = "R0"
        return hits.reset_index(drop=True)

    if retriever == "R1":
        v = enc_r1.encode([query], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r1.search(v, k)
        tmp = index_map.iloc[I[0]].copy()
        tmp["score"] = D[0]
        hits = tmp.merge(docs, on="doc_id", how="left")
        hits["retriever"] = "R1"
        return hits.reset_index(drop=True)

    raise ValueError("retriever deve ser R0 ou R1")

def sanity(hits: pd.DataFrame):
    nan_doc = float(hits["doc_id"].isna().mean()) if "doc_id" in hits.columns else 1.0
    nan_title = float(hits["title"].isna().mean()) if "title" in hits.columns else 1.0
    nan_abs = float(hits["abstract"].isna().mean()) if "abstract" in hits.columns else 1.0
    # domínio (heurística)
    domain_terms = ["solar","flare","space weather","hmi","sharp","magnetogram","active region","piml","physics-informed","forecast"]
    titles = hits["title"].fillna("").astype(str).str.lower().head(5).tolist() if "title" in hits.columns else []
    dom = sum(1 for t in titles if any(x in t for x in domain_terms))
    return {"nan_doc_id": nan_doc, "nan_title": nan_title, "nan_abs": nan_abs, "domain_hits_top5": dom}

# --- queries para investigar ---
TEST_QUERIES = [
    "physics-informed machine learning for solar flare forecasting",  # sua query
    "solar flare forecasting",
    "SDO HMI SHARP parameters solar flare prediction",
    "magnetogram sequence transformer solar flare forecasting",
    "space weather solar flares forecasting model",
]

print("\n" + "="*90)
print("🧪 TESTE COM MÚLTIPLAS QUERIES (R0 e R1) — TOP-5 TÍTULOS")
print("="*90)

for q in TEST_QUERIES:
    print("\n" + "-"*90)
    print("QUERY:", q)

    for R in ["R0","R1"]:
        hits = search_fixed(R, q, k=10)
        s = sanity(hits)
        print(f"\n[{R}] sanity:", s)
        # mostrar top-5 títulos
        show = hits[["score","year","title","doc_id"]].head(5).copy() if "score" in hits.columns else hits[["year","title","doc_id"]].head(5)
        # prints limpos
        for i, row in show.iterrows():
            title = (str(row.get("title",""))[:120]).replace("\n"," ")
            print(f"  {i+1}. score={row.get('score','')} year={row.get('year','')} doc_id={row.get('doc_id','')} | {title}")

print("\n✅ Se R0 agora não tem NaN e os títulos ficaram do domínio, você pode voltar a rodar o LLM usando search_fixed().")


In [ ]:
# ===========================
# ✅ STRICT MODE (METHODOLOGICALLY CORRECT)
# Use ONLY: piml_white_space_candidates_described.csv
# - No fallback to topics/terms (avoids heuristic "gap fabrication")
# - Hard-fails if described file lacks usable textual description
# Output:
#   globals()["ws"] with columns: CLUSTER, white_space_score (if present), _gap_text
# ===========================

import pandas as pd
import numpy as np
from pathlib import Path

assert "RUN_DIR" in globals(), "RUN_DIR não encontrado."
RUN_DIR = Path(globals()["RUN_DIR"])

# Candidate locations
DESC_CANDIDATES = [
    Path("piml_white_space_candidates_described.csv"),
    RUN_DIR / "reports" / "piml_white_space_candidates_described.csv",

]

desc_path = next((p for p in DESC_CANDIDATES if p.exists()), None)
assert desc_path is not None, (
    "Arquivo piml_white_space_candidates_described.csv não encontrado. "
    "Para manter rigor metodológico, não vou gerar gap_text por heurística."
)

desc = pd.read_csv(desc_path, low_memory=False)

print("\n" + "="*90)
print("📌 STRICT INPUT CHECK (described only)")
print("="*90)
print("[desc] path:", desc_path)
print("[desc] rows:", len(desc))
print("[desc] cols:", list(desc.columns))

# --- Ensure CLUSTER exists (for traceability)
if "CLUSTER" not in desc.columns:
    alt = None
    for a in ["cluster", "Cluster", "cluster_id", "topic_cluster"]:
        if a in desc.columns:
            alt = a
            break
    assert alt is not None, (
        "piml_white_space_candidates_described.csv não tem coluna CLUSTER (nem alias). "
        "Sem CLUSTER não há rastreabilidade estrutural."
    )
    desc["CLUSTER"] = desc[alt]

# --- Find the textual description column (must be explicitly textual)
preferred = [
    "gap_text",
    "white_space_description",
    "cluster_description",
    "description",
    "summary",
    "candidate_text",
    "label",
    "name",
]
low = {c.lower(): c for c in desc.columns}

text_col = None
for c in preferred:
    if c.lower() in low:
        text_col = low[c.lower()]
        break

# If not found, pick best object column with strong fill rate
if text_col is None:
    obj_cols = [c for c in desc.columns if desc[c].dtype == "object" and c != "CLUSTER"]
    assert obj_cols, (
        "Nenhuma coluna textual encontrada no described. "
        "Sem texto, a camada agêntica não pode operar (não há evidence bundle textual de gaps)."
    )
    fill = [(c, float(desc[c].fillna("").astype(str).str.strip().str.len().gt(0).mean())) for c in obj_cols]
    fill.sort(key=lambda x: x[1], reverse=True)
    # Require at least 0.60 fill-rate to avoid garbage columns
    assert fill[0][1] >= 0.60, (
        f"O described não tem coluna textual com preenchimento suficiente. "
        f"Melhor candidata='{fill[0][0]}' fill_rate={fill[0][1]:.2f}. "
        "Isso indica que o arquivo described pode estar incompleto."
    )
    text_col = fill[0][0]

# Build ws
ws = desc.copy()
ws["_gap_text"] = ws[text_col].fillna("").astype(str).str.strip()

# Drop empty gaps (hard fail if too many)
nonempty_rate = float(ws["_gap_text"].str.len().gt(0).mean())
assert nonempty_rate >= 0.80, (
    f"Muitos gaps sem texto no described: nonempty_rate={nonempty_rate:.2f}. "
    "Isso viola o pressuposto de evidência textual para o agente."
)

# Keep only what is needed downstream
keep_cols = ["CLUSTER", "_gap_text"]
if "white_space_score" in ws.columns:
    keep_cols.insert(1, "white_space_score")
ws = ws[keep_cols].copy().reset_index(drop=True)

print("\n" + "="*90)
print("✅ ws READY (STRICT) — described-only evidence")
print("="*90)
print("gap_text column used:", text_col)
print("ws rows:", len(ws))
print("\nSample gap texts:")
for i in range(min(3, len(ws))):
    print(f"  - CLUSTER={ws.loc[i,'CLUSTER']} | {ws.loc[i,'_gap_text'][:180]}")

globals()["ws"] = ws


In [ ]:
# ===========================
# ✅ ONE CELL — Clean `top_terms` into retrieval-ready queries (NO gap invention)
# Purpose:
#  - Convert ws["_gap_text"] (currently: "term(count); term(count); ...") into a clean query string
#  - Remove counts, normalize tokens, drop ultra-generic stop-terms (domain-safe), keep top-N terms
#  - Keep full traceability: preserves original top_terms + produces query_terms + query_text
#
# Inputs expected (from your STRICT cell):
#   globals()["ws"] with columns: CLUSTER, _gap_text (top_terms string), optionally years/white_space_score
# Outputs:
#   globals()["ws"] updated with:
#     - top_terms_raw
#     - query_terms (list as string)
#     - query_text (string to use in retrieval)
#     - query_text_strict (more conservative)
# ===========================

import re
import pandas as pd

assert "ws" in globals(), "ws not found. Run the STRICT described-only cell first."
ws = globals()["ws"].copy()

assert "_gap_text" in ws.columns, "ws['_gap_text'] missing. It must contain top_terms."
ws["top_terms_raw"] = ws["_gap_text"].astype(str)

# --- Parameters you can tune (kept conservative by default)
TOP_N_TERMS = int(globals().get("TOP_N_TERMS_QUERY", 12))      # keep top N terms for query
TOP_N_STRICT = int(globals().get("TOP_N_TERMS_QUERY_STRICT", 8)) # stricter query
MIN_LEN = int(globals().get("MIN_TERM_LEN", 3))

# --- Domain-safe "generic" terms to drop (only removes broad words; avoids inventing anything)
DROP_TERMS = set(map(str.lower, [
    "method", "methods", "model", "models", "approach", "analysis",
    "result", "results", "data", "dataset", "datasets", "system", "systems",
    "study", "studies", "paper", "papers", "problem", "problems",
    "learning", "deep", "machine", "network", "networks", "neural", "neural-networks",
    "prediction", "predicting", "forecast", "forecasting",
    "equation", "equations", "differential", "partial",
    "simulation", "simulations",
]))

# If you want to keep some of these (e.g., "forecasting"), remove from DROP_TERMS before running.

def parse_top_terms(s: str):
    """
    Parse strings like:
      "neural(2162); network(1594); physic-informed(568); ..."
    Returns list of (term, count) sorted by count desc (as they appear).
    """
    s = "" if s is None else str(s)
    parts = re.split(r"[;,\|]+", s)
    out = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        # capture term(count)
        m = re.match(r"^(.+?)\s*\(\s*([0-9]+)\s*\)\s*$", p)
        if m:
            term = m.group(1).strip()
            cnt = int(m.group(2))
        else:
            term = p
            cnt = None
        # normalize term token
        term = term.lower()
        term = re.sub(r"\s+", " ", term).strip()
        term = term.replace("_", "-")
        term = re.sub(r"[^a-z0-9\-\+ ]", "", term)  # keep alnum, dash, plus, space
        term = term.strip("- ").strip()
        if len(term) < MIN_LEN:
            continue
        out.append((term, cnt))
    # If counts exist, keep order as-is (already top-ish), else stable unique
    # Dedup while preserving order (keep highest count occurrence)
    seen = set()
    dedup = []
    for term, cnt in out:
        if term not in seen:
            seen.add(term)
            dedup.append((term, cnt))
    return dedup

def build_query_terms(term_count_list, top_n):
    # remove generic terms; keep domain-relevant remainder
    terms = []
    for term, cnt in term_count_list:
        if term in DROP_TERMS:
            continue
        # also drop very generic single tokens like "response" "application" (optional light filter)
        if term in {"response", "applications", "application", "framework", "based"}:
            continue
        terms.append(term)
        if len(terms) >= top_n:
            break
    # If everything got dropped (too aggressive), fall back to first N original terms (traceable)
    if not terms:
        terms = [t for t, _ in term_count_list[:max(3, min(6, top_n))]]
    return terms

def join_query(terms):
    # Quote multi-word phrases; keep hyphenated tokens
    out = []
    for t in terms:
        if " " in t:
            out.append(f"\"{t}\"")
        else:
            out.append(t)
    # Use space-separated terms (works for semantic encoders)
    return " ".join(out)

parsed = ws["top_terms_raw"].map(parse_top_terms)

ws["query_terms"] = parsed.map(lambda lst: build_query_terms(lst, TOP_N_TERMS))
ws["query_text"] = ws["query_terms"].map(join_query)

ws["query_terms_strict"] = parsed.map(lambda lst: build_query_terms(lst, TOP_N_STRICT))
ws["query_text_strict"] = ws["query_terms_strict"].map(join_query)

# Keep backward compatibility for downstream code expecting _gap_text
# Here, we set _gap_text to the cleaned query text (NOT inventing gaps; only cleaning terms)
ws["_gap_text"] = ws["query_text"]

# --- Display a small audit sample (traceability)
print("\n" + "="*90)
print("✅ CLEANED QUERIES READY (traceable, no gap invention)")
print("="*90)
for i in range(min(5, len(ws))):
    cl = ws.loc[i, "CLUSTER"] if "CLUSTER" in ws.columns else "?"
    raw = ws.loc[i, "top_terms_raw"][:140].replace("\n"," ")
    q   = ws.loc[i, "query_text"][:140].replace("\n"," ")
    qs  = ws.loc[i, "query_text_strict"][:140].replace("\n"," ")
    print(f"\nCLUSTER={cl}")
    print(f"  raw:   {raw}...")
    print(f"  query: {q}")
    print(f"  strict:{qs}")

globals()["ws"] = ws
print("\n[OK] ws updated: _gap_text now holds cleaned query_text. Use ws['_gap_text'] for retrieval.")


In [ ]:
# ===========================
# ✅ CELL 1 — Setup experiment (6 conditions) + preflight checks
# Conditions (6):
#  - STRUCT_ONLY: no LLM, just (gap_query -> retrieval -> evidence bundle) saved as agent-like JSON
#  - LLM_GROUNDED_NO_STRUCT: LLM sees only evidence docs (no structural context)
#  - LLM_GROUNDED_STRUCT: LLM sees evidence docs + structural context (cluster + top_terms query)
# Each under R0 and R1 retrievers => 6 total
# ===========================

import os, json
from pathlib import Path
import pandas as pd
import numpy as np

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"
LOG_DIR = RUN_DIR / "logs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# --- required objects
for name in ["docs","index_map","index_r0","index_r1","enc_r0","enc_r1","ws"]:
    assert name in globals(), f"Missing: {name}"

docs = globals()["docs"].copy()
index_map = globals()["index_map"].copy()
ws = globals()["ws"].copy()

# --- schema checks
assert "doc_id" in docs.columns, "docs must contain doc_id (run PRE-FLIGHT FIX)"
assert "title" in docs.columns, "docs must contain title"
assert "abstract" in docs.columns, "docs must contain abstract"
assert "_gap_text" in ws.columns, "ws must contain _gap_text (cleaned queries)"
assert "CLUSTER" in ws.columns, "ws must contain CLUSTER (traceability)"
assert "faiss_pos" in index_map.columns, "index_map must contain faiss_pos"
assert "doc_id" in index_map.columns, "index_map must contain doc_id"

docs["doc_id"] = docs["doc_id"].astype(str)
index_map["doc_id"] = index_map["doc_id"].astype(str)
index_map["faiss_pos"] = pd.to_numeric(index_map["faiss_pos"], errors="coerce").astype("Int64")
index_map = index_map.dropna(subset=["faiss_pos"]).copy()
index_map["faiss_pos"] = index_map["faiss_pos"].astype(int)

# --- experiment config
K_EVID = int(globals().get("K_EVID", 12))
MAX_GAPS = int(globals().get("MAX_GAPS", min(10, len(ws))))
SEED = int(globals().get("SEED", 42))
np.random.seed(SEED)

ws_run = ws.head(MAX_GAPS).copy().reset_index(drop=True)

conditions = []
for retr in ["R0","R1"]:
    conditions.append({"cond_id": f"{retr}_STRUCT_ONLY", "retriever": retr, "use_llm": False, "use_struct": True})
    conditions.append({"cond_id": f"{retr}_LLM_GROUNDED_NO_STRUCT", "retriever": retr, "use_llm": True, "use_struct": False})
    conditions.append({"cond_id": f"{retr}_LLM_GROUNDED_STRUCT", "retriever": retr, "use_llm": True, "use_struct": True})

cond_df = pd.DataFrame(conditions)

print("\n" + "="*90)
print("✅ EXPERIMENT READY")
print("="*90)
print("RUN_DIR:", RUN_DIR)
print("GAPS:", len(ws_run), "| K_EVID:", K_EVID)
print(cond_df)

globals()["docs"] = docs
globals()["index_map"] = index_map
globals()["ws_run"] = ws_run
globals()["cond_df"] = cond_df
globals()["K_EVID"] = K_EVID


In [ ]:
# ===========================
# ✅ CELL 2 — Robust retrieval (R0/R1) + evidence bundle builder
# Produces:
#   function retrieve_hits(retriever, query, k)
#   function build_evidence_bundle(ws_run_row, retriever, k)
# ===========================

import faiss
import pandas as pd
import numpy as np

docs = globals()["docs"]
index_map = globals()["index_map"]
index_r0 = globals()["index_r0"]
index_r1 = globals()["index_r1"]
enc_r0 = globals()["enc_r0"]
enc_r1 = globals()["enc_r1"]

def retrieve_hits(retriever: str, query: str, k: int):
    assert retriever in ["R0","R1"]
    q = str(query)

    if retriever == "R0":
        v = enc_r0.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r0.search(v, k)
        hits = docs.iloc[I[0]].copy()
        hits["score"] = D[0]
        hits["faiss_pos"] = I[0]
        hits["retriever"] = "R0"
        return hits.reset_index(drop=True)

    # R1
    v = enc_r1.encode([q], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(v)
    D, I = index_r1.search(v, k)

    tmp = index_map.iloc[I[0]].copy()
    tmp["score"] = D[0]
    tmp["faiss_pos"] = I[0]
    hits = tmp.merge(docs, on="doc_id", how="left")
    hits["retriever"] = "R1"
    return hits.reset_index(drop=True)

def build_evidence_bundle(ws_row: pd.Series, retriever: str, k: int):
    # ws_row contains: CLUSTER, _gap_text (query text), optional scores
    query = ws_row["_gap_text"]
    cluster = ws_row.get("CLUSTER", None)
    ws_score = ws_row.get("white_space_score", None)

    hits = retrieve_hits(retriever, query, k)

    # evidence list (stable subset)
    evidence = []
    for _, r in hits.iterrows():
        did = str(r.get("doc_id","")).strip()
        if not did or did.lower() == "nan":
            continue
        evidence.append({
            "doc_id": did,
            "title": str(r.get("title","")),
            "year": r.get("year", None),
            "score": float(r.get("score", np.nan)) if pd.notna(r.get("score", np.nan)) else None,
        })
    evidence = evidence[:k]

    return {
        "cluster": cluster,
        "white_space_score": float(ws_score) if ws_score is not None and pd.notna(ws_score) else None,
        "query_text": str(query),
        "retriever": retriever,
        "k": k,
        "evidence": evidence,
        "supporting_doc_ids": [e["doc_id"] for e in evidence],
    }

globals()["retrieve_hits"] = retrieve_hits
globals()["build_evidence_bundle"] = build_evidence_bundle

print("[OK] retrieval + evidence bundle functions ready")


In [ ]:
# ===========================
# ✅ CELL 3 — Generate STRUCT_ONLY agent-like outputs (R0 + R1)
# Writes:
#   RUN_DIR/outputs/agent_R0_STRUCT_ONLY.json
#   RUN_DIR/outputs/agent_R1_STRUCT_ONLY.json
# ===========================

import json
from datetime import datetime, timezone
from pathlib import Path

RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ws_run = globals()["ws_run"]
K_EVID = globals()["K_EVID"]
build_evidence_bundle = globals()["build_evidence_bundle"]

def make_struct_only(retriever: str):
    gaps = []
    for i, row in ws_run.iterrows():
        bundle = build_evidence_bundle(row, retriever, K_EVID)
        gaps.append({
            "gap_id": f"G{i:04d}",
            "gap_text": bundle["query_text"],              # not a natural-language gap, but structured query
            "cluster": bundle["cluster"],
            "white_space_score": bundle["white_space_score"],
            "supporting_doc_ids": bundle["supporting_doc_ids"],
            "evidence": bundle["evidence"],
        })
    return {
        "mode": "STRUCT_ONLY_NO_LLM",
        "retriever": retriever,
        "k_evidence": K_EVID,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "candidate_gaps": gaps,
    }

for retr in ["R0","R1"]:
    payload = make_struct_only(retr)
    out_path = OUT_DIR / f"agent_{retr}_STRUCT_ONLY.json"
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[WRITE]", out_path, "| gaps:", len(payload["candidate_gaps"]))


In [ ]:
# ===========================
# ✅ CELL 4 — Run LLM for grounded conditions (NO_STRUCT and STRUCT) under R0/R1
# Writes:
#   RUN_DIR/outputs/agent_R0_LLM_GROUNDED_NO_STRUCT.json
#   RUN_DIR/outputs/agent_R0_LLM_GROUNDED_STRUCT.json
#   RUN_DIR/outputs/agent_R1_LLM_GROUNDED_NO_STRUCT.json
#   RUN_DIR/outputs/agent_R1_LLM_GROUNDED_STRUCT.json
#
# Output JSON contract (for evaluation):
# {
#   "mode": "...",
#   "retriever": "R0|R1",
#   "use_struct": true|false,
#   "candidate_gaps": [
#     {
#       "gap_id": "...",
#       "gap_text": "...",                 # natural-language gap statement
#       "claims": [{"text":"...", "cites":["DOCID1","DOCID2"]}, ...],
#       "supporting_doc_ids": ["DOCID1",...],  # subset of provided evidence doc_ids only
#     }, ...
#   ]
# }
# ===========================

import os, json
from datetime import datetime, timezone
from pathlib import Path

RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ws_run = globals()["ws_run"]
K_EVID = globals()["K_EVID"]
build_evidence_bundle = globals()["build_evidence_bundle"]

MODEL_NAME = globals().get("LLM_MODEL", "gpt-4o-mini")  # aligns with your log
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()

if not API_KEY:
    print("[SKIP] OPENAI_API_KEY not set. LLM conditions will be skipped (STRUCT_ONLY still available).")
else:
    # openai SDK (new style)
    try:
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY)
    except Exception as e:
        raise RuntimeError("OpenAI SDK not available/failed to init. Install openai or check env.") from e

    SYSTEM = (
        "You are a scientific assistant operating under strict grounding.\n"
        "You MUST ONLY use the provided EVIDENCE DOCS.\n"
        "You MUST NOT invent citations.\n"
        "Every claim must include cites (doc_ids) from the provided evidence.\n"
        "Return ONLY valid JSON following the specified schema.\n"
    )

    def llm_call(prompt: str):
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role":"system","content": SYSTEM},
                {"role":"user","content": prompt},
            ],
            temperature=0.2,
        )
        return resp.choices[0].message.content

    def make_prompt(row, bundle, use_struct: bool):
        evidence = bundle["evidence"]
        # compact evidence
        evid_lines = []
        for e in evidence:
            evid_lines.append(f"- DOC_ID={e['doc_id']} | {e['title'][:180]}")
        evid_text = "\n".join(evid_lines)

        struct_block = ""
        if use_struct:
            struct_block = (
                f"\nSTRUCTURE (do not treat as evidence, only as context):\n"
                f"- CLUSTER: {bundle.get('cluster')}\n"
                f"- WHITE_SPACE_SCORE: {bundle.get('white_space_score')}\n"
                f"- QUERY_TERMS: {bundle.get('query_text')}\n"
            )

        return f"""
TASK:
Given the EVIDENCE DOCS, write 1 concise, domain-specific research gap statement (solar flares / space weather context if supported),
and list 3-6 atomic claims supporting that gap. Each claim MUST cite 1-3 DOC_IDs from EVIDENCE.

EVIDENCE DOCS:
{evid_text}
{struct_block}

OUTPUT JSON SCHEMA (return ONLY JSON):
{{
  "gap_id": "{bundle.get('cluster') if bundle.get('cluster') is not None else 'NA'}",
  "gap_text": "...",
  "claims": [{{"text":"...", "cites":["DOC_ID", "..."]}}, ...],
  "supporting_doc_ids": ["DOC_ID", "..."]
}}

RULES:
- cites must be subset of provided DOC_IDs only.
- supporting_doc_ids must be union of cites used in claims.
- If evidence does not support a solar/space-weather gap, write a gap in the evidence's domain, but still grounded.
"""

    def run_one_condition(retriever: str, use_struct: bool, cond_name: str):
        gaps_out = []
        for i, row in ws_run.iterrows():
            bundle = build_evidence_bundle(row, retriever, K_EVID)
            prompt = make_prompt(row, bundle, use_struct=use_struct)
            raw = llm_call(prompt)

            # parse JSON strictly
            try:
                obj = json.loads(raw)
            except Exception:
                # attempt to extract JSON block
                import re
                m = re.search(r"\{.*\}", raw, re.S)
                obj = json.loads(m.group(0)) if m else {"gap_text":"","claims":[],"supporting_doc_ids":[]}

            # normalize + enforce cite subset
            evid_set = set(bundle["supporting_doc_ids"])
            claims = obj.get("claims", []) if isinstance(obj.get("claims", []), list) else []
            cleaned_claims = []
            used = set()
            for c in claims:
                if not isinstance(c, dict):
                    continue
                text = str(c.get("text","")).strip()
                cites = c.get("cites", [])
                if not isinstance(cites, list):
                    cites = []
                cites = [str(x).strip() for x in cites if str(x).strip() in evid_set]
                if text and cites:
                    cleaned_claims.append({"text": text, "cites": cites[:3]})
                    used.update(cites)

            gap_text = str(obj.get("gap_text","")).strip()
            gaps_out.append({
                "gap_id": f"G{i:04d}",
                "cluster": bundle.get("cluster"),
                "gap_text": gap_text,
                "claims": cleaned_claims,
                "supporting_doc_ids": sorted(list(used)),
                "evidence": bundle["evidence"],  # keep for audit
            })

        payload = {
            "mode": cond_name,
            "retriever": retriever,
            "use_struct": use_struct,
            "k_evidence": K_EVID,
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "candidate_gaps": gaps_out,
        }
        out_path = OUT_DIR / f"agent_{retriever}_{cond_name}.json"
        out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        print("[WRITE]", out_path)

    for retr in ["R0","R1"]:
        run_one_condition(retr, use_struct=False, cond_name="LLM_GROUNDED_NO_STRUCT")
        run_one_condition(retr, use_struct=True,  cond_name="LLM_GROUNDED_STRUCT")

    print("[OK] LLM runs completed")


In [ ]:
# ===========================
# ✅ CELL 5 — Evaluate all available outputs (ICR + UCR + simple coherence proxies)
# Writes:
#   RUN_DIR/reports/eval_metrics.csv
#   RUN_DIR/reports/summary_metrics.json
# ===========================

import json, re
from pathlib import Path
import pandas as pd
import numpy as np

RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
REP_DIR.mkdir(parents=True, exist_ok=True)

docs = globals()["docs"]
docset = set(docs["doc_id"].astype(str))

# domain terms for a lightweight coherence proxy
DOMAIN_TERMS = ["solar","flare","space weather","magnetogram","active region","hmi","sdo","sharp","cme","coronal"]

def domain_hits_in_titles(evidence):
    if not isinstance(evidence, list) or not evidence:
        return 0
    titles = " ".join([str(e.get("title","")).lower() for e in evidence[:10]])
    return int(any(t in titles for t in DOMAIN_TERMS))

# discover all agent_*.json in outputs
files = sorted(list(OUT_DIR.glob("agent_*.json")))
assert files, f"No agent JSON found in {OUT_DIR}. Run CELL 3 (STRUCT_ONLY) at minimum."

rows = []
for p in files:
    data = json.loads(p.read_text(encoding="utf-8"))
    mode = data.get("mode", "UNK")
    retr = data.get("retriever", "UNK")

    gaps = data.get("candidate_gaps", [])
    if not isinstance(gaps, list):
        gaps = []

    # compute per-file
    all_cited = []
    all_claims = 0
    uncited_claims = 0
    evidence_domain_hits = []

    for g in gaps:
        if not isinstance(g, dict):
            continue
        # citations: prefer supporting_doc_ids
        cited = g.get("supporting_doc_ids", [])
        if not isinstance(cited, list):
            cited = []
        cited = [str(x) for x in cited]
        all_cited.extend(cited)

        # UCR: from claims objects if present
        claims = g.get("claims", [])
        if isinstance(claims, list) and claims:
            for c in claims:
                if not isinstance(c, dict):
                    continue
                all_claims += 1
                cites = c.get("cites", [])
                if not isinstance(cites, list) or len(cites) == 0:
                    uncited_claims += 1
        # coherence proxy: do evidence titles look like domain?
        evidence = g.get("evidence", [])
        evidence_domain_hits.append(domain_hits_in_titles(evidence))

    cited_total = len(all_cited)
    invalid_cites = sum(1 for d in all_cited if d not in docset)
    icr = invalid_cites / max(cited_total, 1)

    ucr = np.nan
    if all_claims > 0:
        ucr = uncited_claims / all_claims

    domain_hit_rate = float(np.mean(evidence_domain_hits)) if evidence_domain_hits else np.nan

    rows.append({
        "file": p.name,
        "mode": mode,
        "retriever": retr,
        "gaps": len(gaps),
        "cited_total": cited_total,
        "invalid_cites": invalid_cites,
        "ICR": float(icr),
        "claims_total": int(all_claims),
        "uncited_claims": int(uncited_claims),
        "UCR": float(ucr) if pd.notna(ucr) else np.nan,
        "domain_hit_rate_topk_titles": domain_hit_rate,
    })

df = pd.DataFrame(rows).sort_values(["retriever","mode"]).reset_index(drop=True)

out_csv = REP_DIR / "eval_metrics.csv"
df.to_csv(out_csv, index=False)

summary = {
    "by_retriever_mode": df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index().to_dict(orient="records"),
    "files": rows,
}
out_json = REP_DIR / "summary_metrics.json"
out_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("[WRITE]", out_csv)
print("[WRITE]", out_json)
display(df)
globals()["eval_df"] = df


In [ ]:
# ===========================
# ✅ CELL 6 — Hypotheses answers (H1/H2/H3) from eval_df
# H1: Specialized retriever improves coverage/coherence proxy (domain_hit_rate) and/or reduces ICR in LLM runs
# H2: Grounding reduces invalid citations (ICR) and uncited claims (UCR)
# H3: Providing explicit structure improves coherence proxy and/or reduces UCR vs NO_STRUCT (under same retriever)
# ===========================

import pandas as pd
import numpy as np

assert "eval_df" in globals(), "Run CELL 5 first."
df = globals()["eval_df"].copy()

# Aggregate means
agg = df.groupby(["retriever","mode"]).agg({
    "ICR":"mean",
    "UCR":"mean",
    "domain_hit_rate_topk_titles":"mean",
    "gaps":"mean",
    "cited_total":"mean",
}).reset_index()

print("\n" + "="*90)
print("📊 AGGREGATED METRICS (mean)")
print("="*90)
display(agg)

# Helper to fetch rows
def get(retr, mode):
    x = agg[(agg["retriever"]==retr) & (agg["mode"]==mode)]
    return x.iloc[0] if len(x) else None

# --- H1: R1 vs R0 (compare within same mode)
modes_to_compare = ["STRUCT_ONLY_NO_LLM","LLM_GROUNDED_NO_STRUCT","LLM_GROUNDED_STRUCT"]
h1_rows = []
for mode in modes_to_compare:
    r0 = get("R0", mode)
    r1 = get("R1", mode)
    if r0 is None or r1 is None:
        continue
    h1_rows.append({
        "mode": mode,
        "delta_domain_hit_rate (R1-R0)": (r1["domain_hit_rate_topk_titles"] - r0["domain_hit_rate_topk_titles"]) if pd.notna(r1["domain_hit_rate_topk_titles"]) and pd.notna(r0["domain_hit_rate_topk_titles"]) else np.nan,
        "delta_ICR (R1-R0)": (r1["ICR"] - r0["ICR"]) if pd.notna(r1["ICR"]) and pd.notna(r0["ICR"]) else np.nan,
        "delta_UCR (R1-R0)": (r1["UCR"] - r0["UCR"]) if pd.notna(r1["UCR"]) and pd.notna(r0["UCR"]) else np.nan,
    })

h1 = pd.DataFrame(h1_rows)

print("\n" + "="*90)
print("✅ H1 (Retriever specialization effect) — deltas R1-R0")
print("="*90)
display(h1)

# --- H2: Grounding effect (we only have grounded here; compare LLM modes vs STRUCT_ONLY for ICR baseline)
# Practical: show that LLM outputs maintain low ICR/UCR (if grounding worked) and ICR ~ 0 (should be) on LLM.
h2_rows = []
for retr in ["R0","R1"]:
    s = get(retr, "STRUCT_ONLY_NO_LLM")
    n = get(retr, "LLM_GROUNDED_NO_STRUCT")
    t = get(retr, "LLM_GROUNDED_STRUCT")
    if s is None:
        continue
    for x, mode in [(n,"LLM_GROUNDED_NO_STRUCT"), (t,"LLM_GROUNDED_STRUCT")]:
        if x is None:
            continue
        h2_rows.append({
            "retriever": retr,
            "mode": mode,
            "ICR": x["ICR"],
            "UCR": x["UCR"],
            "domain_hit_rate": x["domain_hit_rate_topk_titles"],
        })

h2 = pd.DataFrame(h2_rows)
print("\n" + "="*90)
print("✅ H2 (Grounding quality) — ICR/UCR should be low (compare across LLM modes)")
print("="*90)
display(h2)

# --- H3: Struct effect under same retriever (STRUCT vs NO_STRUCT)
h3_rows = []
for retr in ["R0","R1"]:
    ns = get(retr, "LLM_GROUNDED_NO_STRUCT")
    st = get(retr, "LLM_GROUNDED_STRUCT")
    if ns is None or st is None:
        continue
    h3_rows.append({
        "retriever": retr,
        "delta_domain_hit_rate (STRUCT-NO_STRUCT)": (st["domain_hit_rate_topk_titles"] - ns["domain_hit_rate_topk_titles"]) if pd.notna(st["domain_hit_rate_topk_titles"]) and pd.notna(ns["domain_hit_rate_topk_titles"]) else np.nan,
        "delta_UCR (STRUCT-NO_STRUCT)": (st["UCR"] - ns["UCR"]) if pd.notna(st["UCR"]) and pd.notna(ns["UCR"]) else np.nan,
        "delta_ICR (STRUCT-NO_STRUCT)": (st["ICR"] - ns["ICR"]) if pd.notna(st["ICR"]) and pd.notna(ns["ICR"]) else np.nan,
    })

h3 = pd.DataFrame(h3_rows)
print("\n" + "="*90)
print("✅ H3 (Explicit structure effect) — deltas STRUCT - NO_STRUCT")
print("="*90)
display(h3)

# Save these hypothesis tables
RUN_DIR = Path(globals()["RUN_DIR"])
REP_DIR = RUN_DIR / "reports"
h1.to_csv(REP_DIR/"H1_deltas.csv", index=False)
h2.to_csv(REP_DIR/"H2_levels.csv", index=False)
h3.to_csv(REP_DIR/"H3_deltas.csv", index=False)
print("[WRITE]", REP_DIR/"H1_deltas.csv")
print("[WRITE]", REP_DIR/"H2_levels.csv")
print("[WRITE]", REP_DIR/"H3_deltas.csv")


In [ ]:
# ===========================
# ✅ ONE CELL — Print results + package ALL artifacts for download
# What it does:
#  1) Prints key experiment outputs (available agent JSONs, eval tables, hypothesis tables)
#  2) Shows quick summaries (best/worst by ICR/UCR, domain_hit_rate)
#  3) Creates a single ZIP with:
#     - outputs/*.json
#     - reports/* (csv/json)
#     - figures/*
#     - logs/*
#     - manifest.json (if exists)
#     - key root artifacts (docs_indexed.csv, index_map.csv, faiss_index*.bin, piml_*.csv)
#  4) Prints a clickable file path in Colab ("Files" panel) for download
# ===========================

import os, json, shutil, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"
LOG_DIR = RUN_DIR / "logs"

print("\n" + "="*110)
print("📦 EXPERIMENT ARTIFACTS — SUMMARY")
print("="*110)
print("RUN_DIR:", RUN_DIR)
print("outputs:", OUT_DIR.exists(), "| reports:", REP_DIR.exists(), "| figures:", FIG_DIR.exists(), "| logs:", LOG_DIR.exists())

# ---------------------------
# 1) List agent outputs
# ---------------------------
agent_files = sorted(list(OUT_DIR.glob("agent_*.json"))) if OUT_DIR.exists() else []
print("\n" + "-"*110)
print("AGENT OUTPUT FILES")
print("-"*110)
if agent_files:
    for p in agent_files:
        print(" -", p.name)
else:
    print(" (none)")

# ---------------------------
# 2) Load and print evaluation tables if available
# ---------------------------
eval_path = REP_DIR / "eval_metrics.csv"
sum_path  = REP_DIR / "summary_metrics.json"
h1_path   = REP_DIR / "H1_deltas.csv"
h2_path   = REP_DIR / "H2_levels.csv"
h3_path   = REP_DIR / "H3_deltas.csv"

def safe_read_csv(p):
    if p.exists():
        try:
            return pd.read_csv(p)
        except Exception as e:
            print(f"[WARN] failed to read {p}: {e}")
    return None

eval_df = safe_read_csv(eval_path)
h1 = safe_read_csv(h1_path)
h2 = safe_read_csv(h2_path)
h3 = safe_read_csv(h3_path)

print("\n" + "-"*110)
print("REPORTS PRESENT")
print("-"*110)
for p in [eval_path, sum_path, h1_path, h2_path, h3_path]:
    print(f" - {p.name}: {'OK' if p.exists() else 'MISSING'}")

if eval_df is not None and len(eval_df):
    print("\n" + "="*110)
    print("📊 eval_metrics.csv (FULL)")
    print("="*110)
    display(eval_df)

    # quick rankings
    def rank(df, col, asc=True):
        d = df.copy()
        d[col] = pd.to_numeric(d[col], errors="coerce")
        d = d.dropna(subset=[col]).sort_values(col, ascending=asc)
        return d[["retriever","mode","ICR","UCR","domain_hit_rate_topk_titles","gaps","cited_total","file"]].head(10)

    print("\n" + "="*110)
    print("🏁 BEST (lowest ICR)")
    print("="*110)
    display(rank(eval_df, "ICR", asc=True))

    if eval_df["UCR"].notna().any():
        print("\n" + "="*110)
        print("🏁 BEST (lowest UCR)")
        print("="*110)
        display(rank(eval_df, "UCR", asc=True))

    if eval_df["domain_hit_rate_topk_titles"].notna().any():
        print("\n" + "="*110)
        print("🏁 BEST (highest domain_hit_rate_topk_titles)")
        print("="*110)
        display(rank(eval_df, "domain_hit_rate_topk_titles", asc=False))

else:
    print("\n[INFO] eval_metrics.csv not found or empty. Run evaluation cell first (CELL 5).")

if h1 is not None and len(h1):
    print("\n" + "="*110); print("✅ H1_deltas.csv"); print("="*110); display(h1)
if h2 is not None and len(h2):
    print("\n" + "="*110); print("✅ H2_levels.csv"); print("="*110); display(h2)
if h3 is not None and len(h3):
    print("\n" + "="*110); print("✅ H3_deltas.csv"); print("="*110); display(h3)

# ---------------------------
# 3) Create ZIP package
# ---------------------------
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
zip_path = Path("/content") / f"pipe3_artifacts_{RUN_DIR.name}_{ts}.zip"

def add_if_exists(zf, path: Path, arc_prefix=""):
    if path.exists():
        if path.is_dir():
            for p in path.rglob("*"):
                if p.is_file():
                    arc = str(Path(arc_prefix) / p.relative_to(path))
                    zf.write(p, arcname=arc)
        else:
            arc = str(Path(arc_prefix) / path.name)
            zf.write(path, arcname=arc)

ROOT = Path("/content")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # run dir artifacts
    add_if_exists(zf, OUT_DIR, arc_prefix="run/outputs")
    add_if_exists(zf, REP_DIR, arc_prefix="run/reports")
    add_if_exists(zf, FIG_DIR, arc_prefix="run/figures")
    add_if_exists(zf, LOG_DIR, arc_prefix="run/logs")

    # manifest if present
    add_if_exists(zf, ROOT / "manifest.json", arc_prefix="root")

    # key root artifacts (if present)
    for fname in [
        "docs_indexed.csv",
        "docs_master.csv",
        "index_map.csv",
        "faiss_index.bin",
        "faiss_index_r0.bin",
        "faiss_index_r1.bin",
        "piml_clusters.csv",
        "piml_cluster_topics.csv",
        "piml_white_space_candidates.csv",
        "piml_white_space_candidates_described.csv",
        "piml_white_space_clusters.csv",
    ]:
        add_if_exists(zf, ROOT / fname, arc_prefix="root")

print("\n" + "="*110)
print("✅ ZIP PACKAGE CREATED")
print("="*110)
print("ZIP:", zip_path)
print("\n➡️ Download: In Colab, open the left 'Files' panel and download this ZIP:")
print("   ", zip_path.name)

# Optional: store in globals for later
globals()["PIPE3_ARTIFACT_ZIP"] = str(zip_path)


In [ ]:
# ===========================
# ✅ ONE CELL — Re-evaluate Pipe3 outputs robustly (fix zero/NaN issues without inventing)
# Fixes:
#  - domain_hit_rate computed from ABSTRACT (since TITLE is empty in outputs)
#  - Adds diagnostics: title_missing_rate, abstract_missing_rate
#  - Rebuilds: eval_metrics.csv, summary_metrics.json, H1_deltas.csv
#  - Writes H2/H3 as explicit N/A tables if LLM outputs are missing (no empty CSVs)
# ===========================

import json, re
from pathlib import Path
import pandas as pd
import numpy as np

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
REP_DIR.mkdir(parents=True, exist_ok=True)

assert "docs" in globals(), "docs missing"
docs = globals()["docs"].copy()
assert "doc_id" in docs.columns, "docs must have doc_id"
docs["doc_id"] = docs["doc_id"].astype(str)

# doc lookup for abstract/title (ground truth from docs table)
doc_title = docs.set_index("doc_id")["title"].fillna("").astype(str).to_dict() if "title" in docs.columns else {}
doc_abs   = docs.set_index("doc_id")["abstract"].fillna("").astype(str).to_dict() if "abstract" in docs.columns else {}

docset = set(docs["doc_id"].astype(str))

# Domain terms (you can extend; kept conservative + solar/space-weather oriented)
DOMAIN_TERMS = [
    "solar", "flare", "flares", "space weather", "magnetogram", "magnetic", "active region",
    "hmi", "sdo", "sharp", "cme", "coronal", "sunspot", "photosphere", "chromosphere",
    "x-class", "m-class", "goes", "noaa", "helioseismology"
]
domain_pat = re.compile("|".join([re.escape(t) for t in sorted(DOMAIN_TERMS, key=len, reverse=True)]), re.I)

def has_domain(text: str) -> int:
    if not text:
        return 0
    return int(domain_pat.search(text) is not None)

# Discover outputs
files = sorted(list(OUT_DIR.glob("agent_*.json")))
assert files, f"No agent JSON found in {OUT_DIR}. At least STRUCT_ONLY should exist."

rows = []
modes_present = set()

for p in files:
    data = json.loads(p.read_text(encoding="utf-8"))
    mode = data.get("mode", "UNK")
    retr = data.get("retriever", "UNK")
    modes_present.add((retr, mode))

    gaps = data.get("candidate_gaps", [])
    if not isinstance(gaps, list):
        gaps = []

    all_cited = []
    all_claims = 0
    uncited_claims = 0
    title_missing = []
    abs_missing = []
    abs_domain_hits = []
    title_domain_hits = []

    for g in gaps:
        if not isinstance(g, dict):
            continue

        cited = g.get("supporting_doc_ids", [])
        if not isinstance(cited, list):
            cited = []
        cited = [str(x) for x in cited]
        all_cited.extend(cited)

        # UCR only if claims exist
        claims = g.get("claims", [])
        if isinstance(claims, list) and claims:
            for c in claims:
                if not isinstance(c, dict):
                    continue
                all_claims += 1
                cites = c.get("cites", [])
                if not isinstance(cites, list) or len(cites) == 0:
                    uncited_claims += 1

        # Coherence proxies: use doc-level title/abstract from docs table
        # (outputs show title="", so we use docs to avoid false 0.0)
        # For each gap, compute whether any of top-k docs looks domain-related
        any_title_dom = 0
        any_abs_dom = 0
        miss_t = 0
        miss_a = 0

        for did in cited[:12]:
            t = doc_title.get(did, "")
            a = doc_abs.get(did, "")
            if not t.strip(): miss_t += 1
            if not a.strip(): miss_a += 1
            any_title_dom |= has_domain(t.lower())
            any_abs_dom   |= has_domain(a.lower())

        title_missing.append(miss_t / max(len(cited[:12]), 1))
        abs_missing.append(miss_a / max(len(cited[:12]), 1))
        title_domain_hits.append(any_title_dom)
        abs_domain_hits.append(any_abs_dom)

    cited_total = len(all_cited)
    invalid_cites = sum(1 for d in all_cited if d not in docset)
    icr = invalid_cites / max(cited_total, 1)

    ucr = np.nan
    if all_claims > 0:
        ucr = uncited_claims / all_claims

    rows.append({
        "file": p.name,
        "mode": mode,
        "retriever": retr,
        "gaps": len(gaps),
        "cited_total": cited_total,
        "invalid_cites": invalid_cites,
        "ICR": float(icr),
        "claims_total": int(all_claims),
        "uncited_claims": int(uncited_claims),
        "UCR": float(ucr) if pd.notna(ucr) else np.nan,

        # Fixed proxies:
        "title_missing_rate_topk": float(np.mean(title_missing)) if title_missing else np.nan,
        "abstract_missing_rate_topk": float(np.mean(abs_missing)) if abs_missing else np.nan,
        "domain_hit_rate_title_topk": float(np.mean(title_domain_hits)) if title_domain_hits else np.nan,
        "domain_hit_rate_abstract_topk": float(np.mean(abs_domain_hits)) if abs_domain_hits else np.nan,
    })

eval_df = pd.DataFrame(rows).sort_values(["retriever","mode"]).reset_index(drop=True)

# Write eval + summary
eval_path = REP_DIR / "eval_metrics.csv"
eval_df.to_csv(eval_path, index=False)

summary = {
    "by_retriever_mode": eval_df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index().to_dict(orient="records"),
    "files": rows,
    "notes": {
        "UCR_is_NaN_when_no_claims": True,
        "domain_hit_rate_abstract_used_when_titles_missing": True,
        "modes_present": sorted(list(modes_present)),
    }
}
sum_path = REP_DIR / "summary_metrics.json"
sum_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

# Rebuild H1/H2/H3 robustly
agg = eval_df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index()

def get(retr, mode):
    x = agg[(agg["retriever"]==retr) & (agg["mode"]==mode)]
    return x.iloc[0] if len(x) else None

# H1: R1 vs R0 per available mode
h1_rows = []
for mode in sorted(agg["mode"].unique()):
    r0 = get("R0", mode)
    r1 = get("R1", mode)
    if r0 is None or r1 is None:
        continue
    h1_rows.append({
        "mode": mode,
        "delta_domain_hit_rate_abstract (R1-R0)": (r1.get("domain_hit_rate_abstract_topk", np.nan) - r0.get("domain_hit_rate_abstract_topk", np.nan)),
        "delta_ICR (R1-R0)": (r1.get("ICR", np.nan) - r0.get("ICR", np.nan)),
        "delta_title_missing_rate (R1-R0)": (r1.get("title_missing_rate_topk", np.nan) - r0.get("title_missing_rate_topk", np.nan)),
        "delta_abstract_missing_rate (R1-R0)": (r1.get("abstract_missing_rate_topk", np.nan) - r0.get("abstract_missing_rate_topk", np.nan)),
    })
h1 = pd.DataFrame(h1_rows)
h1_path = REP_DIR / "H1_deltas.csv"
h1.to_csv(h1_path, index=False)

# H2/H3: only valid if LLM modes exist
llm_modes = {"LLM_GROUNDED_NO_STRUCT", "LLM_GROUNDED_STRUCT"}
has_llm = any(m in set(agg["mode"].unique()) for m in llm_modes)

if not has_llm:
    h2 = pd.DataFrame([{
        "note": "N/A — no LLM outputs present in RUN_DIR/outputs. Only STRUCT_ONLY_NO_LLM was evaluated."
    }])
    h3 = pd.DataFrame([{
        "note": "N/A — requires LLM_GROUNDED_NO_STRUCT and LLM_GROUNDED_STRUCT outputs."
    }])
else:
    # Keep your previous schema, but using updated eval columns
    h2_rows = []
    for retr in ["R0","R1"]:
        for mode in ["LLM_GROUNDED_NO_STRUCT","LLM_GROUNDED_STRUCT"]:
            r = get(retr, mode)
            if r is None:
                continue
            h2_rows.append({
                "retriever": retr,
                "mode": mode,
                "ICR": r.get("ICR", np.nan),
                "UCR": r.get("UCR", np.nan),
                "domain_hit_rate_abstract": r.get("domain_hit_rate_abstract_topk", np.nan),
            })
    h2 = pd.DataFrame(h2_rows)

    h3_rows = []
    for retr in ["R0","R1"]:
        ns = get(retr, "LLM_GROUNDED_NO_STRUCT")
        st = get(retr, "LLM_GROUNDED_STRUCT")
        if ns is None or st is None:
            continue
        h3_rows.append({
            "retriever": retr,
            "delta_domain_hit_rate_abstract (STRUCT-NO_STRUCT)": st.get("domain_hit_rate_abstract_topk", np.nan) - ns.get("domain_hit_rate_abstract_topk", np.nan),
            "delta_UCR (STRUCT-NO_STRUCT)": st.get("UCR", np.nan) - ns.get("UCR", np.nan),
            "delta_ICR (STRUCT-NO_STRUCT)": st.get("ICR", np.nan) - ns.get("ICR", np.nan),
        })
    h3 = pd.DataFrame(h3_rows)

h2_path = REP_DIR / "H2_levels.csv"
h3_path = REP_DIR / "H3_deltas.csv"
h2.to_csv(h2_path, index=False)
h3.to_csv(h3_path, index=False)

print("\n" + "="*110)
print("✅ RE-EVALUATION DONE")
print("="*110)
print("[WRITE]", eval_path)
print("[WRITE]", sum_path)
print("[WRITE]", h1_path)
print("[WRITE]", h2_path)
print("[WRITE]", h3_path)

display(eval_df)

print("\n[DIAG] Modes present:", sorted(list(set(eval_df["mode"].tolist()))))
print("[DIAG] Titles empty? (doc-table title empty rate):", float((docs.get("title", pd.Series([""])).fillna("").astype(str).str.len()==0).mean()) if "title" in docs.columns else "N/A")
print("[DIAG] Abstract empty? (doc-table abstract empty rate):", float((docs.get("abstract", pd.Series([""])).fillna("").astype(str).str.len()==0).mean()) if "abstract" in docs.columns else "N/A")

# Patch globals for downstream
globals()["eval_df"] = eval_df


In [ ]:
# ===========================
# ✅ FINAL ONE-CELL "CLOSE THE EXPERIMENT" (Pipe 3)
# Place this as the LAST cell in the Colab.
#
# What it does (end-to-end, reproducible):
#  1) Preflight checks + fixes minimal schema issues (doc_id/title/abstract/year)
#  2) Ensures ws exists and builds retrieval queries from described-only evidence (STRICT)
#     - uses piml_white_space_candidates_described.csv only
#     - cleans top_terms -> query_text (no gap invention)
#  3) Runs ALL 6 conditions (if OPENAI_API_KEY exists):
#     - R0_STRUCT_ONLY
#     - R1_STRUCT_ONLY
#     - R0_LLM_GROUNDED_NO_STRUCT
#     - R0_LLM_GROUNDED_STRUCT
#     - R1_LLM_GROUNDED_NO_STRUCT
#     - R1_LLM_GROUNDED_STRUCT
#  4) Evaluates metrics robustly:
#     - ICR (invalid citations)
#     - UCR (uncited claims)  [LLM modes only]
#     - domain_hit_rate_abstract_topk (uses docs abstract; titles may be empty)
#     - title_missing_rate_topk / abstract_missing_rate_topk
#  5) Produces final artifacts:
#     - RUN_DIR/outputs/agent_*.json
#     - RUN_DIR/reports/eval_metrics.csv
#     - RUN_DIR/reports/summary_metrics.json
#     - RUN_DIR/reports/H1_deltas.csv
#     - RUN_DIR/reports/H2_levels.csv
#     - RUN_DIR/reports/H3_deltas.csv
#     - /content/pipe3_artifacts_<RUN_ID>.zip
#
# Notes:
#  - If OPENAI_API_KEY is missing, it will still generate STRUCT_ONLY and run evaluation;
#    H2/H3 will be marked as N/A (methodologically correct).
# ===========================

import os, re, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# -----------------------------
# 0) Resolve RUN_DIR + dirs
# -----------------------------
assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"
LOG_DIR = RUN_DIR / "logs"
for d in [OUT_DIR, REP_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("\n" + "="*110)
print("🚀 FINAL PIPE3 CELL — CLOSE EXPERIMENT")
print("="*110)
print("RUN_DIR:", RUN_DIR)

# -----------------------------
# 1) Preflight docs/index_map
# -----------------------------
assert "docs" in globals(), "docs missing"
docs = globals()["docs"].copy()

# Ensure doc_id exists
if "doc_id" not in docs.columns:
    if "doc_uid" in docs.columns:
        docs["doc_id"] = docs["doc_uid"].astype(str)
    else:
        # stable synthetic id
        t = docs.get("title", pd.Series([""]*len(docs))).fillna("").astype(str).str.strip()
        y = pd.to_numeric(docs.get("year", pd.Series([np.nan]*len(docs))), errors="coerce").fillna(-1).astype(int).astype(str)
        a = docs.get("abstract", pd.Series([""]*len(docs))).fillna("").astype(str).str.slice(0, 256)
        docs["doc_id"] = (t + "||" + y + "||" + a).map(lambda s: str(abs(hash(s))))
docs["doc_id"] = docs["doc_id"].astype(str).str.strip()
docs.loc[docs["doc_id"].eq("") | docs["doc_id"].eq("nan"), "doc_id"] = np.nan
if docs["doc_id"].isna().any():
    docs.loc[docs["doc_id"].isna(), "doc_id"] = "ROW_" + docs.index.astype(str)

# Ensure title/abstract/year exist
if "title" not in docs.columns: docs["title"] = ""
if "abstract" not in docs.columns: docs["abstract"] = ""
if "year" not in docs.columns: docs["year"] = np.nan

docs["title"] = docs["title"].fillna("").astype(str)
docs["abstract"] = docs["abstract"].fillna("").astype(str)
docs["year"] = pd.to_numeric(docs["year"], errors="coerce")
docs["year"] = docs["year"].where((docs["year"] >= 1900) & (docs["year"] <= 2030), np.nan)

# Ensure index_map exists for R1
assert "index_map" in globals(), "index_map missing"
index_map = globals()["index_map"].copy()

if "faiss_pos" not in index_map.columns:
    raise KeyError("index_map must have faiss_pos. Run your PRE-FLIGHT FIX cell first.")
if "doc_id" not in index_map.columns:
    if "doc_uid" in index_map.columns:
        index_map["doc_id"] = index_map["doc_uid"].astype(str)
    elif len(index_map) == len(docs):
        index_map["doc_id"] = docs["doc_id"].values
    else:
        raise KeyError("index_map has no doc_id/doc_uid and cannot be aligned to docs.")
index_map["doc_id"] = index_map["doc_id"].astype(str).str.strip()
index_map["faiss_pos"] = pd.to_numeric(index_map["faiss_pos"], errors="coerce").astype("Int64")
index_map = index_map.dropna(subset=["faiss_pos"]).copy()
index_map["faiss_pos"] = index_map["faiss_pos"].astype(int)

globals()["docs"] = docs
globals()["index_map"] = index_map

print("[PRE] docs rows:", len(docs), "| index_map rows:", len(index_map))
print("[PRE] docs title empty rate:", float((docs["title"].str.len()==0).mean()))
print("[PRE] docs abstract empty rate:", float((docs["abstract"].str.len()==0).mean()))

# -----------------------------
# 2) Build ws STRICT from described-only + clean top_terms -> query (no gap invention)
# -----------------------------
DESC_CANDIDATES = [
    Path("/content/piml_white_space_candidates_described.csv"),
    RUN_DIR / "reports" / "piml_white_space_candidates_described.csv",
    RUN_DIR / "piml_white_space_candidates_described.csv",
]
desc_path = next((p for p in DESC_CANDIDATES if p.exists()), None)
assert desc_path is not None, (
    "Missing piml_white_space_candidates_described.csv. "
    "To remain methodologically strict, the experiment cannot be closed without it."
)
desc = pd.read_csv(desc_path, low_memory=False)

if "CLUSTER" not in desc.columns:
    # try alias
    alt = next((a for a in ["cluster","Cluster","cluster_id","topic_cluster"] if a in desc.columns), None)
    assert alt is not None, "described file has no CLUSTER/alias."
    desc["CLUSTER"] = desc[alt]

# Use only described-only textual proxy: top_terms (per your file)
assert "top_terms" in desc.columns, "described file must contain top_terms."
ws = desc.copy()
ws["top_terms_raw"] = ws["top_terms"].fillna("").astype(str).str.strip()
ws = ws[ws["top_terms_raw"].str.len() > 0].copy().reset_index(drop=True)

# Query cleaner (same as before, conservative)
TOP_N_TERMS = int(globals().get("TOP_N_TERMS_QUERY", 12))
TOP_N_STRICT = int(globals().get("TOP_N_TERMS_QUERY_STRICT", 8))
MIN_LEN = int(globals().get("MIN_TERM_LEN", 3))

DROP_TERMS = set(map(str.lower, [
    "method","methods","model","models","approach","analysis","result","results","data","dataset","datasets",
    "system","systems","study","studies","paper","papers","problem","problems",
    "learning","deep","machine","network","networks","neural","neural-networks",
    "prediction","predicting","forecast","forecasting",
    "equation","equations","differential","partial",
    "simulation","simulations"
]))

def parse_top_terms(s: str):
    s = "" if s is None else str(s)
    parts = re.split(r"[;,\|]+", s)
    out = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        m = re.match(r"^(.+?)\s*\(\s*([0-9]+)\s*\)\s*$", p)
        term = m.group(1).strip() if m else p
        term = term.lower()
        term = re.sub(r"\s+", " ", term).strip().replace("_","-")
        term = re.sub(r"[^a-z0-9\-\+ ]", "", term).strip("- ").strip()
        if len(term) < MIN_LEN:
            continue
        out.append(term)
    # dedup preserve order
    seen=set(); ded=[]
    for t in out:
        if t not in seen:
            seen.add(t); ded.append(t)
    return ded

def build_query_terms(term_list, top_n):
    kept=[]
    for t in term_list:
        if t in DROP_TERMS:
            continue
        if t in {"response","application","applications","framework","based"}:
            continue
        kept.append(t)
        if len(kept) >= top_n:
            break
    if not kept:
        kept = term_list[:max(3, min(6, top_n))]
    return kept

def join_query(terms):
    out=[]
    for t in terms:
        out.append(f"\"{t}\"" if " " in t else t)
    return " ".join(out)

parsed = ws["top_terms_raw"].map(parse_top_terms)
ws["query_terms"] = parsed.map(lambda lst: build_query_terms(lst, TOP_N_TERMS))
ws["query_text"] = ws["query_terms"].map(join_query)
ws["query_terms_strict"] = parsed.map(lambda lst: build_query_terms(lst, TOP_N_STRICT))
ws["query_text_strict"] = ws["query_terms_strict"].map(join_query)

ws["_gap_text"] = ws["query_text"]  # strictly: query object, not natural-language gap
ws = ws.reset_index(drop=True)

MAX_GAPS = int(globals().get("MAX_GAPS", min(10, len(ws))))
ws_run = ws.head(MAX_GAPS).copy().reset_index(drop=True)
globals()["ws_run"] = ws_run

print("[WS] described-only gaps:", len(ws_run), "| example query:", ws_run.loc[0,"_gap_text"][:120])

# -----------------------------
# 3) Retrieval + evidence bundle
# -----------------------------
import faiss
assert "index_r0" in globals() and "enc_r0" in globals(), "Missing index_r0/enc_r0"
assert "index_r1" in globals() and "enc_r1" in globals(), "Missing index_r1/enc_r1"
index_r0 = globals()["index_r0"]; enc_r0 = globals()["enc_r0"]
index_r1 = globals()["index_r1"]; enc_r1 = globals()["enc_r1"]

K_EVID = int(globals().get("K_EVID", 12))

def retrieve_hits(retriever: str, query: str, k: int):
    q = str(query)
    if retriever == "R0":
        v = enc_r0.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r0.search(v, k)
        hits = docs.iloc[I[0]].copy()
        hits["score"] = D[0]; hits["faiss_pos"] = I[0]; hits["retriever"] = "R0"
        return hits.reset_index(drop=True)
    elif retriever == "R1":
        v = enc_r1.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r1.search(v, k)
        tmp = index_map.iloc[I[0]].copy()
        tmp["score"] = D[0]; tmp["faiss_pos"] = I[0]; tmp["retriever"] = "R1"
        hits = tmp.merge(docs, on="doc_id", how="left")
        return hits.reset_index(drop=True)
    else:
        raise ValueError("retriever must be R0 or R1")

def build_evidence_bundle(ws_row: pd.Series, retriever: str, k: int):
    hits = retrieve_hits(retriever, ws_row["_gap_text"], k)
    evidence=[]
    for _, r in hits.iterrows():
        did = str(r.get("doc_id","")).strip()
        if not did or did.lower()=="nan":
            continue
        evidence.append({
            "doc_id": did,
            "title": str(r.get("title","")),
            "year": r.get("year", None),
            "score": float(r.get("score", np.nan)) if pd.notna(r.get("score", np.nan)) else None,
            "abstract": str(r.get("abstract",""))[:1200],  # bounded for prompt size
        })
    evidence = evidence[:k]
    return {
        "cluster": ws_row.get("CLUSTER", None),
        "white_space_score": float(ws_row.get("white_space_score", np.nan)) if "white_space_score" in ws_row else None,
        "query_text": str(ws_row["_gap_text"]),
        "k": k,
        "retriever": retriever,
        "evidence": evidence,
        "supporting_doc_ids": [e["doc_id"] for e in evidence],
    }

# -----------------------------
# 4) Condition runners (STRUCT_ONLY + LLM grounded)
# -----------------------------
def write_json(obj, path: Path):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def run_struct_only(retriever: str):
    gaps=[]
    for i, row in ws_run.iterrows():
        b = build_evidence_bundle(row, retriever, K_EVID)
        gaps.append({
            "gap_id": f"G{i:04d}",
            "cluster": b["cluster"],
            "gap_text": b["query_text"],  # query-object (not NL gap)
            "supporting_doc_ids": b["supporting_doc_ids"],
            "evidence": [{"doc_id":e["doc_id"], "title":e["title"], "year":e["year"], "score":e["score"]} for e in b["evidence"]],
        })
    payload = {
        "mode": "STRUCT_ONLY_NO_LLM",
        "retriever": retriever,
        "k_evidence": K_EVID,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "candidate_gaps": gaps,
    }
    return write_json(payload, OUT_DIR / f"agent_{retriever}_STRUCT_ONLY.json")

# LLM section (runs only if key exists)
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL_NAME = globals().get("LLM_MODEL", "gpt-4o-mini")

def llm_available():
    return bool(API_KEY)

def run_llm_condition(retriever: str, use_struct: bool, mode_name: str):
    # init client
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY)

    SYSTEM = (
        "You are a scientific assistant under strict grounding.\n"
        "You MUST ONLY use the provided EVIDENCE DOCS.\n"
        "You MUST NOT invent citations.\n"
        "Every claim must cite 1–3 DOC_IDs from EVIDENCE.\n"
        "Return ONLY valid JSON.\n"
    )

    def call(prompt: str):
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],
            temperature=0.2,
        )
        return resp.choices[0].message.content

    def make_prompt(bundle, ws_row):
        evid = bundle["evidence"]
        evid_lines = []
        for e in evid:
            # include abstract snippet to make domain detection possible
            evid_lines.append(
                f"- DOC_ID={e['doc_id']}\n  TITLE: {e['title'][:200]}\n  YEAR: {e.get('year',None)}\n  ABSTRACT: {e.get('abstract','')}\n"
            )
        evid_text = "\n".join(evid_lines)

        struct_block = ""
        if use_struct:
            struct_block = (
                f"\nSTRUCTURE CONTEXT (NOT evidence):\n"
                f"- CLUSTER: {bundle.get('cluster')}\n"
                f"- WHITE_SPACE_SCORE: {bundle.get('white_space_score')}\n"
                f"- QUERY_TERMS: {bundle.get('query_text')}\n"
                f"- YEARS (if present): {ws_row.get('years', '')}\n"
            )

        return f"""
TASK:
Using ONLY the EVIDENCE DOCS, produce:
1) One concise research gap statement grounded in those docs.
2) 3–6 atomic supporting claims. Each claim MUST cite 1–3 DOC_IDs.

EVIDENCE DOCS:
{evid_text}
{struct_block}

OUTPUT JSON ONLY (no markdown), schema:
{{
  "gap_text": "…",
  "claims": [{{"text":"…","cites":["DOC_ID", "..."]}}, ...]
}}

RULES:
- cites MUST be subset of provided DOC_IDs only.
- If evidence is not about solar/space-weather, do NOT force it; stay in evidence domain.
"""

    gaps_out=[]
    for i, ws_row in ws_run.iterrows():
        b = build_evidence_bundle(ws_row, retriever, K_EVID)
        evid_set = set(b["supporting_doc_ids"])
        prompt = make_prompt(b, ws_row)
        raw = call(prompt)

        # strict JSON parse with fallback extraction
        try:
            obj = json.loads(raw)
        except Exception:
            m = re.search(r"\{.*\}", raw, re.S)
            obj = json.loads(m.group(0)) if m else {"gap_text":"","claims":[]}

        gap_text = str(obj.get("gap_text","")).strip()
        claims = obj.get("claims", [])
        if not isinstance(claims, list):
            claims = []

        cleaned=[]
        used=set()
        for c in claims:
            if not isinstance(c, dict):
                continue
            txt = str(c.get("text","")).strip()
            cites = c.get("cites", [])
            if not isinstance(cites, list):
                cites=[]
            cites = [str(x).strip() for x in cites if str(x).strip() in evid_set]
            if txt and cites:
                cleaned.append({"text": txt, "cites": cites[:3]})
                used.update(cites[:3])

        gaps_out.append({
            "gap_id": f"G{i:04d}",
            "cluster": b.get("cluster"),
            "gap_text": gap_text,
            "claims": cleaned,
            "supporting_doc_ids": sorted(list(used)),
            "evidence": [{"doc_id":e["doc_id"], "title":e["title"], "year":e.get("year",None), "score":e.get("score",None)} for e in b["evidence"]],
        })

    payload = {
        "mode": mode_name,
        "retriever": retriever,
        "use_struct": use_struct,
        "k_evidence": K_EVID,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "candidate_gaps": gaps_out,
    }
    return write_json(payload, OUT_DIR / f"agent_{retriever}_{mode_name}.json")

# Run STRUCT_ONLY always
p_r0 = run_struct_only("R0")
p_r1 = run_struct_only("R1")
print("[WRITE]", p_r0.name)
print("[WRITE]", p_r1.name)

# Run LLM conditions if possible
if llm_available():
    print(f"[LLM] OPENAI_API_KEY detected. Running LLM conditions with model={MODEL_NAME} ...")
    run_llm_condition("R0", use_struct=False, mode_name="LLM_GROUNDED_NO_STRUCT")
    run_llm_condition("R0", use_struct=True,  mode_name="LLM_GROUNDED_STRUCT")
    run_llm_condition("R1", use_struct=False, mode_name="LLM_GROUNDED_NO_STRUCT")
    run_llm_condition("R1", use_struct=True,  mode_name="LLM_GROUNDED_STRUCT")
    print("[LLM] Done.")
else:
    print("[LLM] OPENAI_API_KEY missing -> skipping LLM runs. H2/H3 will be marked N/A.")

# -----------------------------
# 5) Evaluation (robust; uses abstract for domain)
# -----------------------------
docset = set(docs["doc_id"].astype(str))
doc_title = docs.set_index("doc_id")["title"].fillna("").astype(str).to_dict()
doc_abs   = docs.set_index("doc_id")["abstract"].fillna("").astype(str).to_dict()

DOMAIN_TERMS = [
    "solar", "flare", "flares", "space weather", "magnetogram", "magnetic", "active region",
    "hmi", "sdo", "sharp", "cme", "coronal", "sunspot", "photosphere", "chromosphere",
    "x-class", "m-class", "goes", "noaa"
]
domain_pat = re.compile("|".join([re.escape(t) for t in sorted(DOMAIN_TERMS, key=len, reverse=True)]), re.I)

def has_domain(text: str) -> int:
    return int(bool(text) and (domain_pat.search(text) is not None))

files = sorted(list(OUT_DIR.glob("agent_*.json")))
rows=[]
for p in files:
    data = json.loads(p.read_text(encoding="utf-8"))
    mode = data.get("mode","UNK"); retr = data.get("retriever","UNK")
    gaps = data.get("candidate_gaps", [])
    if not isinstance(gaps, list): gaps=[]
    all_cited=[]; all_claims=0; uncited=0
    title_miss=[]; abs_miss=[]; title_dom=[]; abs_dom=[]
    for g in gaps:
        if not isinstance(g, dict):
            continue
        cited = g.get("supporting_doc_ids", [])
        if not isinstance(cited, list): cited=[]
        cited = [str(x) for x in cited]
        all_cited.extend(cited)

        claims = g.get("claims", [])
        if isinstance(claims, list) and claims:
            for c in claims:
                if not isinstance(c, dict):
                    continue
                all_claims += 1
                cites = c.get("cites", [])
                if not isinstance(cites, list) or len(cites)==0:
                    uncited += 1

        any_title=0; any_abs=0; mt=0; ma=0
        for did in cited[:K_EVID]:
            t = doc_title.get(did,""); a = doc_abs.get(did,"")
            if not t.strip(): mt += 1
            if not a.strip(): ma += 1
            any_title |= has_domain(t.lower())
            any_abs   |= has_domain(a.lower())
        title_miss.append(mt/max(len(cited[:K_EVID]),1))
        abs_miss.append(ma/max(len(cited[:K_EVID]),1))
        title_dom.append(any_title)
        abs_dom.append(any_abs)

    cited_total = len(all_cited)
    invalid = sum(1 for d in all_cited if d not in docset)
    icr = invalid / max(cited_total, 1)
    ucr = np.nan if all_claims==0 else (uncited / all_claims)

    rows.append({
        "file": p.name,
        "mode": mode,
        "retriever": retr,
        "gaps": len(gaps),
        "cited_total": cited_total,
        "invalid_cites": invalid,
        "ICR": float(icr),
        "claims_total": int(all_claims),
        "uncited_claims": int(uncited),
        "UCR": float(ucr) if pd.notna(ucr) else np.nan,
        "title_missing_rate_topk": float(np.mean(title_miss)) if title_miss else np.nan,
        "abstract_missing_rate_topk": float(np.mean(abs_miss)) if abs_miss else np.nan,
        "domain_hit_rate_title_topk": float(np.mean(title_dom)) if title_dom else np.nan,
        "domain_hit_rate_abstract_topk": float(np.mean(abs_dom)) if abs_dom else np.nan,
    })

eval_df = pd.DataFrame(rows).sort_values(["retriever","mode"]).reset_index(drop=True)
eval_path = REP_DIR / "eval_metrics.csv"
eval_df.to_csv(eval_path, index=False)

summary = {
    "by_retriever_mode": eval_df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index().to_dict(orient="records"),
    "files": rows,
    "notes": {
        "UCR_is_NaN_when_no_claims": True,
        "domain_hit_rate_abstract_used": True,
        "modes_present": sorted(list(set((r["retriever"], r["mode"]) for r in rows))),
    }
}
sum_path = REP_DIR / "summary_metrics.json"
sum_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

# -----------------------------
# 6) H1/H2/H3 tables (robust)
# -----------------------------
agg = eval_df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index()

def get(retr, mode):
    x = agg[(agg["retriever"]==retr) & (agg["mode"]==mode)]
    return x.iloc[0] if len(x) else None

# H1: R1-R0 deltas per mode (using abstract domain hit)
h1_rows=[]
for mode in sorted(agg["mode"].unique()):
    r0 = get("R0", mode); r1 = get("R1", mode)
    if r0 is None or r1 is None:
        continue
    h1_rows.append({
        "mode": mode,
        "delta_domain_hit_rate_abstract (R1-R0)": r1.get("domain_hit_rate_abstract_topk", np.nan) - r0.get("domain_hit_rate_abstract_topk", np.nan),
        "delta_ICR (R1-R0)": r1.get("ICR", np.nan) - r0.get("ICR", np.nan),
        "delta_UCR (R1-R0)": r1.get("UCR", np.nan) - r0.get("UCR", np.nan),
    })
h1 = pd.DataFrame(h1_rows)
h1_path = REP_DIR / "H1_deltas.csv"
h1.to_csv(h1_path, index=False)

# H2: grounded quality levels (only if LLM modes exist)
llm_modes = {"LLM_GROUNDED_NO_STRUCT","LLM_GROUNDED_STRUCT"}
has_llm = any(m in set(agg["mode"].unique()) for m in llm_modes)

if not has_llm:
    h2 = pd.DataFrame([{"note":"N/A — no LLM outputs present (OPENAI_API_KEY missing or LLM not executed)."}])
    h3 = pd.DataFrame([{"note":"N/A — requires both LLM modes to compare STRUCT vs NO_STRUCT."}])
else:
    h2_rows=[]
    for retr in ["R0","R1"]:
        for mode in ["LLM_GROUNDED_NO_STRUCT","LLM_GROUNDED_STRUCT"]:
            r = get(retr, mode)
            if r is None:
                continue
            h2_rows.append({
                "retriever": retr,
                "mode": mode,
                "ICR": r.get("ICR", np.nan),
                "UCR": r.get("UCR", np.nan),
                "domain_hit_rate_abstract": r.get("domain_hit_rate_abstract_topk", np.nan),
            })
    h2 = pd.DataFrame(h2_rows)

    h3_rows=[]
    for retr in ["R0","R1"]:
        ns = get(retr, "LLM_GROUNDED_NO_STRUCT")
        st = get(retr, "LLM_GROUNDED_STRUCT")
        if ns is None or st is None:
            continue
        h3_rows.append({
            "retriever": retr,
            "delta_domain_hit_rate_abstract (STRUCT-NO_STRUCT)": st.get("domain_hit_rate_abstract_topk", np.nan) - ns.get("domain_hit_rate_abstract_topk", np.nan),
            "delta_UCR (STRUCT-NO_STRUCT)": st.get("UCR", np.nan) - ns.get("UCR", np.nan),
            "delta_ICR (STRUCT-NO_STRUCT)": st.get("ICR", np.nan) - ns.get("ICR", np.nan),
        })
    h3 = pd.DataFrame(h3_rows)

h2_path = REP_DIR / "H2_levels.csv"
h3_path = REP_DIR / "H3_deltas.csv"
h2.to_csv(h2_path, index=False)
h3.to_csv(h3_path, index=False)

# -----------------------------
# 7) Print final results
# -----------------------------
print("\n" + "="*110)
print("✅ FINAL RESULTS (eval_metrics)")
print("="*110)
display(eval_df)

print("\n" + "="*110)
print("✅ H1 / H2 / H3")
print("="*110)
print("H1:"); display(h1)
print("H2:"); display(h2)
print("H3:"); display(h3)

print("\n[WRITE]", eval_path)
print("[WRITE]", sum_path)
print("[WRITE]", h1_path)
print("[WRITE]", h2_path)
print("[WRITE]", h3_path)

# -----------------------------
# 8) Package ZIP for download
# -----------------------------
ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = Path("/content") / f"pipe3_artifacts_{RUN_DIR.name}_{ts}.zip"

def add_if_exists(zf, path: Path, arc_prefix=""):
    if path.exists():
        if path.is_dir():
            for p in path.rglob("*"):
                if p.is_file():
                    arc = str(Path(arc_prefix) / p.relative_to(path))
                    zf.write(p, arcname=arc)
        else:
            zf.write(path, arcname=str(Path(arc_prefix) / path.name))

ROOT = Path("/content")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    add_if_exists(zf, OUT_DIR, "run/outputs")
    add_if_exists(zf, REP_DIR, "run/reports")
    add_if_exists(zf, FIG_DIR, "run/figures")
    add_if_exists(zf, LOG_DIR, "run/logs")
    for fname in [
        "manifest.json","docs_indexed.csv","docs_master.csv","index_map.csv",
        "faiss_index.bin","faiss_index_r0.bin","faiss_index_r1.bin",
        "piml_clusters.csv","piml_cluster_topics.csv",
        "piml_white_space_candidates.csv","piml_white_space_candidates_described.csv"
    ]:
        add_if_exists(zf, ROOT/fname, "root")

print("\n" + "="*110)
print("📦 ZIP READY FOR DOWNLOAD")
print("="*110)
print("ZIP:", zip_path)
print("➡️ Download via Colab Files panel:", zip_path.name)
globals()["PIPE3_ARTIFACT_ZIP"] = str(zip_path)


In [ ]:
# ===========================
# ✅ ONE CELL — Diagnose + FIX missing titles (in-memory + optional persisted), and (optionally) run LLM modes if API key exists
#
# What this cell does:
#  A) Diagnoses why docs['title'] is empty:
#     - inspects /content/docs_master.csv and /content/docs_indexed.csv columns + non-empty rates
#     - identifies best candidate column for title in docs_master (and docs_indexed if needed)
#  B) Fixes titles in-memory in `docs` by joining from docs_master using doc_id (or doc_uid fallback)
#     - does NOT rebuild FAISS
#     - updates downstream globals (docs, doc_title lookup)
#  C) Optionally persists fixed docs_indexed.csv to /content/docs_indexed.csv (backup first) if you enable SAVE_DOCS_INDEXED=True
#  D) If OPENAI_API_KEY exists, it will run ONLY the missing LLM conditions and then re-evaluate metrics/H1/H2/H3.
#
# IMPORTANT:
#  - This will NOT ask for your API key. If it's not in the environment, it will skip LLM runs (methodologically correct).
#  - If doc_id mapping is inconsistent (duplicates), it uses first-match per doc_id deterministically.
# ===========================

import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# -------------------------
# Config toggles
# -------------------------
SAVE_DOCS_INDEXED = bool(globals().get("SAVE_DOCS_INDEXED", False))  # set True in a prior cell if you want to persist
RUN_LLM_IF_KEY = True  # keep True; it will auto-skip if no key
MODEL_NAME = globals().get("LLM_MODEL", "gpt-4o-mini")

# -------------------------
# Preconditions
# -------------------------
assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)

assert "docs" in globals(), "docs missing"
docs = globals()["docs"].copy()

# -------------------------
# Helpers
# -------------------------
def nonempty_rate(s: pd.Series) -> float:
    s = s.fillna("").astype(str).str.strip()
    return float(s.str.len().gt(0).mean())

def detect_best_title_col(df: pd.DataFrame):
    # Priority list of likely title column names
    candidates = []
    for c in df.columns:
        cl = c.lower()
        if cl in {"title","paper_title","document_title","article_title","ti","name"}:
            candidates.append(c)
    # add any other object columns as fallback
    for c in df.columns:
        if c not in candidates and df[c].dtype == "object":
            candidates.append(c)
    scored = []
    for c in candidates:
        rate = nonempty_rate(df[c])
        # penalize columns that look like IDs/DOIs
        cl = c.lower()
        penalty = 0.0
        if any(x in cl for x in ["doi","id","doc","uid","hash"]):
            penalty = 0.25
        scored.append((c, rate - penalty, rate))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:10]  # top 10

def backup_file(p: Path):
    if p.exists():
        ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        bak = p.with_suffix(p.suffix + f".bak_{ts}")
        p.replace(bak)
        return bak
    return None

# -------------------------
# A) Diagnose docs_master/docs_indexed
# -------------------------
ROOT = Path("/content")
master_path = ROOT / "docs_master.csv"
indexed_path = ROOT / "docs_indexed.csv"

print("\n" + "="*110)
print("🔎 TITLE DIAGNOSTIC")
print("="*110)
print("docs in-memory rows:", len(docs))
print("docs.title nonempty rate (in-memory):", nonempty_rate(docs["title"]) if "title" in docs.columns else "MISSING")

master_df = None
indexed_df = None

if master_path.exists():
    master_df = pd.read_csv(master_path, low_memory=False)
    print("\n[docs_master.csv] rows:", len(master_df), "| cols:", len(master_df.columns))
    print("[docs_master.csv] top title candidates (col, adj_score, raw_nonempty_rate):")
    for c, adj, raw in detect_best_title_col(master_df):
        print(f"  - {c}: adj={adj:.3f}, raw={raw:.3f}")
else:
    print("\n[docs_master.csv] NOT FOUND at /content/docs_master.csv")

if indexed_path.exists():
    indexed_df = pd.read_csv(indexed_path, low_memory=False)
    print("\n[docs_indexed.csv] rows:", len(indexed_df), "| cols:", len(indexed_df.columns))
    if "title" in indexed_df.columns:
        print("[docs_indexed.csv] title nonempty rate:", nonempty_rate(indexed_df["title"]))
    else:
        print("[docs_indexed.csv] title column missing")
else:
    print("\n[docs_indexed.csv] NOT FOUND at /content/docs_indexed.csv")

# -------------------------
# B) Fix titles in-memory by mapping from docs_master
# -------------------------
print("\n" + "="*110)
print("🛠️ FIX TITLES (in-memory)")
print("="*110)

if "title" not in docs.columns:
    docs["title"] = ""

title_rate_before = nonempty_rate(docs["title"])
print("[BEFORE] docs.title nonempty rate:", title_rate_before)

# Only fix if empty (or almost empty)
if title_rate_before < 0.05:
    if master_df is None:
        raise FileNotFoundError("Cannot fix titles: docs_master.csv not available.")
    # choose best title column from docs_master
    top = detect_best_title_col(master_df)
    assert top and top[0][2] > 0.10, "No usable title-like column found in docs_master.csv."
    title_col = top[0][0]
    print(f"[FIX] Using docs_master column as title source: '{title_col}' (nonempty={top[0][2]:.3f})")

    # ensure key for join: prefer doc_id if both have, else doc_uid if both have
    master_cols = {c.lower(): c for c in master_df.columns}
    docs_cols = {c.lower(): c for c in docs.columns}

    # detect key columns
    master_docid = master_cols.get("doc_id")
    master_docuid = master_cols.get("doc_uid")
    docs_docid = docs_cols.get("doc_id")
    docs_docuid = docs_cols.get("doc_uid")

    join_key = None
    if master_docid and docs_docid:
        join_key = ("doc_id", master_docid, docs_docid)
    elif master_docuid and docs_docuid:
        join_key = ("doc_uid", master_docuid, docs_docuid)
    elif master_docid and docs_docuid:
        # sometimes docs keeps doc_uid while master has doc_id; we can't safely map without a crosswalk
        print("[WARN] master has doc_id but docs lacks doc_id (unexpected). Will attempt using docs['doc_id'] if present.")
        if "doc_id" in docs.columns:
            join_key = ("doc_id", master_docid, "doc_id")
    else:
        raise KeyError("No compatible join key between docs and docs_master (need doc_id or doc_uid in both).")

    key_name, master_key_col, docs_key_col = join_key
    print(f"[FIX] Joining titles on key={key_name} (master:{master_key_col} -> docs:{docs_key_col})")

    # Build mapping with deterministic first occurrence (drop duplicates on key)
    tmp = master_df[[master_key_col, title_col]].copy()
    tmp[master_key_col] = tmp[master_key_col].astype(str).str.strip()
    tmp[title_col] = tmp[title_col].fillna("").astype(str).str.strip()
    tmp = tmp[tmp[master_key_col].str.len() > 0].copy()
    tmp = tmp.drop_duplicates(subset=[master_key_col], keep="first")

    title_map = dict(zip(tmp[master_key_col].tolist(), tmp[title_col].tolist()))

    docs_key = docs[docs_key_col].astype(str).str.strip()
    fill = docs_key.map(lambda k: title_map.get(k, ""))
    # apply
    docs["title"] = docs["title"].fillna("").astype(str)
    need = docs["title"].fillna("").astype(str).str.strip().str.len().eq(0)
    docs.loc[need, "title"] = fill[need].fillna("").astype(str)

    title_rate_after = nonempty_rate(docs["title"])
    print("[AFTER] docs.title nonempty rate:", title_rate_after)
    if title_rate_after < 0.05:
        print("[WARN] Title recovery still low. This suggests the join key doesn't match across artifacts.")
else:
    print("[SKIP] docs.title already populated (rate >= 0.05).")

# Update globals
globals()["docs"] = docs

# Optional persist docs_indexed.csv (only affects future loads; not required for this run)
if SAVE_DOCS_INDEXED and indexed_path.exists():
    print("\n" + "="*110)
    print("💾 PERSIST FIXED docs_indexed.csv (optional)")
    print("="*110)
    bak = backup_file(indexed_path)
    if bak:
        print("[BACKUP]", bak)
    # persist minimal: overwrite title col in docs_indexed if present; else add it
    idx = indexed_df if indexed_df is not None else pd.read_csv(bak, low_memory=False)
    if "doc_id" in idx.columns and "doc_id" in docs.columns:
        idx["doc_id"] = idx["doc_id"].astype(str).str.strip()
        dmap = docs.set_index("doc_id")["title"].fillna("").astype(str).to_dict()
        idx["title"] = idx["doc_id"].map(lambda k: dmap.get(k, ""))
        idx.to_csv(indexed_path, index=False)
        print("[WRITE]", indexed_path)
    else:
        print("[SKIP] cannot persist: docs_indexed.csv lacks doc_id")

# -------------------------
# C) (Optional) Run missing LLM conditions if API key exists
# -------------------------
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()

def llm_available():
    return RUN_LLM_IF_KEY and bool(API_KEY)

# Ensure ws_run exists (reuse from prior final cell if present)
if "ws_run" not in globals():
    # rebuild quickly from described-only if needed
    assert "ws_run" in globals(), "ws_run missing. Run your final cell once to create ws_run."

# We need retrieval objects
for n in ["index_r0","enc_r0","index_r1","enc_r1","index_map"]:
    assert n in globals(), f"Missing {n}. Run earlier retrieval/index cells."

import faiss

index_r0 = globals()["index_r0"]; enc_r0 = globals()["enc_r0"]
index_r1 = globals()["index_r1"]; enc_r1 = globals()["enc_r1"]
index_map = globals()["index_map"].copy()
ws_run = globals()["ws_run"].copy()
K_EVID = int(globals().get("K_EVID", 12))

def retrieve_hits(retriever: str, query: str, k: int):
    q = str(query)
    if retriever == "R0":
        v = enc_r0.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r0.search(v, k)
        hits = docs.iloc[I[0]].copy()
        hits["score"] = D[0]; hits["faiss_pos"] = I[0]
        return hits.reset_index(drop=True)
    v = enc_r1.encode([q], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(v)
    D, I = index_r1.search(v, k)
    tmp = index_map.iloc[I[0]].copy()
    tmp["score"] = D[0]; tmp["faiss_pos"] = I[0]
    hits = tmp.merge(docs, on="doc_id", how="left")
    return hits.reset_index(drop=True)

def build_bundle(ws_row, retriever):
    hits = retrieve_hits(retriever, ws_row["_gap_text"], K_EVID)
    evid=[]
    for _, r in hits.iterrows():
        did = str(r.get("doc_id","")).strip()
        if not did or did.lower()=="nan":
            continue
        evid.append({
            "doc_id": did,
            "title": str(r.get("title",""))[:200],
            "year": r.get("year", None),
            "score": float(r.get("score", np.nan)) if pd.notna(r.get("score", np.nan)) else None,
            "abstract": str(r.get("abstract",""))[:1200],
        })
    evid = evid[:K_EVID]
    return {
        "cluster": ws_row.get("CLUSTER", None),
        "white_space_score": float(ws_row.get("white_space_score", np.nan)) if "white_space_score" in ws_row else None,
        "query_text": str(ws_row["_gap_text"]),
        "evidence": evid,
        "supporting_doc_ids": [e["doc_id"] for e in evid],
    }

def write_json(obj, path: Path):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

# Run only if files missing
def ensure_struct_only():
    for retr in ["R0","R1"]:
        p = OUT_DIR / f"agent_{retr}_STRUCT_ONLY.json"
        if p.exists():
            continue
        gaps=[]
        for i, row in ws_run.iterrows():
            b = build_bundle(row, retr)
            gaps.append({
                "gap_id": f"G{i:04d}",
                "cluster": b["cluster"],
                "gap_text": b["query_text"],
                "supporting_doc_ids": b["supporting_doc_ids"],
                "evidence": [{"doc_id":e["doc_id"],"title":e["title"],"year":e["year"],"score":e["score"]} for e in b["evidence"]],
            })
        payload = {"mode":"STRUCT_ONLY_NO_LLM","retriever":retr,"k_evidence":K_EVID,"generated_at_utc":datetime.now(timezone.utc).isoformat(),"candidate_gaps":gaps}
        write_json(payload, p)
        print("[WRITE]", p.name)

ensure_struct_only()

if llm_available():
    print("\n" + "="*110)
    print(f"🤖 RUNNING LLM CONDITIONS (model={MODEL_NAME})")
    print("="*110)
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY)

    SYSTEM = (
        "You are a scientific assistant under strict grounding.\n"
        "Use ONLY the provided EVIDENCE DOCS.\n"
        "Do NOT invent citations.\n"
        "Every claim must cite 1–3 DOC_IDs from EVIDENCE.\n"
        "Return ONLY valid JSON.\n"
    )

    def call(prompt: str):
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],
            temperature=0.2,
        )
        return resp.choices[0].message.content

    def run_mode(retriever: str, use_struct: bool, mode_name: str):
        out_path = OUT_DIR / f"agent_{retriever}_{mode_name}.json"
        if out_path.exists():
            print("[SKIP exists]", out_path.name)
            return
        gaps=[]
        for i, ws_row in ws_run.iterrows():
            b = build_bundle(ws_row, retriever)
            evid_lines = []
            for e in b["evidence"]:
                evid_lines.append(f"- DOC_ID={e['doc_id']}\n  TITLE: {e['title']}\n  YEAR: {e.get('year',None)}\n  ABSTRACT: {e.get('abstract','')}\n")
            evid_text = "\n".join(evid_lines)

            struct_block = ""
            if use_struct:
                struct_block = (
                    f"\nSTRUCTURE CONTEXT (NOT evidence):\n"
                    f"- CLUSTER: {b.get('cluster')}\n"
                    f"- WHITE_SPACE_SCORE: {b.get('white_space_score')}\n"
                    f"- QUERY_TERMS: {b.get('query_text')}\n"
                )

            prompt = f"""
TASK:
Using ONLY the EVIDENCE DOCS, produce:
1) One concise research gap statement grounded in those docs.
2) 3–6 atomic supporting claims. Each claim MUST cite 1–3 DOC_IDs.

EVIDENCE DOCS:
{evid_text}
{struct_block}

OUTPUT JSON ONLY:
{{
  "gap_text": "…",
  "claims": [{{"text":"…","cites":["DOC_ID", "..."]}}, ...]
}}
RULES:
- cites MUST be subset of provided DOC_IDs only.
- If evidence is not about solar/space-weather, do NOT force it; stay in evidence domain.
"""
            raw = call(prompt)
            try:
                obj = json.loads(raw)
            except Exception:
                m = re.search(r"\{.*\}", raw, re.S)
                obj = json.loads(m.group(0)) if m else {"gap_text":"","claims":[]}

            evid_set = set(b["supporting_doc_ids"])
            gap_text = str(obj.get("gap_text","")).strip()
            claims = obj.get("claims", [])
            if not isinstance(claims, list): claims=[]
            cleaned=[]; used=set()
            for c in claims:
                if not isinstance(c, dict): continue
                txt = str(c.get("text","")).strip()
                cites = c.get("cites", [])
                if not isinstance(cites, list): cites=[]
                cites = [str(x).strip() for x in cites if str(x).strip() in evid_set]
                if txt and cites:
                    cleaned.append({"text": txt, "cites": cites[:3]})
                    used.update(cites[:3])

            gaps.append({
                "gap_id": f"G{i:04d}",
                "cluster": b.get("cluster"),
                "gap_text": gap_text,
                "claims": cleaned,
                "supporting_doc_ids": sorted(list(used)),
                "evidence": [{"doc_id":e["doc_id"],"title":e["title"],"year":e.get("year",None),"score":e.get("score",None)} for e in b["evidence"]],
            })
        payload = {"mode":mode_name,"retriever":retriever,"use_struct":use_struct,"k_evidence":K_EVID,"generated_at_utc":datetime.now(timezone.utc).isoformat(),"candidate_gaps":gaps}
        write_json(payload, out_path)
        print("[WRITE]", out_path.name)

    for retr in ["R0","R1"]:
        run_mode(retr, False, "LLM_GROUNDED_NO_STRUCT")
        run_mode(retr, True,  "LLM_GROUNDED_STRUCT")
else:
    print("\n[LLM] OPENAI_API_KEY missing -> LLM conditions cannot be executed in this notebook run.")

# -------------------------
# D) Re-evaluate metrics + H1/H2/H3 (robust, uses abstract domain)
# -------------------------
docset = set(docs["doc_id"].astype(str))
doc_title = docs.set_index("doc_id")["title"].fillna("").astype(str).to_dict()
doc_abs   = docs.set_index("doc_id")["abstract"].fillna("").astype(str).to_dict()

DOMAIN_TERMS = [
    "solar","flare","flares","space weather","magnetogram","magnetic","active region",
    "hmi","sdo","sharp","cme","coronal","sunspot","photosphere","chromosphere","goes","noaa"
]
domain_pat = re.compile("|".join([re.escape(t) for t in sorted(DOMAIN_TERMS, key=len, reverse=True)]), re.I)
def has_domain(text: str) -> int:
    return int(bool(text) and (domain_pat.search(text) is not None))

files = sorted(list(OUT_DIR.glob("agent_*.json")))
rows=[]
for p in files:
    data = json.loads(p.read_text(encoding="utf-8"))
    mode = data.get("mode","UNK"); retr = data.get("retriever","UNK")
    gaps = data.get("candidate_gaps", [])
    if not isinstance(gaps, list): gaps=[]
    all_cited=[]; all_claims=0; uncited=0
    title_miss=[]; abs_miss=[]; title_dom=[]; abs_dom=[]
    for g in gaps:
        if not isinstance(g, dict): continue
        cited = g.get("supporting_doc_ids", [])
        if not isinstance(cited, list): cited=[]
        cited = [str(x) for x in cited]
        all_cited.extend(cited)

        claims = g.get("claims", [])
        if isinstance(claims, list) and claims:
            for c in claims:
                if not isinstance(c, dict): continue
                all_claims += 1
                cites = c.get("cites", [])
                if not isinstance(cites, list) or len(cites)==0:
                    uncited += 1

        any_title=0; any_abs=0; mt=0; ma=0
        for did in cited[:K_EVID]:
            t = doc_title.get(did,""); a = doc_abs.get(did,"")
            if not t.strip(): mt += 1
            if not a.strip(): ma += 1
            any_title |= has_domain(t.lower())
            any_abs   |= has_domain(a.lower())
        title_miss.append(mt/max(len(cited[:K_EVID]),1))
        abs_miss.append(ma/max(len(cited[:K_EVID]),1))
        title_dom.append(any_title)
        abs_dom.append(any_abs)

    cited_total = len(all_cited)
    invalid = sum(1 for d in all_cited if d not in docset)
    icr = invalid / max(cited_total, 1)
    ucr = np.nan if all_claims==0 else (uncited / all_claims)

    rows.append({
        "file": p.name,
        "mode": mode,
        "retriever": retr,
        "gaps": len(gaps),
        "cited_total": cited_total,
        "invalid_cites": invalid,
        "ICR": float(icr),
        "claims_total": int(all_claims),
        "uncited_claims": int(uncited),
        "UCR": float(ucr) if pd.notna(ucr) else np.nan,
        "title_missing_rate_topk": float(np.mean(title_miss)) if title_miss else np.nan,
        "abstract_missing_rate_topk": float(np.mean(abs_miss)) if abs_miss else np.nan,
        "domain_hit_rate_title_topk": float(np.mean(title_dom)) if title_dom else np.nan,
        "domain_hit_rate_abstract_topk": float(np.mean(abs_dom)) if abs_dom else np.nan,
    })

eval_df = pd.DataFrame(rows).sort_values(["retriever","mode"]).reset_index(drop=True)
(eval_df).to_csv(REP_DIR/"eval_metrics.csv", index=False)
(REP_DIR/"summary_metrics.json").write_text(json.dumps({
    "by_retriever_mode": eval_df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index().to_dict(orient="records"),
    "files": rows
}, ensure_ascii=False, indent=2), encoding="utf-8")

agg = eval_df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index()
def get(retr, mode):
    x = agg[(agg["retriever"]==retr) & (agg["mode"]==mode)]
    return x.iloc[0] if len(x) else None

# H1
h1_rows=[]
for mode in sorted(agg["mode"].unique()):
    r0 = get("R0", mode); r1 = get("R1", mode)
    if r0 is None or r1 is None: continue
    h1_rows.append({
        "mode": mode,
        "delta_domain_hit_rate_abstract (R1-R0)": r1.get("domain_hit_rate_abstract_topk", np.nan) - r0.get("domain_hit_rate_abstract_topk", np.nan),
        "delta_ICR (R1-R0)": r1.get("ICR", np.nan) - r0.get("ICR", np.nan),
        "delta_UCR (R1-R0)": r1.get("UCR", np.nan) - r0.get("UCR", np.nan),
    })
h1 = pd.DataFrame(h1_rows)
h1.to_csv(REP_DIR/"H1_deltas.csv", index=False)

# H2/H3
llm_modes = {"LLM_GROUNDED_NO_STRUCT","LLM_GROUNDED_STRUCT"}
has_llm = any(m in set(agg["mode"].unique()) for m in llm_modes)

if not has_llm:
    h2 = pd.DataFrame([{"note":"N/A — no LLM outputs present (OPENAI_API_KEY missing or LLM not executed)."}])
    h3 = pd.DataFrame([{"note":"N/A — requires both LLM modes to compare STRUCT vs NO_STRUCT."}])
else:
    h2_rows=[]
    for retr in ["R0","R1"]:
        for mode in ["LLM_GROUNDED_NO_STRUCT","LLM_GROUNDED_STRUCT"]:
            r = get(retr, mode)
            if r is None: continue
            h2_rows.append({"retriever":retr,"mode":mode,"ICR":r.get("ICR",np.nan),"UCR":r.get("UCR",np.nan),"domain_hit_rate_abstract":r.get("domain_hit_rate_abstract_topk",np.nan)})
    h2 = pd.DataFrame(h2_rows)

    h3_rows=[]
    for retr in ["R0","R1"]:
        ns = get(retr, "LLM_GROUNDED_NO_STRUCT")
        st = get(retr, "LLM_GROUNDED_STRUCT")
        if ns is None or st is None: continue
        h3_rows.append({
            "retriever": retr,
            "delta_domain_hit_rate_abstract (STRUCT-NO_STRUCT)": st.get("domain_hit_rate_abstract_topk", np.nan) - ns.get("domain_hit_rate_abstract_topk", np.nan),
            "delta_UCR (STRUCT-NO_STRUCT)": st.get("UCR", np.nan) - ns.get("UCR", np.nan),
            "delta_ICR (STRUCT-NO_STRUCT)": st.get("ICR", np.nan) - ns.get("ICR", np.nan),
        })
    h3 = pd.DataFrame(h3_rows)

h2.to_csv(REP_DIR/"H2_levels.csv", index=False)
h3.to_csv(REP_DIR/"H3_deltas.csv", index=False)

print("\n" + "="*110)
print("✅ UPDATED RESULTS")
print("="*110)
print("docs.title nonempty rate (after fix):", nonempty_rate(docs["title"]))
print("agent outputs present:", [p.name for p in sorted(OUT_DIR.glob("agent_*.json"))])
display(eval_df)
print("\nH1:"); display(h1)
print("\nH2:"); display(h2)
print("\nH3:"); display(h3)
print("\n[WRITE] reports updated in:", REP_DIR)

# -------------------------
# E) Zip package (fresh)
# -------------------------
ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = Path("/content") / f"pipe3_artifacts_{RUN_DIR.name}_{ts}.zip"

def add_if_exists(zf, path: Path, arc_prefix=""):
    if path.exists():
        if path.is_dir():
            for p in path.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(Path(arc_prefix) / p.relative_to(path)))
        else:
            zf.write(path, arcname=str(Path(arc_prefix) / path.name))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    add_if_exists(zf, OUT_DIR, "run/outputs")
    add_if_exists(zf, REP_DIR, "run/reports")
    add_if_exists(zf, FIG_DIR, "run/figures")
    add_if_exists(zf, LOG_DIR, "run/logs")
    # root inputs (best-effort)
    for fname in ["docs_master.csv","docs_indexed.csv","index_map.csv","faiss_index.bin","faiss_index_r0.bin","faiss_index_r1.bin",
                  "piml_white_space_candidates_described.csv","piml_white_space_candidates.csv","piml_cluster_topics.csv","piml_clusters.csv","manifest.json"]:
        add_if_exists(zf, ROOT/fname, "root")

print("\n📦 ZIP READY:", zip_path)
print("➡️ Download via Colab Files panel:", zip_path.name)
globals()["PIPE3_ARTIFACT_ZIP"] = str(zip_path)


In [ ]:
# ===========================
# ✅ ONE CELL — Run missing LLM conditions (4 runs) using ABSTRACT as audit-title + regenerate eval + H1/H2/H3
# Put this as a SINGLE cell at the end of the Colab.
#
# 🔑 Set your key here (do NOT commit/share it):
OPENAI_<REDACTED_SECRET>
#
# What this cell does:
#  1) Ensures STRUCT_ONLY outputs exist (R0/R1)
#  2) Runs 4 LLM grounded conditions:
#     - R0_LLM_GROUNDED_NO_STRUCT
#     - R0_LLM_GROUNDED_STRUCT
#     - R1_LLM_GROUNDED_NO_STRUCT
#     - R1_LLM_GROUNDED_STRUCT
#     using ABSTRACT as the audit field when title is missing.
#  3) Re-evaluates metrics and writes:
#     - reports/eval_metrics.csv
#     - reports/summary_metrics.json
#     - reports/H1_deltas.csv
#     - reports/H2_levels.csv
#     - reports/H3_deltas.csv
#  4) Creates a ZIP package in /content for download.
#
# Assumes you already have in memory:
#   docs, ws_run, index_r0/index_r1, enc_r0/enc_r1, index_map, RUN_DIR
# ===========================

import os, re, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# -------------------------
# 0) Preconditions
# -------------------------
assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"
LOG_DIR = RUN_DIR / "logs"
for d in [OUT_DIR, REP_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for n in ["docs","ws_run","index_r0","index_r1","enc_r0","enc_r1","index_map"]:
    assert n in globals(), f"Missing {n}"

docs = globals()["docs"].copy()
ws_run = globals()["ws_run"].copy()
index_r0 = globals()["index_r0"]; enc_r0 = globals()["enc_r0"]
index_r1 = globals()["index_r1"]; enc_r1 = globals()["enc_r1"]
index_map = globals()["index_map"].copy()

assert "doc_id" in docs.columns, "docs must have doc_id"
docs["doc_id"] = docs["doc_id"].astype(str)

# Use ABSTRACT as the audit field if title is empty
docs["title_audit"] = docs.get("title", "").fillna("").astype(str).str.strip()
abs_series = docs.get("abstract", pd.Series([""]*len(docs))).fillna("").astype(str).str.strip()
mask = docs["title_audit"].str.len().eq(0)
docs.loc[mask, "title_audit"] = abs_series.loc[mask].str.slice(0, 220)  # short audit title

# -------------------------
# 1) Retrieval + evidence bundle
# -------------------------
import faiss

K_EVID = int(globals().get("K_EVID", 12))

def retrieve_hits(retriever: str, query: str, k: int):
    q = str(query)
    if retriever == "R0":
        v = enc_r0.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r0.search(v, k)
        hits = docs.iloc[I[0]].copy()
        hits["score"] = D[0]; hits["faiss_pos"] = I[0]
        return hits.reset_index(drop=True)
    elif retriever == "R1":
        v = enc_r1.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r1.search(v, k)
        tmp = index_map.iloc[I[0]].copy()
        tmp["score"] = D[0]; tmp["faiss_pos"] = I[0]
        hits = tmp.merge(docs, on="doc_id", how="left")
        return hits.reset_index(drop=True)
    else:
        raise ValueError("retriever must be R0 or R1")

def build_bundle(ws_row, retriever):
    hits = retrieve_hits(retriever, ws_row["_gap_text"], K_EVID)
    evid=[]
    for _, r in hits.iterrows():
        did = str(r.get("doc_id","")).strip()
        if not did or did.lower()=="nan":
            continue
        evid.append({
            "doc_id": did,
            "title_audit": str(r.get("title_audit",""))[:220],
            "year": r.get("year", None),
            "score": float(r.get("score", np.nan)) if pd.notna(r.get("score", np.nan)) else None,
            "abstract": str(r.get("abstract",""))[:1200],
        })
    evid = evid[:K_EVID]
    return {
        "cluster": ws_row.get("CLUSTER", None),
        "white_space_score": float(ws_row.get("white_space_score", np.nan)) if "white_space_score" in ws_row else None,
        "query_text": str(ws_row["_gap_text"]),
        "evidence": evid,
        "supporting_doc_ids": [e["doc_id"] for e in evid],
    }

def write_json(obj, path: Path):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

# -------------------------
# 2) Ensure STRUCT_ONLY exists (R0/R1)
# -------------------------
def ensure_struct_only():
    for retr in ["R0","R1"]:
        p = OUT_DIR / f"agent_{retr}_STRUCT_ONLY.json"
        if p.exists():
            continue
        gaps=[]
        for i, row in ws_run.iterrows():
            b = build_bundle(row, retr)
            gaps.append({
                "gap_id": f"G{i:04d}",
                "cluster": b["cluster"],
                "gap_text": b["query_text"],
                "supporting_doc_ids": b["supporting_doc_ids"],
                "evidence": [{"doc_id":e["doc_id"],"title":e["title_audit"],"year":e["year"],"score":e["score"]} for e in b["evidence"]],
            })
        payload = {"mode":"STRUCT_ONLY_NO_LLM","retriever":retr,"k_evidence":K_EVID,"generated_at_utc":datetime.now(timezone.utc).isoformat(),"candidate_gaps":gaps}
        write_json(payload, p)
        print("[WRITE]", p.name)

ensure_struct_only()

# -------------------------
# 3) Run LLM conditions (4 runs)
# -------------------------
if not OPENAI_API_KEY.strip():
    raise ValueError("OPENAI_API_KEY is empty. Paste your key into the variable at the top of this cell.")

# OpenAI client
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY.strip())

SYSTEM = (
    "You are a scientific assistant under strict grounding.\n"
    "Use ONLY the provided EVIDENCE DOCS.\n"
    "Do NOT invent citations.\n"
    "Every claim must cite 1–3 DOC_IDs from EVIDENCE.\n"
    "Return ONLY valid JSON.\n"
)

def call(prompt: str):
    resp = client.chat.completions.create(
        model=globals().get("LLM_MODEL", "gpt-4o-mini"),
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content

def run_mode(retriever: str, use_struct: bool, mode_name: str):
    out_path = OUT_DIR / f"agent_{retriever}_{mode_name}.json"
    if out_path.exists():
        print("[SKIP exists]", out_path.name)
        return

    gaps=[]
    for i, ws_row in ws_run.iterrows():
        b = build_bundle(ws_row, retriever)
        evid_set = set(b["supporting_doc_ids"])

        evid_lines=[]
        for e in b["evidence"]:
            evid_lines.append(
                f"- DOC_ID={e['doc_id']}\n"
                f"  AUDIT_TITLE: {e['title_audit']}\n"
                f"  YEAR: {e.get('year',None)}\n"
                f"  ABSTRACT: {e.get('abstract','')}\n"
            )
        evid_text = "\n".join(evid_lines)

        struct_block = ""
        if use_struct:
            struct_block = (
                f"\nSTRUCTURE CONTEXT (NOT evidence):\n"
                f"- CLUSTER: {b.get('cluster')}\n"
                f"- WHITE_SPACE_SCORE: {b.get('white_space_score')}\n"
                f"- QUERY_TERMS: {b.get('query_text')}\n"
            )

        prompt = f"""
TASK:
Using ONLY the EVIDENCE DOCS, produce:
1) One concise research gap statement grounded in those docs.
2) 3–6 atomic supporting claims. Each claim MUST cite 1–3 DOC_IDs.

EVIDENCE DOCS:
{evid_text}
{struct_block}

OUTPUT JSON ONLY:
{{
  "gap_text": "…",
  "claims": [{{"text":"…","cites":["DOC_ID", "..."]}}, ...]
}}

RULES:
- cites MUST be subset of provided DOC_IDs only.
- If evidence is not about solar/space-weather, do NOT force it; stay in evidence domain.
"""
        raw = call(prompt)
        try:
            obj = json.loads(raw)
        except Exception:
            m = re.search(r"\{.*\}", raw, re.S)
            obj = json.loads(m.group(0)) if m else {"gap_text":"","claims":[]}

        gap_text = str(obj.get("gap_text","")).strip()
        claims = obj.get("claims", [])
        if not isinstance(claims, list):
            claims = []

        cleaned=[]; used=set()
        for c in claims:
            if not isinstance(c, dict):
                continue
            txt = str(c.get("text","")).strip()
            cites = c.get("cites", [])
            if not isinstance(cites, list):
                cites=[]
            cites = [str(x).strip() for x in cites if str(x).strip() in evid_set]
            if txt and cites:
                cleaned.append({"text": txt, "cites": cites[:3]})
                used.update(cites[:3])

        gaps.append({
            "gap_id": f"G{i:04d}",
            "cluster": b.get("cluster"),
            "gap_text": gap_text,
            "claims": cleaned,
            "supporting_doc_ids": sorted(list(used)),
            "evidence": [{"doc_id":e["doc_id"],"title":e["title_audit"],"year":e.get("year",None),"score":e.get("score",None)} for e in b["evidence"]],
        })

    payload = {
        "mode": mode_name,
        "retriever": retriever,
        "use_struct": use_struct,
        "k_evidence": K_EVID,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "candidate_gaps": gaps,
    }
    write_json(payload, out_path)
    print("[WRITE]", out_path.name)

for retr in ["R0","R1"]:
    run_mode(retr, False, "LLM_GROUNDED_NO_STRUCT")
    run_mode(retr, True,  "LLM_GROUNDED_STRUCT")

# -------------------------
# 4) Re-evaluate metrics + H1/H2/H3
# -------------------------
docset = set(docs["doc_id"].astype(str))
doc_title = docs.set_index("doc_id")["title_audit"].fillna("").astype(str).to_dict()
doc_abs   = docs.set_index("doc_id")["abstract"].fillna("").astype(str).to_dict()

DOMAIN_TERMS = [
    "solar","flare","flares","space weather","magnetogram","magnetic","active region",
    "hmi","sdo","sharp","cme","coronal","sunspot","photosphere","chromosphere","goes","noaa"
]
domain_pat = re.compile("|".join([re.escape(t) for t in sorted(DOMAIN_TERMS, key=len, reverse=True)]), re.I)
def has_domain(text: str) -> int:
    return int(bool(text) and (domain_pat.search(text) is not None))

agent_files = sorted(list(OUT_DIR.glob("agent_*.json")))
rows=[]
for p in agent_files:
    data = json.loads(p.read_text(encoding="utf-8"))
    mode = data.get("mode","UNK"); retr = data.get("retriever","UNK")
    gaps = data.get("candidate_gaps", [])
    if not isinstance(gaps, list): gaps=[]
    all_cited=[]; all_claims=0; uncited=0
    title_miss=[]; abs_miss=[]; title_dom=[]; abs_dom=[]
    for g in gaps:
        if not isinstance(g, dict): continue
        cited = g.get("supporting_doc_ids", [])
        if not isinstance(cited, list): cited=[]
        cited = [str(x) for x in cited]
        all_cited.extend(cited)

        claims = g.get("claims", [])
        if isinstance(claims, list) and claims:
            for c in claims:
                if not isinstance(c, dict): continue
                all_claims += 1
                cites = c.get("cites", [])
                if not isinstance(cites, list) or len(cites)==0:
                    uncited += 1

        any_title=0; any_abs=0; mt=0; ma=0
        for did in cited[:K_EVID]:
            t = doc_title.get(did,""); a = doc_abs.get(did,"")
            if not t.strip(): mt += 1
            if not a.strip(): ma += 1
            any_title |= has_domain(t.lower())
            any_abs   |= has_domain(a.lower())
        title_miss.append(mt/max(len(cited[:K_EVID]),1))
        abs_miss.append(ma/max(len(cited[:K_EVID]),1))
        title_dom.append(any_title)
        abs_dom.append(any_abs)

    cited_total = len(all_cited)
    invalid = sum(1 for d in all_cited if d not in docset)
    icr = invalid / max(cited_total, 1)
    ucr = np.nan if all_claims==0 else (uncited / all_claims)

    rows.append({
        "file": p.name,
        "mode": mode,
        "retriever": retr,
        "gaps": len(gaps),
        "cited_total": cited_total,
        "invalid_cites": invalid,
        "ICR": float(icr),
        "claims_total": int(all_claims),
        "uncited_claims": int(uncited),
        "UCR": float(ucr) if pd.notna(ucr) else np.nan,
        "title_missing_rate_topk": float(np.mean(title_miss)) if title_miss else np.nan,
        "abstract_missing_rate_topk": float(np.mean(abs_miss)) if abs_miss else np.nan,
        "domain_hit_rate_title_topk": float(np.mean(title_dom)) if title_dom else np.nan,
        "domain_hit_rate_abstract_topk": float(np.mean(abs_dom)) if abs_dom else np.nan,
    })

eval_df = pd.DataFrame(rows).sort_values(["retriever","mode"]).reset_index(drop=True)
eval_df.to_csv(REP_DIR/"eval_metrics.csv", index=False)

summary = {
    "by_retriever_mode": eval_df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index().to_dict(orient="records"),
    "files": rows,
}
(REP_DIR/"summary_metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

agg = eval_df.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index()
def get(retr, mode):
    x = agg[(agg["retriever"]==retr) & (agg["mode"]==mode)]
    return x.iloc[0] if len(x) else None

# H1 (R1-R0 deltas)
h1_rows=[]
for mode in sorted(agg["mode"].unique()):
    r0 = get("R0", mode); r1 = get("R1", mode)
    if r0 is None or r1 is None: continue
    h1_rows.append({
        "mode": mode,
        "delta_domain_hit_rate_abstract (R1-R0)": r1.get("domain_hit_rate_abstract_topk", np.nan) - r0.get("domain_hit_rate_abstract_topk", np.nan),
        "delta_ICR (R1-R0)": r1.get("ICR", np.nan) - r0.get("ICR", np.nan),
        "delta_UCR (R1-R0)": r1.get("UCR", np.nan) - r0.get("UCR", np.nan),
    })
h1 = pd.DataFrame(h1_rows)
h1.to_csv(REP_DIR/"H1_deltas.csv", index=False)

# H2 levels
h2_rows=[]
for retr in ["R0","R1"]:
    for mode in ["LLM_GROUNDED_NO_STRUCT","LLM_GROUNDED_STRUCT"]:
        r = get(retr, mode)
        if r is None: continue
        h2_rows.append({
            "retriever": retr,
            "mode": mode,
            "ICR": r.get("ICR", np.nan),
            "UCR": r.get("UCR", np.nan),
            "domain_hit_rate_abstract": r.get("domain_hit_rate_abstract_topk", np.nan),
        })
h2 = pd.DataFrame(h2_rows)
h2.to_csv(REP_DIR/"H2_levels.csv", index=False)

# H3 deltas (STRUCT - NO_STRUCT)
h3_rows=[]
for retr in ["R0","R1"]:
    ns = get(retr, "LLM_GROUNDED_NO_STRUCT")
    st = get(retr, "LLM_GROUNDED_STRUCT")
    if ns is None or st is None: continue
    h3_rows.append({
        "retriever": retr,
        "delta_domain_hit_rate_abstract (STRUCT-NO_STRUCT)": st.get("domain_hit_rate_abstract_topk", np.nan) - ns.get("domain_hit_rate_abstract_topk", np.nan),
        "delta_UCR (STRUCT-NO_STRUCT)": st.get("UCR", np.nan) - ns.get("UCR", np.nan),
        "delta_ICR (STRUCT-NO_STRUCT)": st.get("ICR", np.nan) - ns.get("ICR", np.nan),
    })
h3 = pd.DataFrame(h3_rows)
h3.to_csv(REP_DIR/"H3_deltas.csv", index=False)

# -------------------------
# 5) Print results
# -------------------------
print("\n" + "="*110)
print("✅ EXPERIMENT COMPLETE — RESULTS")
print("="*110)
display(eval_df)

print("\nH1_deltas:"); display(h1)
print("\nH2_levels:"); display(h2)
print("\nH3_deltas:"); display(h3)

# -------------------------
# 6) Zip artifacts
# -------------------------
ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = Path("/content") / f"pipe3_artifacts_{RUN_DIR.name}_{ts}.zip"

def add_if_exists(zf, path: Path, arc_prefix=""):
    if path.exists():
        if path.is_dir():
            for p in path.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(Path(arc_prefix) / p.relative_to(path)))
        else:
            zf.write(path, arcname=str(Path(arc_prefix) / path.name))

ROOT = Path("/content")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    add_if_exists(zf, OUT_DIR, "run/outputs")
    add_if_exists(zf, REP_DIR, "run/reports")
    add_if_exists(zf, FIG_DIR, "run/figures")
    add_if_exists(zf, LOG_DIR, "run/logs")
    for fname in [
        "docs_master.csv","docs_indexed.csv","index_map.csv",
        "faiss_index.bin","faiss_index_r0.bin","faiss_index_r1.bin",
        "piml_white_space_candidates_described.csv","piml_white_space_candidates.csv",
        "piml_cluster_topics.csv","piml_clusters.csv","manifest.json"
    ]:
        add_if_exists(zf, ROOT/fname, "root")

print("\n📦 ZIP READY:", zip_path)
print("➡️ Download via Colab Files panel:", zip_path.name)
globals()["PIPE3_ARTIFACT_ZIP"] = str(zip_path)


In [ ]:
# ===========================
# ✅ ONE CELL — Build "PAPER PACK" for writing the article + ZIP for download
# Outputs:
#  - /content/PIPE3_PAPER_PACK_<run>.zip   (download)
#  - Inside:
#     run/outputs/agent_*.json
#     run/reports/eval_metrics.csv, summary_metrics.json, H1/H2/H3
#     paper/paper_tables.xlsx
#     paper/paper_notes.md
#     paper/experiment_manifest.json (conditions + params)
# ===========================

import json, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timezone

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
assert OUT_DIR.exists() and REP_DIR.exists(), "Missing outputs/reports. Run the experiment first."

# --- load core report files
eval_path = REP_DIR / "eval_metrics.csv"
sum_path  = REP_DIR / "summary_metrics.json"
h1_path   = REP_DIR / "H1_deltas.csv"
h2_path   = REP_DIR / "H2_levels.csv"
h3_path   = REP_DIR / "H3_deltas.csv"

eval_df = pd.read_csv(eval_path)
h1 = pd.read_csv(h1_path)
h2 = pd.read_csv(h2_path)
h3 = pd.read_csv(h3_path)
summary = json.loads(sum_path.read_text(encoding="utf-8"))

# --- Build a clean conditions table (for Methods section)
cond_rows = []
for _, r in eval_df.iterrows():
    cond_rows.append({
        "retriever": r["retriever"],
        "mode": r["mode"],
        "has_llm": int("LLM" in str(r["mode"])),
        "has_structure": int("STRUCT" in str(r["mode"]) and "NO_LLM" not in str(r["mode"])),
        "gaps": int(r["gaps"]),
        "k_evidence_nominal": 12,  # if you used another value, update here or set globals()["K_EVID"]
        "note": "STRUCT_ONLY uses top-k as supporting_doc_ids; LLM modes use cited subset as supporting_doc_ids",
    })

cond_df = pd.DataFrame(cond_rows).drop_duplicates(subset=["retriever","mode"]).sort_values(["retriever","mode"])

# --- Build a compact metrics table (Results section)
metrics_cols = [
    "retriever","mode","gaps","cited_total","ICR","claims_total","UCR",
    "domain_hit_rate_abstract_topk"
]
metrics_df = eval_df[metrics_cols].copy()

# --- Write paper notes (no extrapolation)
notes = []
notes.append("# Pipe 3 — Paper Notes (auto-generated)\n")
notes.append("## What the experiment *does* support (from artifacts)\n")
notes.append("- All 6 conditions produced outputs (STRUCT_ONLY + 4 LLM grounded + 2 retrievers).\n")
notes.append("- Invalid Citation Rate (ICR) is 0.0 across all conditions (no citations to non-existent doc_ids).\n")
notes.append("- Uncited Claim Rate (UCR) is 0.0 in LLM modes (all recorded claims carry at least one citation).\n")
notes.append("\n## Important measurement caveat\n")
notes.append("- In LLM modes, `supporting_doc_ids` corresponds to the **subset of retrieved evidence that the agent actually cited**, not the full top-k retrieved set.\n")
notes.append("- Therefore, `domain_hit_rate_abstract_topk` in LLM modes is computed over cited docs only, and is not directly comparable to STRUCT_ONLY where it reflects top-k retrieval.\n")
notes.append("\n## What to report as limitations (strictly from observed data)\n")
notes.append("- Document titles are missing in the corpus (`docs_master.csv` has `title` column empty); auditing relies on abstracts.\n")
notes.append("- Domain-hit proxy depends on a keyword list; it is a coarse indicator.\n")
notes.append("- For a fair H1 retrieval comparison, compute domain-hit on the **top-k retrieved evidence** for each condition (requires saving top-k list separately for LLM modes).\n")

paper_notes_md = "\n".join(notes)

# --- Create an experiment manifest JSON for transparency
manifest = {
    "run_dir": str(RUN_DIR),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "files": {
        "eval_metrics": str(eval_path),
        "summary_metrics": str(sum_path),
        "H1_deltas": str(h1_path),
        "H2_levels": str(h2_path),
        "H3_deltas": str(h3_path),
    },
    "conditions": cond_df.to_dict(orient="records"),
    "metrics_columns": list(eval_df.columns),
    "notes": {
        "LLM_modes_supporting_doc_ids": "cited subset",
        "STRUCT_ONLY_supporting_doc_ids": "top-k retrieved list",
    }
}

# --- Write paper_tables.xlsx
paper_dir = Path("/content") / f"paper_pack_{RUN_DIR.name}"
paper_dir.mkdir(parents=True, exist_ok=True)

xlsx_path = paper_dir / "paper_tables.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as w:
    cond_df.to_excel(w, index=False, sheet_name="Conditions")
    metrics_df.to_excel(w, index=False, sheet_name="Metrics_by_condition")
    h1.to_excel(w, index=False, sheet_name="H1_deltas")
    h2.to_excel(w, index=False, sheet_name="H2_levels")
    h3.to_excel(w, index=False, sheet_name="H3_deltas")

# --- Write notes + manifest
notes_path = paper_dir / "paper_notes.md"
notes_path.write_text(paper_notes_md, encoding="utf-8")

manifest_path = paper_dir / "experiment_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

# --- Zip minimal necessary artifacts for the paper
zip_path = Path("/content") / f"PIPE3_PAPER_PACK_{RUN_DIR.name}.zip"

def add_if_exists(zf, path: Path, arc_prefix=""):
    if path.exists():
        if path.is_dir():
            for p in path.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(Path(arc_prefix) / p.relative_to(path)))
        else:
            zf.write(path, arcname=str(Path(arc_prefix) / path.name))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    add_if_exists(zf, OUT_DIR, "run/outputs")
    add_if_exists(zf, REP_DIR, "run/reports")
    add_if_exists(zf, paper_dir, "paper")

print("\n✅ PAPER PACK READY")
print("ZIP:", zip_path)
print("➡️ Download via Colab Files panel:", zip_path.name)
print("Included:", "\n - run/outputs (agent JSONs)\n - run/reports (metrics + H tables)\n - paper/paper_tables.xlsx\n - paper/paper_notes.md\n - paper/experiment_manifest.json")


In [ ]:
# ===========================
# ✅ ONE CELL (FINAL) — Save & evaluate TOP-K retrieved evidence for LLM modes (comparability fix)
#
# Why:
#  - In LLM modes, supporting_doc_ids = cited subset → domain_hit_rate becomes "cited-only"
#  - This cell adds a retrieval-side metric comparable across ALL modes:
#      domain_hit_rate_abstract_retrieved_topk
#      title_missing_rate_retrieved_topk
#      abstract_missing_rate_retrieved_topk
#  - It does NOT change your agent outputs; it only builds parallel "topk bundles" for evaluation.
#
# What it does:
#  1) For each agent file in RUN_DIR/outputs/agent_*.json:
#     - Reconstructs the TOP-K retrieved list for each gap using ws_run + retrieve_hits (R0/R1)
#     - Saves bundle files: RUN_DIR/outputs/topk_<agent_file>.json
#  2) Recomputes evaluation table with BOTH:
#     - cited-only metrics (existing)
#     - retrieved-topk metrics (new, comparable)
#  3) Writes:
#     - RUN_DIR/reports/eval_metrics_v2_with_retrieved_topk.csv
#     - RUN_DIR/reports/H1_deltas_v2.csv
#     - RUN_DIR/reports/H3_deltas_v2.csv
#     - RUN_DIR/reports/summary_metrics_v2.json
#
# Assumes in memory:
#   docs, ws_run, index_r0/index_r1, enc_r0/enc_r1, index_map, RUN_DIR
# ===========================

import json, re
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import faiss

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)

for n in ["docs","ws_run","index_r0","index_r1","enc_r0","enc_r1","index_map"]:
    assert n in globals(), f"Missing {n}"

docs = globals()["docs"].copy()
ws_run = globals()["ws_run"].copy()
index_r0 = globals()["index_r0"]; enc_r0 = globals()["enc_r0"]
index_r1 = globals()["index_r1"]; enc_r1 = globals()["enc_r1"]
index_map = globals()["index_map"].copy()

assert "doc_id" in docs.columns, "docs must have doc_id"
docs["doc_id"] = docs["doc_id"].astype(str)

# Use audit title: title if present else abstract snippet
if "title_audit" not in docs.columns:
    docs["title_audit"] = docs.get("title", "").fillna("").astype(str).str.strip()
    abs_series = docs.get("abstract", pd.Series([""]*len(docs))).fillna("").astype(str).str.strip()
    mask = docs["title_audit"].str.len().eq(0)
    docs.loc[mask, "title_audit"] = abs_series.loc[mask].str.slice(0, 220)

doc_title = docs.set_index("doc_id")["title_audit"].fillna("").astype(str).to_dict()
doc_abs   = docs.set_index("doc_id")["abstract"].fillna("").astype(str).to_dict()
docset = set(docs["doc_id"].astype(str))

# Domain keyword proxy (same as before; keep stable)
DOMAIN_TERMS = [
    "solar","flare","flares","space weather","magnetogram","magnetic","active region",
    "hmi","sdo","sharp","cme","coronal","sunspot","photosphere","chromosphere","goes","noaa"
]
domain_pat = re.compile("|".join([re.escape(t) for t in sorted(DOMAIN_TERMS, key=len, reverse=True)]), re.I)
def has_domain(text: str) -> int:
    return int(bool(text) and (domain_pat.search(text) is not None))

K_EVID = int(globals().get("K_EVID", 12))

def retrieve_hits(retriever: str, query: str, k: int):
    q = str(query)
    if retriever == "R0":
        v = enc_r0.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r0.search(v, k)
        hits = docs.iloc[I[0]].copy()
        hits["score"] = D[0]
        hits["faiss_pos"] = I[0]
        return hits.reset_index(drop=True)
    elif retriever == "R1":
        v = enc_r1.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r1.search(v, k)
        tmp = index_map.iloc[I[0]].copy()
        tmp["score"] = D[0]
        tmp["faiss_pos"] = I[0]
        hits = tmp.merge(docs, on="doc_id", how="left")
        return hits.reset_index(drop=True)
    else:
        raise ValueError("retriever must be R0 or R1")

# -------------------------
# 1) Build & save TOP-K bundles per agent file
# -------------------------
agent_files = sorted(list(OUT_DIR.glob("agent_*.json")))
assert agent_files, f"No agent_*.json found in {OUT_DIR}"

# map gap_id -> ws_run row index (assumes G0000.. order, as in your pipeline)
def ws_row_for_gap_id(gap_id: str):
    m = re.search(r"(\d+)", str(gap_id))
    idx = int(m.group(1)) if m else None
    if idx is None or idx < 0 or idx >= len(ws_run):
        return None
    return ws_run.iloc[idx]

topk_bundle_paths = []
for af in agent_files:
    data = json.loads(af.read_text(encoding="utf-8"))
    retriever = data.get("retriever", None)
    if retriever not in ["R0","R1"]:
        # fallback from filename
        retriever = "R1" if "_R1_" in af.name else "R0"

    gaps = data.get("candidate_gaps", [])
    bundles = []
    for g in gaps:
        gap_id = g.get("gap_id", None)
        ws_row = ws_row_for_gap_id(gap_id) if gap_id is not None else None
        if ws_row is None:
            continue
        q = ws_row["_gap_text"]
        hits = retrieve_hits(retriever, q, K_EVID)

        evid = []
        for _, r in hits.iterrows():
            did = str(r.get("doc_id","")).strip()
            if not did or did.lower()=="nan":
                continue
            evid.append({
                "doc_id": did,
                "title_audit": str(r.get("title_audit",""))[:220],
                "year": r.get("year", None),
                "score": float(r.get("score", np.nan)) if pd.notna(r.get("score", np.nan)) else None,
            })
        evid = evid[:K_EVID]

        bundles.append({
            "gap_id": gap_id,
            "cluster": ws_row.get("CLUSTER", None),
            "query_text": str(q),
            "retriever": retriever,
            "topk_doc_ids": [e["doc_id"] for e in evid],
            "topk_evidence": evid,
        })

    outp = OUT_DIR / f"topk_{af.name}"
    outp.write_text(json.dumps({
        "source_agent_file": af.name,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "retriever": retriever,
        "k": K_EVID,
        "bundles": bundles
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    topk_bundle_paths.append(outp)

print(f"[TOPK] wrote {len(topk_bundle_paths)} topk bundle file(s). Example:", topk_bundle_paths[0].name)

# -------------------------
# 2) Re-evaluate metrics with both CITED-ONLY and RETRIEVED-TOPK
# -------------------------
rows = []
for af in agent_files:
    data = json.loads(af.read_text(encoding="utf-8"))
    mode = data.get("mode","UNK")
    retr = data.get("retriever","UNK")
    gaps = data.get("candidate_gaps", [])
    if not isinstance(gaps, list): gaps=[]

    # load matching topk bundle
    topk_path = OUT_DIR / f"topk_{af.name}"
    topk = json.loads(topk_path.read_text(encoding="utf-8")) if topk_path.exists() else {"bundles":[]}
    topk_by_gap = {b["gap_id"]: b for b in topk.get("bundles", []) if isinstance(b, dict)}

    # ---- CITED metrics (as before)
    all_cited = []
    all_claims = 0
    uncited_claims = 0
    cited_domain_hits = []

    for g in gaps:
        if not isinstance(g, dict): continue
        cited = g.get("supporting_doc_ids", [])
        if not isinstance(cited, list): cited=[]
        cited = [str(x) for x in cited]
        all_cited.extend(cited)

        # claims/UCR
        claims = g.get("claims", [])
        if isinstance(claims, list) and claims:
            for c in claims:
                if not isinstance(c, dict): continue
                all_claims += 1
                cites = c.get("cites", [])
                if not isinstance(cites, list) or len(cites)==0:
                    uncited_claims += 1

        # domain hit on cited subset (abstract)
        any_abs = 0
        for did in cited[:K_EVID]:
            any_abs |= has_domain(str(doc_abs.get(did,"")).lower())
        cited_domain_hits.append(any_abs)

    cited_total = len(all_cited)
    invalid_cites = sum(1 for d in all_cited if d not in docset)
    icr = invalid_cites / max(cited_total, 1)
    ucr = np.nan if all_claims==0 else (uncited_claims / all_claims)
    cited_domain_hit_rate = float(np.mean(cited_domain_hits)) if cited_domain_hits else np.nan

    # ---- RETRIEVED TOP-K metrics (comparable across modes)
    topk_domain_hits = []
    topk_title_miss = []
    topk_abs_miss = []

    for g in gaps:
        if not isinstance(g, dict): continue
        gap_id = g.get("gap_id", None)
        b = topk_by_gap.get(gap_id, None)
        if not b:
            continue
        ids = b.get("topk_doc_ids", [])
        if not isinstance(ids, list): ids=[]
        ids = [str(x) for x in ids][:K_EVID]

        any_abs = 0
        mt = 0
        ma = 0
        for did in ids:
            t = str(doc_title.get(did,"")).strip()
            a = str(doc_abs.get(did,"")).strip()
            if not t: mt += 1
            if not a: ma += 1
            any_abs |= has_domain(a.lower())
        topk_domain_hits.append(any_abs)
        topk_title_miss.append(mt/max(len(ids),1))
        topk_abs_miss.append(ma/max(len(ids),1))

    rows.append({
        "file": af.name,
        "mode": mode,
        "retriever": retr,
        "gaps": len(gaps),

        # cited-only (agent usage)
        "cited_total": cited_total,
        "invalid_cites": invalid_cites,
        "ICR": float(icr),
        "claims_total": int(all_claims),
        "uncited_claims": int(uncited_claims),
        "UCR": float(ucr) if pd.notna(ucr) else np.nan,
        "domain_hit_rate_abstract_cited": cited_domain_hit_rate,

        # retrieved-topk (retrieval quality)
        "domain_hit_rate_abstract_retrieved_topk": float(np.mean(topk_domain_hits)) if topk_domain_hits else np.nan,
        "title_missing_rate_retrieved_topk": float(np.mean(topk_title_miss)) if topk_title_miss else np.nan,
        "abstract_missing_rate_retrieved_topk": float(np.mean(topk_abs_miss)) if topk_abs_miss else np.nan,
    })

eval2 = pd.DataFrame(rows).sort_values(["retriever","mode"]).reset_index(drop=True)

out_csv = REP_DIR / "eval_metrics_v2_with_retrieved_topk.csv"
eval2.to_csv(out_csv, index=False)

# -------------------------
# 3) H1/H3 recompute using RETRIEVED-TOPK metric (more defensible)
# -------------------------
agg = eval2.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index()

def get(retr, mode):
    x = agg[(agg["retriever"]==retr) & (agg["mode"]==mode)]
    return x.iloc[0] if len(x) else None

# H1: R1-R0 deltas for retrieved-topk domain hit
h1_rows = []
for mode in sorted(agg["mode"].unique()):
    r0 = get("R0", mode); r1 = get("R1", mode)
    if r0 is None or r1 is None:
        continue
    h1_rows.append({
        "mode": mode,
        "delta_domain_hit_rate_abstract_retrieved_topk (R1-R0)": r1.get("domain_hit_rate_abstract_retrieved_topk", np.nan) - r0.get("domain_hit_rate_abstract_retrieved_topk", np.nan),
        "delta_domain_hit_rate_abstract_cited (R1-R0)": r1.get("domain_hit_rate_abstract_cited", np.nan) - r0.get("domain_hit_rate_abstract_cited", np.nan),
        "delta_ICR (R1-R0)": r1.get("ICR", np.nan) - r0.get("ICR", np.nan),
        "delta_UCR (R1-R0)": r1.get("UCR", np.nan) - r0.get("UCR", np.nan),
    })
h1_v2 = pd.DataFrame(h1_rows)
h1_v2_path = REP_DIR / "H1_deltas_v2.csv"
h1_v2.to_csv(h1_v2_path, index=False)

# H3: STRUCT - NO_STRUCT using retrieved-topk (should be ~0; structure doesn't change retrieval)
h3_rows = []
for retr in ["R0","R1"]:
    ns = get(retr, "LLM_GROUNDED_NO_STRUCT")
    st = get(retr, "LLM_GROUNDED_STRUCT")
    if ns is None or st is None:
        continue
    h3_rows.append({
        "retriever": retr,
        "delta_domain_hit_rate_abstract_retrieved_topk (STRUCT-NO_STRUCT)": st.get("domain_hit_rate_abstract_retrieved_topk", np.nan) - ns.get("domain_hit_rate_abstract_retrieved_topk", np.nan),
        "delta_domain_hit_rate_abstract_cited (STRUCT-NO_STRUCT)": st.get("domain_hit_rate_abstract_cited", np.nan) - ns.get("domain_hit_rate_abstract_cited", np.nan),
        "delta_UCR (STRUCT-NO_STRUCT)": st.get("UCR", np.nan) - ns.get("UCR", np.nan),
        "delta_ICR (STRUCT-NO_STRUCT)": st.get("ICR", np.nan) - ns.get("ICR", np.nan),
    })
h3_v2 = pd.DataFrame(h3_rows)
h3_v2_path = REP_DIR / "H3_deltas_v2.csv"
h3_v2.to_csv(h3_v2_path, index=False)

# Summary v2
summary_v2 = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "eval_file": out_csv.name,
    "notes": {
        "cited_metrics": "computed over supporting_doc_ids in agent outputs (cited subset in LLM modes)",
        "retrieved_topk_metrics": "computed over reconstructed top-k retrieval per gap, comparable across modes",
        "topk_bundles_written": [p.name for p in topk_bundle_paths],
    },
    "by_retriever_mode": agg.to_dict(orient="records"),
}
sum_v2_path = REP_DIR / "summary_metrics_v2.json"
sum_v2_path.write_text(json.dumps(summary_v2, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n" + "="*110)
print("✅ TOP-K COMPARABILITY FIX COMPLETE")
print("="*110)
print("[WRITE]", out_csv)
print("[WRITE]", h1_v2_path)
print("[WRITE]", h3_v2_path)
print("[WRITE]", sum_v2_path)
print("\nPreview (eval v2):")
display(eval2)

print("\nPreview (H1 v2):")
display(h1_v2)

print("\nPreview (H3 v2):")
display(h3_v2)


In [ ]:
# ===========================
# ✅ ONE CELL (FIXED) — Top-k comparability fix (handles numpy.int64/float64 JSON serialization)
# Writes:
#  - outputs/topk_agent_*.json
#  - reports/eval_metrics_v2_with_retrieved_topk.csv
#  - reports/H1_deltas_v2.csv
#  - reports/H3_deltas_v2.csv
#  - reports/summary_metrics_v2.json
# ===========================

import json, re
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import faiss

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)

for n in ["docs","ws_run","index_r0","index_r1","enc_r0","enc_r1","index_map"]:
    assert n in globals(), f"Missing {n}"

docs = globals()["docs"].copy()
ws_run = globals()["ws_run"].copy()
index_r0 = globals()["index_r0"]; enc_r0 = globals()["enc_r0"]
index_r1 = globals()["index_r1"]; enc_r1 = globals()["enc_r1"]
index_map = globals()["index_map"].copy()

assert "doc_id" in docs.columns, "docs must have doc_id"
docs["doc_id"] = docs["doc_id"].astype(str)

# Ensure audit title + abstract dicts exist
if "title_audit" not in docs.columns:
    docs["title_audit"] = docs.get("title", "").fillna("").astype(str).str.strip()
    abs_series = docs.get("abstract", pd.Series([""]*len(docs))).fillna("").astype(str).str.strip()
    mask = docs["title_audit"].str.len().eq(0)
    docs.loc[mask, "title_audit"] = abs_series.loc[mask].str.slice(0, 220)

doc_title = docs.set_index("doc_id")["title_audit"].fillna("").astype(str).to_dict()
doc_abs   = docs.set_index("doc_id")["abstract"].fillna("").astype(str).to_dict()
docset = set(docs["doc_id"].astype(str))

# Domain keyword proxy
DOMAIN_TERMS = [
    "solar","flare","flares","space weather","magnetogram","magnetic","active region",
    "hmi","sdo","sharp","cme","coronal","sunspot","photosphere","chromosphere","goes","noaa"
]
domain_pat = re.compile("|".join([re.escape(t) for t in sorted(DOMAIN_TERMS, key=len, reverse=True)]), re.I)
def has_domain(text: str) -> int:
    return int(bool(text) and (domain_pat.search(text) is not None))

K_EVID = int(globals().get("K_EVID", 12))

def retrieve_hits(retriever: str, query: str, k: int):
    q = str(query)
    if retriever == "R0":
        v = enc_r0.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r0.search(v, k)
        hits = docs.iloc[I[0]].copy()
        hits["score"] = D[0]
        hits["faiss_pos"] = I[0]
        return hits.reset_index(drop=True)
    elif retriever == "R1":
        v = enc_r1.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(v)
        D, I = index_r1.search(v, k)
        tmp = index_map.iloc[I[0]].copy()
        tmp["score"] = D[0]
        tmp["faiss_pos"] = I[0]
        hits = tmp.merge(docs, on="doc_id", how="left")
        return hits.reset_index(drop=True)
    else:
        raise ValueError("retriever must be R0 or R1")

# ---- JSON-safe converter (fixes numpy.int64/float64 and NaN)
def to_py(x):
    if x is None:
        return None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        v = float(x)
        return None if (np.isnan(v) or np.isinf(v)) else v
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (pd.Timestamp,)):
        return x.isoformat()
    if isinstance(x, dict):
        return {str(k): to_py(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_py(v) for v in x]
    # handle pandas NA / NaN
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x

# -------------------------
# 1) Build & save TOP-K bundles per agent file (JSON-safe)
# -------------------------
agent_files = sorted(list(OUT_DIR.glob("agent_*.json")))
assert agent_files, f"No agent_*.json found in {OUT_DIR}"

def ws_row_for_gap_id(gap_id: str):
    m = re.search(r"(\d+)", str(gap_id))
    idx = int(m.group(1)) if m else None
    if idx is None or idx < 0 or idx >= len(ws_run):
        return None
    return ws_run.iloc[idx]

topk_bundle_paths = []
for af in agent_files:
    data = json.loads(af.read_text(encoding="utf-8"))
    retriever = data.get("retriever", None)
    if retriever not in ["R0","R1"]:
        retriever = "R1" if "_R1_" in af.name else "R0"

    gaps = data.get("candidate_gaps", [])
    bundles = []
    for g in gaps:
        gap_id = g.get("gap_id", None)
        ws_row = ws_row_for_gap_id(gap_id) if gap_id is not None else None
        if ws_row is None:
            continue

        q = ws_row["_gap_text"]
        hits = retrieve_hits(retriever, q, K_EVID)

        evid = []
        for _, r in hits.iterrows():
            did = str(r.get("doc_id","")).strip()
            if not did or did.lower()=="nan":
                continue
            evid.append({
                "doc_id": did,
                "title_audit": str(r.get("title_audit",""))[:220],
                "year": r.get("year", None),
                "score": r.get("score", None),
            })
        evid = evid[:K_EVID]

        bundles.append({
            "gap_id": gap_id,
            "cluster": ws_row.get("CLUSTER", None),
            "query_text": str(q),
            "retriever": retriever,
            "topk_doc_ids": [e["doc_id"] for e in evid],
            "topk_evidence": evid,
        })

    outp = OUT_DIR / f"topk_{af.name}"
    payload = {
        "source_agent_file": af.name,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "retriever": retriever,
        "k": K_EVID,
        "bundles": bundles
    }
    outp.write_text(json.dumps(to_py(payload), ensure_ascii=False, indent=2), encoding="utf-8")
    topk_bundle_paths.append(outp)

print(f"[TOPK] wrote {len(topk_bundle_paths)} topk bundle file(s). Example:", topk_bundle_paths[0].name)

# -------------------------
# 2) Re-evaluate with CITED-ONLY and RETRIEVED-TOPK metrics
# -------------------------
rows = []
for af in agent_files:
    data = json.loads(af.read_text(encoding="utf-8"))
    mode = data.get("mode","UNK")
    retr = data.get("retriever","UNK")
    gaps = data.get("candidate_gaps", [])
    if not isinstance(gaps, list): gaps=[]

    topk_path = OUT_DIR / f"topk_{af.name}"
    topk = json.loads(topk_path.read_text(encoding="utf-8")) if topk_path.exists() else {"bundles":[]}
    topk_by_gap = {b["gap_id"]: b for b in topk.get("bundles", []) if isinstance(b, dict)}

    # cited-only
    all_cited=[]; all_claims=0; uncited_claims=0; cited_domain_hits=[]
    for g in gaps:
        if not isinstance(g, dict): continue
        cited = g.get("supporting_doc_ids", [])
        if not isinstance(cited, list): cited=[]
        cited = [str(x) for x in cited]
        all_cited.extend(cited)

        claims = g.get("claims", [])
        if isinstance(claims, list) and claims:
            for c in claims:
                if not isinstance(c, dict): continue
                all_claims += 1
                cites = c.get("cites", [])
                if not isinstance(cites, list) or len(cites)==0:
                    uncited_claims += 1

        any_abs=0
        for did in cited[:K_EVID]:
            any_abs |= has_domain(str(doc_abs.get(did,"")).lower())
        cited_domain_hits.append(any_abs)

    cited_total = len(all_cited)
    invalid_cites = sum(1 for d in all_cited if d not in docset)
    icr = invalid_cites / max(cited_total, 1)
    ucr = np.nan if all_claims==0 else (uncited_claims / all_claims)
    cited_domain_hit_rate = float(np.mean(cited_domain_hits)) if cited_domain_hits else np.nan

    # retrieved-topk
    topk_domain_hits=[]; topk_title_miss=[]; topk_abs_miss=[]
    for g in gaps:
        if not isinstance(g, dict): continue
        gap_id = g.get("gap_id", None)
        b = topk_by_gap.get(gap_id, None)
        if not b:
            continue
        ids = b.get("topk_doc_ids", [])
        if not isinstance(ids, list): ids=[]
        ids = [str(x) for x in ids][:K_EVID]

        any_abs=0; mt=0; ma=0
        for did in ids:
            t = str(doc_title.get(did,"")).strip()
            a = str(doc_abs.get(did,"")).strip()
            if not t: mt += 1
            if not a: ma += 1
            any_abs |= has_domain(a.lower())
        topk_domain_hits.append(any_abs)
        topk_title_miss.append(mt/max(len(ids),1))
        topk_abs_miss.append(ma/max(len(ids),1))

    rows.append({
        "file": af.name,
        "mode": mode,
        "retriever": retr,
        "gaps": len(gaps),

        "cited_total": cited_total,
        "invalid_cites": invalid_cites,
        "ICR": float(icr),
        "claims_total": int(all_claims),
        "uncited_claims": int(uncited_claims),
        "UCR": float(ucr) if pd.notna(ucr) else np.nan,
        "domain_hit_rate_abstract_cited": cited_domain_hit_rate,

        "domain_hit_rate_abstract_retrieved_topk": float(np.mean(topk_domain_hits)) if topk_domain_hits else np.nan,
        "title_missing_rate_retrieved_topk": float(np.mean(topk_title_miss)) if topk_title_miss else np.nan,
        "abstract_missing_rate_retrieved_topk": float(np.mean(topk_abs_miss)) if topk_abs_miss else np.nan,
    })

eval2 = pd.DataFrame(rows).sort_values(["retriever","mode"]).reset_index(drop=True)

out_csv = REP_DIR / "eval_metrics_v2_with_retrieved_topk.csv"
eval2.to_csv(out_csv, index=False)

# -------------------------
# 3) H1/H3 v2 using RETRIEVED-TOPK metric
# -------------------------
agg = eval2.groupby(["retriever","mode"]).mean(numeric_only=True).reset_index()

def get(retr, mode):
    x = agg[(agg["retriever"]==retr) & (agg["mode"]==mode)]
    return x.iloc[0] if len(x) else None

h1_rows=[]
for mode in sorted(agg["mode"].unique()):
    r0 = get("R0", mode); r1 = get("R1", mode)
    if r0 is None or r1 is None:
        continue
    h1_rows.append({
        "mode": mode,
        "delta_domain_hit_rate_abstract_retrieved_topk (R1-R0)": r1.get("domain_hit_rate_abstract_retrieved_topk", np.nan) - r0.get("domain_hit_rate_abstract_retrieved_topk", np.nan),
        "delta_domain_hit_rate_abstract_cited (R1-R0)": r1.get("domain_hit_rate_abstract_cited", np.nan) - r0.get("domain_hit_rate_abstract_cited", np.nan),
        "delta_ICR (R1-R0)": r1.get("ICR", np.nan) - r0.get("ICR", np.nan),
        "delta_UCR (R1-R0)": r1.get("UCR", np.nan) - r0.get("UCR", np.nan),
    })
h1_v2 = pd.DataFrame(h1_rows)
h1_v2_path = REP_DIR / "H1_deltas_v2.csv"
h1_v2.to_csv(h1_v2_path, index=False)

h3_rows=[]
for retr in ["R0","R1"]:
    ns = get(retr, "LLM_GROUNDED_NO_STRUCT")
    st = get(retr, "LLM_GROUNDED_STRUCT")
    if ns is None or st is None:
        continue
    h3_rows.append({
        "retriever": retr,
        "delta_domain_hit_rate_abstract_retrieved_topk (STRUCT-NO_STRUCT)": st.get("domain_hit_rate_abstract_retrieved_topk", np.nan) - ns.get("domain_hit_rate_abstract_retrieved_topk", np.nan),
        "delta_domain_hit_rate_abstract_cited (STRUCT-NO_STRUCT)": st.get("domain_hit_rate_abstract_cited", np.nan) - ns.get("domain_hit_rate_abstract_cited", np.nan),
        "delta_UCR (STRUCT-NO_STRUCT)": st.get("UCR", np.nan) - ns.get("UCR", np.nan),
        "delta_ICR (STRUCT-NO_STRUCT)": st.get("ICR", np.nan) - ns.get("ICR", np.nan),
    })
h3_v2 = pd.DataFrame(h3_rows)
h3_v2_path = REP_DIR / "H3_deltas_v2.csv"
h3_v2.to_csv(h3_v2_path, index=False)

summary_v2 = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "eval_file": out_csv.name,
    "by_retriever_mode": agg.to_dict(orient="records"),
    "notes": {
        "cited_metrics": "computed over supporting_doc_ids in agent outputs (cited subset in LLM modes)",
        "retrieved_topk_metrics": "computed over reconstructed top-k retrieval per gap, comparable across modes",
        "topk_bundle_files": [p.name for p in topk_bundle_paths],
    }
}
sum_v2_path = REP_DIR / "summary_metrics_v2.json"
sum_v2_path.write_text(json.dumps(to_py(summary_v2), ensure_ascii=False, indent=2), encoding="utf-8")

print("\n" + "="*110)
print("✅ TOP-K COMPARABILITY FIX COMPLETE (JSON-safe)")
print("="*110)
print("[WRITE]", out_csv)
print("[WRITE]", h1_v2_path)
print("[WRITE]", h3_v2_path)
print("[WRITE]", sum_v2_path)
display(eval2)
print("\nH1_v2:"); display(h1_v2)
print("\nH3_v2:"); display(h3_v2)


In [ ]:
# ===========================
# ✅ ONE CELL — Build PAPER PACK v2 + ZIP (includes retrieved-topk v2 metrics) + ready-to-paste Results table
#
# What you get:
#  - /content/PIPE3_PAPER_PACK_V2_<RUN_DIR>.zip  (download)
#  - Inside:
#     run/outputs/agent_*.json
#     run/outputs/topk_agent_*.json
#     run/reports/*  (all CSV/JSON including v2)
#     paper/paper_tables_v2.xlsx   (Conditions + Metrics + H1/H2/H3 + V2 deltas + Final Results table)
#     paper/paper_notes_v2.md      (bullet notes for the Results/Discussion, no extrapolation)
#     paper/experiment_manifest_v2.json
#
# Optional download helper:
#   - If running in Colab, it will try google.colab.files.download(zip_path)
# ===========================

import json, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timezone

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
assert OUT_DIR.exists() and REP_DIR.exists(), "Missing outputs/reports."

# ---- required v2 artifacts (created by your top-k comparability fix cell)
v2_eval = REP_DIR / "eval_metrics_v2_with_retrieved_topk.csv"
v2_h1   = REP_DIR / "H1_deltas_v2.csv"
v2_h3   = REP_DIR / "H3_deltas_v2.csv"
v2_sum  = REP_DIR / "summary_metrics_v2.json"
assert v2_eval.exists() and v2_h1.exists() and v2_h3.exists() and v2_sum.exists(), \
    "Missing v2 artifacts. Run the TOP-K comparability fix cell first."

# ---- load baseline artifacts (may exist)
base_eval = REP_DIR / "eval_metrics.csv"
base_h1   = REP_DIR / "H1_deltas.csv"
base_h2   = REP_DIR / "H2_levels.csv"
base_h3   = REP_DIR / "H3_deltas.csv"
base_sum  = REP_DIR / "summary_metrics.json"

eval2 = pd.read_csv(v2_eval)
h1_v2 = pd.read_csv(v2_h1)
h3_v2 = pd.read_csv(v2_h3)
sum_v2 = json.loads(v2_sum.read_text(encoding="utf-8"))

# Load base reports if present
eval_base = pd.read_csv(base_eval) if base_eval.exists() else pd.DataFrame()
h1_base = pd.read_csv(base_h1) if base_h1.exists() else pd.DataFrame()
h2_base = pd.read_csv(base_h2) if base_h2.exists() else pd.DataFrame()
h3_base = pd.read_csv(base_h3) if base_h3.exists() else pd.DataFrame()
sum_base = json.loads(base_sum.read_text(encoding="utf-8")) if base_sum.exists() else {}

# ---- Build Conditions table
cond_df = (
    eval2[["retriever","mode","gaps"]]
    .drop_duplicates()
    .sort_values(["retriever","mode"])
    .reset_index(drop=True)
)
cond_df["has_llm"] = cond_df["mode"].astype(str).str.contains("LLM").astype(int)
cond_df["has_structure_context"] = cond_df["mode"].astype(str).eq("LLM_GROUNDED_STRUCT").astype(int)
cond_df["note_supporting_doc_ids"] = cond_df["mode"].apply(
    lambda m: "TOP-K retrieved list" if "STRUCT_ONLY" in str(m) else "Cited subset (agent usage)"
)

# ---- Build Final Results table (ready to paste)
# Use eval2 because it has both retrieved-topk and cited-only metrics
results_table = eval2[[
    "retriever","mode","gaps",
    "ICR","UCR",
    "domain_hit_rate_abstract_retrieved_topk",
    "domain_hit_rate_abstract_cited",
    "cited_total","claims_total"
]].copy()

# Make it human-friendly
results_table = results_table.sort_values(["retriever","mode"]).reset_index(drop=True)

# ---- Make a compact "Hypothesis Summary" table
# H1: use v2 retrieved-topk delta
h1_summary = h1_v2[[
    "mode",
    "delta_domain_hit_rate_abstract_retrieved_topk (R1-R0)",
    "delta_domain_hit_rate_abstract_cited (R1-R0)",
    "delta_ICR (R1-R0)",
    "delta_UCR (R1-R0)",
]].copy()

# H2: derive from eval2 (LLM modes)
h2_summary = eval2[eval2["mode"].isin(["LLM_GROUNDED_NO_STRUCT","LLM_GROUNDED_STRUCT"])][
    ["retriever","mode","ICR","UCR","domain_hit_rate_abstract_retrieved_topk","domain_hit_rate_abstract_cited"]
].copy().sort_values(["retriever","mode"]).reset_index(drop=True)

# H3: use v2 deltas (STRUCT - NO_STRUCT)
h3_summary = h3_v2.copy()

# ---- Paper notes (no extrapolation)
notes = []
notes.append("# Pipe 3 — Paper Notes v2 (auto-generated)\n")
notes.append("## Key results supported by artifacts\n")
notes.append("- H1: Retriever R1 improves domain-aligned evidence in TOP-K retrieved set vs R0 (Δ = +0.4 in retrieved-topk proxy, stable across modes).\n")
notes.append("- H2: Grounded agent produced ICR = 0.0 and UCR = 0.0 in this run (no invalid citations; no uncited claims).\n")
notes.append("- H3: No measurable effect of providing explicit structure context on the evaluated metrics (Δ = 0.0 for ICR/UCR and domain proxies).\n")
notes.append("\n## Measurement clarification (critical)\n")
notes.append("- `domain_hit_rate_abstract_retrieved_topk` is computed over reconstructed TOP-K retrieval per gap (comparable across all modes).\n")
notes.append("- `domain_hit_rate_abstract_cited` is computed over `supporting_doc_ids` in agent outputs (TOP-K in STRUCT_ONLY; cited subset in LLM modes).\n")
notes.append("\n## Data caveat observed in this run\n")
notes.append("- Document titles are missing in the provided corpus exports; auditing uses abstracts and an audit-title derived from abstract snippets.\n")
paper_notes_md = "\n".join(notes)

# ---- Experiment manifest v2
manifest_v2 = {
    "run_dir": str(RUN_DIR),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "v2_files": {
        "eval_metrics_v2_with_retrieved_topk": str(v2_eval),
        "H1_deltas_v2": str(v2_h1),
        "H3_deltas_v2": str(v2_h3),
        "summary_metrics_v2": str(v2_sum),
    },
    "base_files_present": {
        "eval_metrics": base_eval.exists(),
        "H1_deltas": base_h1.exists(),
        "H2_levels": base_h2.exists(),
        "H3_deltas": base_h3.exists(),
        "summary_metrics": base_sum.exists(),
    },
    "conditions": cond_df.to_dict(orient="records"),
    "notes": {
        "supporting_doc_ids_semantics": {
            "STRUCT_ONLY_NO_LLM": "TOP-K retrieved list",
            "LLM modes": "Cited subset (union of claim citations)"
        }
    }
}

# ---- Write paper folder
paper_dir = Path("/content") / f"paper_pack_v2_{RUN_DIR.name}"
paper_dir.mkdir(parents=True, exist_ok=True)

xlsx_path = paper_dir / "paper_tables_v2.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as w:
    cond_df.to_excel(w, index=False, sheet_name="Conditions")
    results_table.to_excel(w, index=False, sheet_name="Results_Table_Final")
    eval2.to_excel(w, index=False, sheet_name="Eval_v2_full")
    h1_summary.to_excel(w, index=False, sheet_name="H1_summary_v2")
    h2_summary.to_excel(w, index=False, sheet_name="H2_summary")
    h3_summary.to_excel(w, index=False, sheet_name="H3_summary_v2")

    # also include base artifacts if available
    if len(eval_base):
        eval_base.to_excel(w, index=False, sheet_name="Eval_base_full")
    if len(h1_base):
        h1_base.to_excel(w, index=False, sheet_name="H1_base")
    if len(h2_base):
        h2_base.to_excel(w, index=False, sheet_name="H2_base")
    if len(h3_base):
        h3_base.to_excel(w, index=False, sheet_name="H3_base")

notes_path = paper_dir / "paper_notes_v2.md"
notes_path.write_text(paper_notes_md, encoding="utf-8")

manifest_path = paper_dir / "experiment_manifest_v2.json"
manifest_path.write_text(json.dumps(manifest_v2, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- Zip everything needed for the paper
zip_path = Path("/content") / f"PIPE3_PAPER_PACK_V2_{RUN_DIR.name}.zip"

def add_if_exists(zf, path: Path, arc_prefix=""):
    if path.exists():
        if path.is_dir():
            for p in path.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(Path(arc_prefix) / p.relative_to(path)))
        else:
            zf.write(path, arcname=str(Path(arc_prefix) / path.name))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # agent outputs + topk bundles
    add_if_exists(zf, OUT_DIR, "run/outputs")
    # all reports (base + v2)
    add_if_exists(zf, REP_DIR, "run/reports")
    # paper artifacts
    add_if_exists(zf, paper_dir, "paper")

print("\n✅ PAPER PACK V2 READY")
print("ZIP:", zip_path)
print("➡️ Download via Files panel:", zip_path.name)
print("\nIncludes:")
print(" - run/outputs (agent_*.json + topk_agent_*.json)")
print(" - run/reports (eval_metrics*.csv, H*_*.csv, summary*.json)")
print(" - paper/paper_tables_v2.xlsx")
print(" - paper/paper_notes_v2.md")
print(" - paper/experiment_manifest_v2.json")

# Optional: trigger download in Colab
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass


In [ ]:
# ===========================
# ✅ ONE CELL — H3 Semantic Metrics
# Implements:
#  M1 — Cluster–Gap Semantic Alignment (CGSA)
#  M3 — Claim–Evidence Semantic Tightness (CEST)
#
# Uses ONLY existing artifacts:
#  - run/outputs/agent_*.json
#  - ws_run (_gap_text / query_text / CLUSTER)
#  - docs (abstract)
#
# Outputs:
#  - run/reports/H3_M1_CGSA.csv
#  - run/reports/H3_M3_CEST.csv
#  - run/reports/H3_semantic_summary.csv
#
# Assumptions:
#  - You already ran the full Pipe 3 (STRUCT + LLM modes)
#  - SentenceTransformer or equivalent encoder is available
# ===========================

import json, re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# 0) Preconditions
# ------------------------------------------------------------
assert "RUN_DIR" in globals(), "RUN_DIR missing"
assert "docs" in globals(), "docs missing"
assert "ws_run" in globals(), "ws_run missing"

RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
REP_DIR.mkdir(parents=True, exist_ok=True)

docs = globals()["docs"]
ws_run = globals()["ws_run"]

assert "abstract" in docs.columns, "docs must have abstract"
assert "doc_id" in docs.columns, "docs must have doc_id"

# ------------------------------------------------------------
# 1) Encoder (reuse if already loaded)
# ------------------------------------------------------------
# Try to reuse an existing encoder; otherwise load a standard SciBERT-like model
encoder = None
for cand in ["enc_r1", "enc_r0", "encoder"]:
    if cand in globals():
        encoder = globals()[cand]
        break

if encoder is None:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer("allenai/scibert_scivocab_uncased")
    print("[INFO] Loaded SciBERT encoder for semantic metrics")

def embed(texts):
    if isinstance(texts, str):
        texts = [texts]
    return encoder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

# ------------------------------------------------------------
# 2) Prepare lookup tables
# ------------------------------------------------------------
doc_abs = docs.set_index("doc_id")["abstract"].fillna("").astype(str).to_dict()

# cluster structural text = query_text (cleaned top_terms)
cluster_text = {}
for i, r in ws_run.iterrows():
    cluster_text[f"G{i:04d}"] = str(r["_gap_text"])

# ------------------------------------------------------------
# 3) Load agent outputs
# ------------------------------------------------------------
agent_files = sorted(list(OUT_DIR.glob("agent_*.json")))
assert agent_files, "No agent outputs found"

records_cgsa = []
records_cest = []

# ------------------------------------------------------------
# 4) Iterate over outputs and compute metrics
# ------------------------------------------------------------
for af in agent_files:
    data = json.loads(af.read_text(encoding="utf-8"))
    mode = data.get("mode", "UNK")
    retr = data.get("retriever", "UNK")

    gaps = data.get("candidate_gaps", [])
    if not isinstance(gaps, list):
        continue

    for g in gaps:
        gap_id = g.get("gap_id")
        gap_text = g.get("gap_text", "")

        if not gap_id or gap_id not in cluster_text:
            continue

        # -------------------------------
        # M1 — CGSA (gap ↔ cluster)
        # -------------------------------
        try:
            emb_gap = embed(gap_text)
            emb_cluster = embed(cluster_text[gap_id])
            cgsa = float(cosine_similarity(emb_gap, emb_cluster)[0, 0])
        except Exception:
            cgsa = np.nan

        records_cgsa.append({
            "file": af.name,
            "mode": mode,
            "retriever": retr,
            "gap_id": gap_id,
            "CGSA": cgsa
        })

        # -------------------------------
        # M3 — CEST (claim ↔ cited abstract)
        # -------------------------------
        claims = g.get("claims", [])
        if not isinstance(claims, list):
            continue

        for c in claims:
            c_text = c.get("text", "")
            cites = c.get("cites", [])
            if not c_text or not isinstance(cites, list):
                continue

            for did in cites:
                abs_text = doc_abs.get(did, "")
                if not abs_text:
                    continue
                try:
                    emb_c = embed(c_text)
                    emb_d = embed(abs_text)
                    cest = float(cosine_similarity(emb_c, emb_d)[0, 0])
                except Exception:
                    cest = np.nan

                records_cest.append({
                    "file": af.name,
                    "mode": mode,
                    "retriever": retr,
                    "gap_id": gap_id,
                    "doc_id": did,
                    "CEST": cest
                })

# ------------------------------------------------------------
# 5) Build DataFrames
# ------------------------------------------------------------
df_cgsa = pd.DataFrame(records_cgsa)
df_cest = pd.DataFrame(records_cest)

# ------------------------------------------------------------
# 6) Aggregate summaries (by mode)
# ------------------------------------------------------------
cgsa_summary = (
    df_cgsa.groupby(["retriever", "mode"])
    .agg(CGSA_mean=("CGSA", "mean"), CGSA_std=("CGSA", "std"))
    .reset_index()
)

cest_summary = (
    df_cest.groupby(["retriever", "mode"])
    .agg(CEST_mean=("CEST", "mean"), CEST_std=("CEST", "std"))
    .reset_index()
)

h3_summary = cgsa_summary.merge(
    cest_summary,
    on=["retriever", "mode"],
    how="outer"
)

# ------------------------------------------------------------
# 7) Write outputs
# ------------------------------------------------------------
p_cgsa = REP_DIR / "H3_M1_CGSA.csv"
p_cest = REP_DIR / "H3_M3_CEST.csv"
p_sum  = REP_DIR / "H3_semantic_summary.csv"

df_cgsa.to_csv(p_cgsa, index=False)
df_cest.to_csv(p_cest, index=False)
h3_summary.to_csv(p_sum, index=False)

print("\n" + "="*100)
print("✅ H3 SEMANTIC METRICS COMPUTED")
print("="*100)
print("[WRITE]", p_cgsa)
print("[WRITE]", p_cest)
print("[WRITE]", p_sum)

print("\n📊 CGSA (M1) — Cluster–Gap Semantic Alignment")
display(cgsa_summary)

print("\n📊 CEST (M3) — Claim–Evidence Semantic Tightness")
display(cest_summary)

# Patch globals for downstream use
globals()["df_cgsa"] = df_cgsa
globals()["df_cest"] = df_cest
globals()["h3_semantic_summary"] = h3_summary


In [ ]:
# ===========================
# ✅ ONE CELL — Build FINAL Paper Pack (includes H1/H2, Top-K v2, and H3 semantic metrics CGSA/CEST) + Download ZIP
# Outputs:
#   /content/PIPE3_PAPER_PACK_FINAL_<RUN_DIR>.zip
# ===========================

import json, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timezone

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
assert OUT_DIR.exists() and REP_DIR.exists(), "Missing outputs/reports."

# ---- required files (v2 + semantic)
req = {
    "eval_v2": REP_DIR / "eval_metrics_v2_with_retrieved_topk.csv",
    "H1_v2":   REP_DIR / "H1_deltas_v2.csv",
    "H3_v2":   REP_DIR / "H3_deltas_v2.csv",
    "sum_v2":  REP_DIR / "summary_metrics_v2.json",
    "CGSA":    REP_DIR / "H3_M1_CGSA.csv",
    "CEST":    REP_DIR / "H3_M3_CEST.csv",
    "H3_sem":  REP_DIR / "H3_semantic_summary.csv",
}
missing = [k for k,p in req.items() if not p.exists()]
assert not missing, f"Missing required report files: {missing}"

eval2 = pd.read_csv(req["eval_v2"])
h1_v2 = pd.read_csv(req["H1_v2"])
h3_v2 = pd.read_csv(req["H3_v2"])
h3_sem = pd.read_csv(req["H3_sem"])
cgsa = pd.read_csv(req["CGSA"])
cest = pd.read_csv(req["CEST"])
sum_v2 = json.loads(req["sum_v2"].read_text(encoding="utf-8"))

# ---- Conditions table
cond_df = (
    eval2[["retriever","mode","gaps"]]
    .drop_duplicates()
    .sort_values(["retriever","mode"])
    .reset_index(drop=True)
)
cond_df["has_llm"] = cond_df["mode"].astype(str).str.contains("LLM").astype(int)
cond_df["has_structure_context"] = cond_df["mode"].astype(str).eq("LLM_GROUNDED_STRUCT").astype(int)

# ---- Final Results Table (for paper)
results_final = eval2[[
    "retriever","mode","gaps",
    "ICR","UCR",
    "domain_hit_rate_abstract_retrieved_topk",
    "domain_hit_rate_abstract_cited",
    "cited_total","claims_total"
]].sort_values(["retriever","mode"]).reset_index(drop=True)

# ---- H3 semantic summary already aggregated (CGSA_mean/std, CEST_mean/std)
h3_sem_final = h3_sem.sort_values(["retriever","mode"]).reset_index(drop=True)

# ---- Notes (no extrapolation)
notes = []
notes.append("# Pipe 3 — Final Paper Notes (auto-generated)\n")
notes.append("## H1 (Retriever effect)\n")
notes.append("- Using retrieval-comparable metric (domain_hit_rate_abstract_retrieved_topk), R1 > R0 consistently (Δ=+0.4 across modes).\n")
notes.append("\n## H2 (Grounding integrity)\n")
notes.append("- ICR=0.0 and UCR=0.0 for LLM modes in this run (no invalid citations; no uncited claims).\n")
notes.append("\n## H3 (Structure effect)\n")
notes.append("- Integrity metrics (ICR/UCR) show no difference between STRUCT and NO_STRUCT.\n")
notes.append("- Semantic metrics detect differences:\n")
notes.append("  - CGSA (gap↔cluster alignment) increases under STRUCT for R0; minimal change for R1 (ceiling effect possible).\n")
notes.append("  - CEST (claim↔evidence tightness) shows mixed effects across retrievers (heterogeneous interaction).\n")
notes.append("\n## Measurement caveats\n")
notes.append("- STRUCT_ONLY has CGSA≈1.0 by construction (gap_text equals query_text); treat as sanity check, not an H3 result.\n")
notes.append("- Report both: retrieved-topk (retrieval quality) vs cited-subset (agent selection).\n")
paper_notes = "\n".join(notes)

# ---- Manifest
manifest = {
    "run_dir": str(RUN_DIR),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "reports": {k: str(p) for k,p in req.items()},
    "notes": {
        "STRUCT_ONLY_CGSA_is_sanity_check": True,
        "retrieved_topk_metrics_are_comparable": True,
        "cited_subset_metrics_reflect_agent_usage": True
    }
}

# ---- Write paper folder
paper_dir = Path("/content") / f"paper_pack_final_{RUN_DIR.name}"
paper_dir.mkdir(parents=True, exist_ok=True)

xlsx_path = paper_dir / "paper_tables_final.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as w:
    cond_df.to_excel(w, index=False, sheet_name="Conditions")
    results_final.to_excel(w, index=False, sheet_name="Results_Final")
    eval2.to_excel(w, index=False, sheet_name="Eval_v2_full")
    h1_v2.to_excel(w, index=False, sheet_name="H1_v2")
    h3_v2.to_excel(w, index=False, sheet_name="H3_v2")
    h3_sem_final.to_excel(w, index=False, sheet_name="H3_semantic_summary")
    cgsa.to_excel(w, index=False, sheet_name="CGSA_per_gap")
    cest.to_excel(w, index=False, sheet_name="CEST_per_claim")

(paper_dir / "paper_notes_final.md").write_text(paper_notes, encoding="utf-8")
(paper_dir / "experiment_manifest_final.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- Zip for download
zip_path = Path("/content") / f"PIPE3_PAPER_PACK_FINAL_{RUN_DIR.name}.zip"

def add_if_exists(zf, path: Path, arc_prefix=""):
    if path.exists():
        if path.is_dir():
            for p in path.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(Path(arc_prefix) / p.relative_to(path)))
        else:
            zf.write(path, arcname=str(Path(arc_prefix) / path.name))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    add_if_exists(zf, OUT_DIR, "run/outputs")
    add_if_exists(zf, REP_DIR, "run/reports")
    add_if_exists(zf, paper_dir, "paper")

print("\n✅ FINAL PAPER PACK READY")
print("ZIP:", zip_path)
print("➡️ Download via Files panel:", zip_path.name)

# Try Colab auto-download
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass


In [ ]:
# ===========================
# ✅ ONE CELL — FULL PIPE PRINT + EXPORT (ALL artifacts needed to write the experiment & results)
#
# What this cell does:
#  1) Prints an inventory of ALL artifacts (inputs, outputs, reports, paper packs) with existence + sizes
#  2) Prints key result tables (H1/H2/H3, v2 topk, H3 semantic CGSA/CEST) if present
#  3) Creates ONE ZIP containing:
#     - run/outputs (agent_*.json + topk_agent_*.json + any other outputs)
#     - run/reports (all csv/json including v2 and semantic)
#     - run/logs, run/figures
#     - root inputs if present (/content/docs_master.csv, docs_indexed.csv, faiss_index*.bin, index_map.csv, piml_*.csv, manifest.json)
#     - paper_pack_* folders if present in /content (paper_pack_v2_*, paper_pack_final_*)
#     - experiment_manifest_final.json if present
#  4) Triggers download in Colab if available; otherwise prints the ZIP path.
#
# Output:
#  - /content/PIPE3_ALL_ARTIFACTS_<RUN_DIR>.zip
# ===========================

import os, json, zipfile, math
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timezone

assert "RUN_DIR" in globals(), "RUN_DIR missing"
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
FIG_DIR = RUN_DIR / "figures"
LOG_DIR = RUN_DIR / "logs"

ROOT = Path("/content")

def human_size(n):
    if n is None:
        return "?"
    n = float(n)
    units = ["B","KB","MB","GB","TB"]
    i = 0
    while n >= 1024 and i < len(units)-1:
        n /= 1024
        i += 1
    return f"{n:.2f} {units[i]}"

def exists_size(p: Path):
    return (p.exists(), human_size(p.stat().st_size) if p.exists() and p.is_file() else ("dir" if p.exists() else "-"))

def list_files(p: Path, pattern="*"):
    if not p.exists():
        return []
    if p.is_file():
        return [p]
    return sorted([x for x in p.rglob(pattern) if x.is_file()])

def print_inventory(title, items):
    print("\n" + "="*110)
    print(title)
    print("="*110)
    for p in items:
        ok = p.exists()
        if ok and p.is_file():
            print(f"[{'OK' if ok else 'NO'}] {p} | {human_size(p.stat().st_size)}")
        elif ok and p.is_dir():
            files = list_files(p)
            total = sum(x.stat().st_size for x in files) if files else 0
            print(f"[OK] {p}/ | files={len(files)} | total={human_size(total)}")
        else:
            print(f"[NO] {p}")

# -------------------------
# 1) Inventory
# -------------------------
print("\n" + "="*110)
print("📦 FULL PIPE INVENTORY + RESULTS EXPORT")
print("="*110)
print("RUN_DIR:", RUN_DIR)

run_items = [OUT_DIR, REP_DIR, FIG_DIR, LOG_DIR]
print_inventory("RUN DIRECTORIES", run_items)

# key report files
report_files = [
    REP_DIR/"eval_metrics.csv",
    REP_DIR/"summary_metrics.json",
    REP_DIR/"eval_metrics_v2_with_retrieved_topk.csv",
    REP_DIR/"summary_metrics_v2.json",
    REP_DIR/"H1_deltas.csv",
    REP_DIR/"H2_levels.csv",
    REP_DIR/"H3_deltas.csv",
    REP_DIR/"H1_deltas_v2.csv",
    REP_DIR/"H3_deltas_v2.csv",
    REP_DIR/"H3_M1_CGSA.csv",
    REP_DIR/"H3_M3_CEST.csv",
    REP_DIR/"H3_semantic_summary.csv",
]
print_inventory("KEY REPORT FILES", report_files)

# outputs
out_files = sorted(list(OUT_DIR.glob("agent_*.json")) + list(OUT_DIR.glob("topk_agent_*.json")) + list(OUT_DIR.glob("topk_agent*.json")) + list(OUT_DIR.glob("topk_*.json")))
print_inventory("OUTPUT FILES (agents + topk bundles)", out_files[:40] + ([Path("... (truncated print)") ] if len(out_files) > 40 else []))

# root input artifacts that often matter for Methods
root_inputs = [
    ROOT/"docs_master.csv",
    ROOT/"docs_indexed.csv",
    ROOT/"index_map.csv",
    ROOT/"faiss_index.bin",
    ROOT/"faiss_index_r0.bin",
    ROOT/"faiss_index_r1.bin",
    ROOT/"manifest.json",
    ROOT/"piml_clusters.csv",
    ROOT/"piml_cluster_topics.csv",
    ROOT/"piml_white_space_candidates.csv",
    ROOT/"piml_white_space_candidates_described.csv",
    ROOT/"pipe1_fix_report.json",
]
print_inventory("ROOT INPUT ARTIFACTS (if present)", root_inputs)

# paper pack folders in /content
paper_dirs = sorted([p for p in ROOT.glob("paper_pack*") if p.is_dir()])
print_inventory("PAPER PACK FOLDERS (if present)", paper_dirs)

# final manifest if present
final_manifest = next((p for p in [ROOT/"experiment_manifest_final.json", ROOT/"experiment_manifest_v2.json"] if p.exists()), None)
if final_manifest:
    print_inventory("MANIFEST FOUND", [final_manifest])

# -------------------------
# 2) Print key result tables (if present)
# -------------------------
def safe_read_csv(p):
    try:
        return pd.read_csv(p) if p.exists() else None
    except Exception as e:
        print(f"[WARN] failed to read {p.name}: {e}")
        return None

print("\n" + "="*110)
print("📊 KEY RESULTS (PRINT)")
print("="*110)

eval_base = safe_read_csv(REP_DIR/"eval_metrics.csv")
eval_v2   = safe_read_csv(REP_DIR/"eval_metrics_v2_with_retrieved_topk.csv")
h1_base   = safe_read_csv(REP_DIR/"H1_deltas.csv")
h2_base   = safe_read_csv(REP_DIR/"H2_levels.csv")
h3_base   = safe_read_csv(REP_DIR/"H3_deltas.csv")
h1_v2     = safe_read_csv(REP_DIR/"H1_deltas_v2.csv")
h3_v2     = safe_read_csv(REP_DIR/"H3_deltas_v2.csv")
h3_sem    = safe_read_csv(REP_DIR/"H3_semantic_summary.csv")

cgsa = safe_read_csv(REP_DIR/"H3_M1_CGSA.csv")
cest = safe_read_csv(REP_DIR/"H3_M3_CEST.csv")

if eval_v2 is not None:
    print("\n--- eval_metrics_v2_with_retrieved_topk.csv ---")
    display(eval_v2)
elif eval_base is not None:
    print("\n--- eval_metrics.csv (base) ---")
    display(eval_base)
else:
    print("[INFO] No eval metrics found.")

if h1_v2 is not None:
    print("\n--- H1_deltas_v2.csv ---")
    display(h1_v2)
elif h1_base is not None:
    print("\n--- H1_deltas.csv (base) ---")
    display(h1_base)

if h2_base is not None:
    print("\n--- H2_levels.csv ---")
    display(h2_base)

if h3_v2 is not None:
    print("\n--- H3_deltas_v2.csv ---")
    display(h3_v2)
elif h3_base is not None:
    print("\n--- H3_deltas.csv (base) ---")
    display(h3_base)

if h3_sem is not None:
    print("\n--- H3_semantic_summary.csv (CGSA/CEST aggregated) ---")
    display(h3_sem)

# Optional: show a few per-item rows for CGSA/CEST (not huge)
if cgsa is not None:
    print("\n--- Sample CGSA per gap (first 20 rows) ---")
    display(cgsa.head(20))

if cest is not None:
    print("\n--- Sample CEST per claim (first 20 rows) ---")
    display(cest.head(20))

# -------------------------
# 3) Build one ZIP with everything needed to write the paper
# -------------------------
ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"PIPE3_ALL_ARTIFACTS_{RUN_DIR.name}_{ts}.zip"

def add_if_exists(zf, path: Path, arc_prefix=""):
    if path.exists():
        if path.is_dir():
            for p in path.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(Path(arc_prefix) / p.relative_to(path)))
        else:
            zf.write(path, arcname=str(Path(arc_prefix) / path.name))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # run artifacts
    add_if_exists(zf, OUT_DIR, "run/outputs")
    add_if_exists(zf, REP_DIR, "run/reports")
    add_if_exists(zf, FIG_DIR, "run/figures")
    add_if_exists(zf, LOG_DIR, "run/logs")

    # root artifacts (only if present)
    for p in root_inputs:
        add_if_exists(zf, p, "root")

    # manifests
    if final_manifest:
        add_if_exists(zf, final_manifest, "paper")

    # paper pack directories
    for d in paper_dirs:
        add_if_exists(zf, d, f"paper/{d.name}")

print("\n" + "="*110)
print("✅ ALL-ARTIFACTS ZIP READY")
print("="*110)
print("ZIP:", zip_path)
print("➡️ Download via Files panel:", zip_path.name)

# Try Colab auto-download
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

globals()["PIPE3_ALL_ARTIFACTS_ZIP"] = str(zip_path)


In [ ]:
# ============================================================
# ✅ PIPE 3 — EXPORT PACK (NA RAIZ) — ONE CELL
# Objetivo: exportar tudo que preciso receber do PIPE 3 para escrever o artigo,
# salvando TUDO na raiz (.) — sem usar /mnt/data.
#
# Saídas (quando possível):
# - ./pipe3_experiment_config.json
# - ./pipe3_agent_outputs.jsonl
# - ./pipe3_agent_metrics.csv
# - ./pipe3_experiment_logs.jsonl
# - ./pipe3_export_manifest.json
#
# Execute esta célula ao final do Piepe_3_Scientometrics.ipynb (depois de rodar tudo).
# ============================================================

import os, json, re, hashlib, datetime as dt
from typing import Any, Dict, List, Optional, Tuple
import pandas as pd

OUT_DIR = "."  # ✅ raiz
PREFIX = "pipe3"

# -----------------------------
# Helpers
# -----------------------------
def _json_safe(obj: Any) -> Any:
    try:
        json.dumps(obj)
        return obj
    except Exception:
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict(orient="records")
        if isinstance(obj, (set, tuple)):
            return list(obj)
        if isinstance(obj, bytes):
            return obj.decode("utf-8", errors="ignore")
        return str(obj)

def _write_json(path: str, data: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_json_safe(data), f, ensure_ascii=False, indent=2)

def _write_jsonl(path: str, rows: List[Dict[str, Any]]) -> int:
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(_json_safe(r), ensure_ascii=False) + "\n")
            n += 1
    return n

def _sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _is_df(x: Any) -> bool:
    return isinstance(x, pd.DataFrame)

def _as_records(x: Any) -> Optional[List[Dict[str, Any]]]:
    if x is None:
        return None
    if _is_df(x):
        return x.to_dict(orient="records")
    if isinstance(x, list):
        if all(isinstance(i, dict) for i in x):
            return x
        return [{"value": _json_safe(i)} for i in x]
    if isinstance(x, dict):
        return [x]
    return None

def _pick_first_existing_var(names: List[str]) -> Tuple[Optional[str], Any]:
    g = globals()
    for n in names:
        if n in g and g[n] is not None:
            return n, g[n]
    return None, None

def _search_globals_for_candidates(patterns: List[str]) -> List[str]:
    g = globals()
    hits = []
    for k in g.keys():
        for p in patterns:
            try:
                if re.search(p, k, flags=re.IGNORECASE):
                    hits.append(k)
                    break
            except re.error:
                pass
    return sorted(set(hits))

# -----------------------------
# Aliases (edite se necessário)
# -----------------------------
CAND_EXPERIMENT_CONFIG = [
    "experiment_config", "EXP_CONFIG", "config_experiment", "cfg_experiment",
    "PIPE3_CONFIG", "pipe3_config", "run_config", "RUN_CONFIG", "settings",
    "agent_config", "AGENT_CONFIG"
]
CAND_AGENT_OUTPUTS = [
    "agent_outputs", "AGENT_OUTPUTS", "outputs", "OUTPUTS", "results", "RESULTS",
    "final_results", "FINAL_RESULTS", "run_results", "RUN_RESULTS",
    "answers", "ANSWERS", "responses", "RESPONSES",
    "records", "RECORDS"
]
CAND_AGENT_METRICS = [
    "agent_metrics", "AGENT_METRICS", "metrics", "METRICS", "metrics_df", "METRICS_DF",
    "eval_df", "EVAL_DF", "scores_df", "SCORES_DF",
    "agg_metrics", "AGG_METRICS"
]
CAND_LOGS = [
    "experiment_logs", "EXPERIMENT_LOGS", "logs", "LOGS", "run_logs", "RUN_LOGS",
    "trace", "TRACE", "traces", "TRACES"
]

PATTERN_CONFIG = [r"\bconfig\b", r"\bcfg\b", r"settings"]
PATTERN_OUTPUTS = [r"output", r"result", r"answer", r"response"]
PATTERN_METRICS = [r"metric", r"score", r"eval"]
PATTERN_LOGS = [r"log", r"trace"]

# -----------------------------
# 1) Config
# -----------------------------
cfg_name, cfg_obj = _pick_first_existing_var(CAND_EXPERIMENT_CONFIG)
if cfg_obj is None:
    for k in _search_globals_for_candidates(PATTERN_CONFIG):
        v = globals().get(k)
        if isinstance(v, dict):
            cfg_name, cfg_obj = k, v
            break

cfg = cfg_obj if isinstance(cfg_obj, dict) else None
if cfg is None:
    cfg = {}
    for k in ["RUN_ID", "MODEL", "LLM_MODEL", "TEMPERATURE", "TOP_K", "K", "N_RUNS", "SEED"]:
        if k in globals():
            cfg[k] = globals()[k]
    cfg["note"] = "Config dict não encontrado explicitamente; extração best-effort de globals."
    cfg_name = cfg_name or "best_effort_from_globals"

config_path = os.path.join(OUT_DIR, f"{PREFIX}_experiment_config.json")
_write_json(config_path, {"source_var": cfg_name, "config": cfg})

# -----------------------------
# 2) Outputs do agente
# -----------------------------
out_name, out_obj = _pick_first_existing_var(CAND_AGENT_OUTPUTS)
if out_obj is None:
    for k in _search_globals_for_candidates(PATTERN_OUTPUTS):
        v = globals().get(k)
        if isinstance(v, list) and len(v) > 0 and all(isinstance(i, dict) for i in v[:5]):
            out_name, out_obj = k, v
            break
        if _is_df(v) and len(v) > 0:
            out_name, out_obj = k, v
            break

out_rows = _as_records(out_obj) or []

def _normalize_output_row(r: Dict[str, Any]) -> Dict[str, Any]:
    rr = dict(r)
    keymap = [
        (["query", "question", "prompt"], "query"),
        (["retriever", "retriever_name", "retriever_type"], "retriever"),
        (["retrieved_docs", "docs", "contexts", "evidence", "citations", "sources"], "retrieved_docs"),
        (["llm_answer", "answer", "response", "final_answer"], "llm_answer"),
        (["run_id", "run", "trial", "iteration"], "run_id"),
        (["timestamp", "ts", "time"], "timestamp"),
    ]
    norm = {}
    for aliases, target in keymap:
        for a in aliases:
            if a in rr and rr[a] is not None:
                norm[target] = rr[a]
                break
    norm["_raw"] = rr
    return norm

out_rows_norm = [_normalize_output_row(r) for r in out_rows]
outputs_path = os.path.join(OUT_DIR, f"{PREFIX}_agent_outputs.jsonl")
n_out = _write_jsonl(outputs_path, out_rows_norm)

# -----------------------------
# 3) Métricas do agente
# -----------------------------
met_name, met_obj = _pick_first_existing_var(CAND_AGENT_METRICS)
if met_obj is None:
    for k in _search_globals_for_candidates(PATTERN_METRICS):
        v = globals().get(k)
        if _is_df(v) and len(v) > 0:
            met_name, met_obj = k, v
            break

if _is_df(met_obj):
    metrics_df = met_obj.copy()
elif isinstance(met_obj, dict):
    metrics_df = pd.DataFrame([met_obj])
elif isinstance(met_obj, list) and met_obj and all(isinstance(i, dict) for i in met_obj):
    metrics_df = pd.DataFrame(met_obj)
else:
    # Derivadas mínimas dos outputs
    def _count_docs(x: Any) -> int:
        if x is None: return 0
        if isinstance(x, list): return len(x)
        if isinstance(x, dict): return len(x)
        return 1
    rows = []
    for r in out_rows_norm:
        rows.append({
            "retriever": r.get("retriever"),
            "run_id": r.get("run_id"),
            "query": r.get("query"),
            "n_retrieved_docs": _count_docs(r.get("retrieved_docs")),
            "answer_len_chars": len(str(r.get("llm_answer",""))),
        })
    metrics_df = pd.DataFrame(rows)
    met_name = met_name or "derived_from_agent_outputs"

metrics_path = os.path.join(OUT_DIR, f"{PREFIX}_agent_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

# -----------------------------
# 4) Logs
# -----------------------------
log_name, log_obj = _pick_first_existing_var(CAND_LOGS)
if log_obj is None:
    for k in _search_globals_for_candidates(PATTERN_LOGS):
        v = globals().get(k)
        if isinstance(v, list) and len(v) > 0 and all(isinstance(i, dict) for i in v[:5]):
            log_name, log_obj = k, v
            break
        if _is_df(v) and len(v) > 0:
            log_name, log_obj = k, v
            break

log_rows = _as_records(log_obj) or []
logs_path = os.path.join(OUT_DIR, f"{PREFIX}_experiment_logs.jsonl")
n_logs = _write_jsonl(logs_path, log_rows)

# -----------------------------
# 5) Manifest
# -----------------------------
manifest = {
    "generated_at_utc": dt.datetime.utcnow().isoformat() + "Z",
    "out_dir": os.path.abspath(OUT_DIR),
    "files": [],
    "source_vars": {"config": cfg_name, "agent_outputs": out_name, "agent_metrics": met_name, "logs": log_name},
    "counts": {"agent_outputs_rows": n_out, "logs_rows": n_logs, "metrics_rows": int(len(metrics_df))},
    "notes": []
}

for p in [config_path, outputs_path, metrics_path, logs_path]:
    if os.path.exists(p):
        manifest["files"].append({"path": os.path.abspath(p), "size_bytes": os.path.getsize(p), "sha256": _sha256_file(p)})
    else:
        manifest["notes"].append(f"Missing expected file: {p}")

if n_out == 0:
    manifest["notes"].append("agent_outputs.jsonl saiu com 0 linhas. Ajuste CAND_AGENT_OUTPUTS para o nome da sua variável.")
if int(len(metrics_df)) == 0:
    manifest["notes"].append("agent_metrics.csv saiu com 0 linhas. Ajuste CAND_AGENT_METRICS.")
if n_logs == 0:
    manifest["notes"].append("experiment_logs.jsonl saiu com 0 linhas. Ajuste CAND_LOGS.")

manifest_path = os.path.join(OUT_DIR, f"{PREFIX}_export_manifest.json")
_write_json(manifest_path, manifest)

print("✅ PIPE 3 export pack (na raiz) concluído.\n")
print("Arquivos gerados:")
for f in manifest["files"]:
    print(f" - {f['path']}  ({f['size_bytes']} bytes)")
print("\nContagens:", manifest["counts"])
if manifest["notes"]:
    print("\n⚠️ Avisos:")
    for n in manifest["notes"]:
        print(" -", n)


In [ ]:
# ============================================================
# ✅ PIPE 3 — ZIP + DOWNLOAD AUTOMÁTICO (NA RAIZ) — ONE CELL
# Objetivo: criar ./pipe3_export_pack.zip e disparar download automaticamente (Colab).
#
# Funciona em Google Colab.
# Em Jupyter local: cria o ZIP e informa o caminho.
# ============================================================

import os, json, zipfile
from pathlib import Path

# Detecta Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ROOT = Path(".").resolve()
ZIP_PATH = ROOT / "pipe3_export_pack.zip"
MANIFEST_PATH = ROOT / "pipe3_export_manifest.json"

assert MANIFEST_PATH.exists(), (
    "❌ Manifest não encontrado na raiz. Rode antes a célula 'PIPE 3 — EXPORT PACK (NA RAIZ)'.\n"
    f"Esperado: {MANIFEST_PATH}"
)

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
paths = [f["path"] for f in manifest.get("files", []) if "path" in f and os.path.exists(f["path"])]

# fallback
fallback = [
    ROOT / "pipe3_experiment_config.json",
    ROOT / "pipe3_agent_outputs.jsonl",
    ROOT / "pipe3_agent_metrics.csv",
    ROOT / "pipe3_experiment_logs.jsonl",
    ROOT / "pipe3_export_manifest.json",
]
for p in fallback:
    if p.exists() and str(p) not in paths:
        paths.append(str(p))

assert paths, "❌ Nenhum arquivo encontrado para zipar. Verifique a exportação."

# cria zip
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, arcname=os.path.basename(p))

print(f"📦 ZIP pronto: {ZIP_PATH}")
print("Conteúdo:")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for name in z.namelist():
        print(" -", name)

# download automático no Colab
if IN_COLAB:
    print("\n⬇️ Iniciando download automático…")
    files.download(str(ZIP_PATH))
else:
    print("\nℹ️ Não estou em ambiente Colab. Faça o download manual do ZIP no diretório atual.")


In [ ]:
# ============================================================
# ✅ PIPE 3 — AUTO-DISCOVER LOGS + EXPORT (NA RAIZ) — ONE CELL
# - Procura automaticamente variáveis de log no notebook (globals)
# - Mostra candidatos (nome, tipo, tamanho)
# - Exporta o melhor candidato para ./pipe3_experiment_logs.jsonl
# - Atualiza ./pipe3_export_manifest.json
# ============================================================

import os, re, json, hashlib
from pathlib import Path
import pandas as pd

ROOT = Path(".").resolve()
LOGS_PATH = ROOT / "pipe3_experiment_logs.jsonl"
MANIFEST_PATH = ROOT / "pipe3_export_manifest.json"

def _json_safe(x):
    try:
        json.dumps(x)
        return x
    except Exception:
        if isinstance(x, pd.DataFrame):
            return x.to_dict(orient="records")
        if isinstance(x, (set, tuple)):
            return list(x)
        return str(x)

def _write_jsonl(path, rows):
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(_json_safe(r), ensure_ascii=False) + "\n")
            n += 1
    return n

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _score_candidate(name, obj):
    """Heurística para escolher logs."""
    score = 0
    # nome "bom"
    if re.search(r"log|trace|event|audit|telemetry|history|record", name, re.I):
        score += 3
    # tipo "bom"
    if isinstance(obj, pd.DataFrame):
        score += 3
    if isinstance(obj, list):
        score += 2
    if isinstance(obj, dict):
        score += 1

    # conteúdo "bom"
    try:
        if isinstance(obj, pd.DataFrame):
            n = len(obj)
            score += min(5, n / 1000)
        elif isinstance(obj, list):
            n = len(obj)
            score += min(5, n / 1000)
            # bônus se parecer list[dict]
            if n > 0 and all(isinstance(x, dict) for x in obj[:5]):
                score += 2
        elif isinstance(obj, dict):
            score += 0.5
    except Exception:
        pass

    # penaliza gigantes óbvios (ex.: embeddings)
    if re.search(r"embed|faiss|index|vector|matrix", name, re.I):
        score -= 4

    return score

def _to_records(obj):
    if obj is None:
        return []
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, list):
        if all(isinstance(i, dict) for i in obj):
            return obj
        return [{"value": _json_safe(i)} for i in obj]
    if isinstance(obj, dict):
        return [obj]
    return [{"value": _json_safe(obj)}]

# 1) coletar candidatos em globals
g = globals()
candidates = []
for k, v in g.items():
    if k.startswith("_"):
        continue
    # heurística de seleção inicial por nome ou tipo
    if re.search(r"log|trace|event|audit|telemetry|history|record", k, re.I) or isinstance(v, (list, dict, pd.DataFrame)):
        # evita coisas muito pequenas irrelevantes
        try:
            size = len(v) if hasattr(v, "__len__") else 1
        except Exception:
            size = 1
        candidates.append((k, v, size, _score_candidate(k, v)))

# ordenar por score desc
candidates = sorted(candidates, key=lambda x: x[3], reverse=True)

# 2) mostrar top 25
print("🔎 Top candidatos a logs (nome | tipo | tamanho | score):")
for (k, v, size, sc) in candidates[:25]:
    print(f" - {k:35} | {type(v).__name__:12} | {size:7} | {sc:.2f}")

# 3) escolher automaticamente o melhor candidato "com cara de log"
best = None
for (k, v, size, sc) in candidates:
    # preferir algo com nome de log e com conteúdo
    if re.search(r"log|trace|event|audit|telemetry|history|record", k, re.I) and size and size > 0 and sc > 3:
        best = (k, v, size, sc)
        break
# fallback: melhor score com conteúdo
if best is None:
    for (k, v, size, sc) in candidates:
        if size and size > 0 and sc > 4:
            best = (k, v, size, sc)
            break

if best is None:
    print("\n❌ Não encontrei nenhuma variável com cara de log e conteúdo > 0.")
    print("➡️ Se você tiver um log em outra variável, me diga o nome dela.")
else:
    k, v, size, sc = best
    rows = _to_records(v)
    n = _write_jsonl(LOGS_PATH, rows)
    print(f"\n✅ Exportado logs de '{k}' -> {LOGS_PATH} (linhas: {n})")

    # 4) atualizar manifest
    if MANIFEST_PATH.exists():
        manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    else:
        manifest = {"files": [], "notes": [], "counts": {}, "source_vars": {}}

    # remove registro anterior do logs, se existir
    manifest["files"] = [f for f in manifest.get("files", []) if not str(f.get("path","")).endswith("pipe3_experiment_logs.jsonl")]

    manifest.setdefault("source_vars", {})
    manifest["source_vars"]["logs"] = k

    manifest.setdefault("counts", {})
    manifest["counts"]["logs_rows"] = n

    manifest["files"].append({
        "path": str(LOGS_PATH),
        "size_bytes": os.path.getsize(LOGS_PATH),
        "sha256": _sha256_file(LOGS_PATH)
    })

    # remove aviso antigo
    notes = manifest.get("notes", [])
    notes = [x for x in notes if "experiment_logs.jsonl saiu com 0 linhas" not in x]
    manifest["notes"] = notes

    MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"✅ Manifest atualizado: {MANIFEST_PATH}")


In [ ]:
# ============================================================
# ✅ PIPE 3 — ONE CELL (FIXED): EXPORT TUDO + ZIP + DOWNLOAD (NA RAIZ)
# Correção: evita "dictionary changed size during iteration" fazendo snapshot de globals().
#
# Exporta (na raiz):
# - ./pipe3_experiment_config.json
# - ./pipe3_agent_metrics.csv
# - ./pipe3_agent_outputs_aggregated.jsonl
# - ./pipe3_runs.jsonl                (se existir)
# - ./pipe3_retrieval_rankings.jsonl   (se existir)
# - ./pipe3_experiment_logs.jsonl      (se existir)
# - ./pipe3_export_manifest.json
# - ./pipe3_export_pack.zip  (+ download automático no Colab)
# ============================================================

import os, re, json, zipfile, hashlib
from pathlib import Path
import pandas as pd
import datetime as dt
from typing import Any, Dict, List, Optional, Tuple

ROOT = Path(".").resolve()
PREFIX = "pipe3"

# ---------- Colab download
try:
    from google.colab import files as colab_files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

NOW_UTC = dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")

# ============================================================
# Helpers
# ============================================================
def _json_safe(obj: Any) -> Any:
    try:
        json.dumps(obj)
        return obj
    except Exception:
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict(orient="records")
        if isinstance(obj, (set, tuple)):
            return list(obj)
        if isinstance(obj, bytes):
            return obj.decode("utf-8", errors="ignore")
        return str(obj)

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _write_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(_json_safe(data), ensure_ascii=False, indent=2), encoding="utf-8")

def _write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> int:
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(_json_safe(r), ensure_ascii=False) + "\n")
            n += 1
    return n

def _is_df(x: Any) -> bool:
    return isinstance(x, pd.DataFrame)

def _to_records(x: Any) -> List[Dict[str, Any]]:
    if x is None:
        return []
    if _is_df(x):
        return x.to_dict(orient="records")
    if isinstance(x, list):
        if all(isinstance(i, dict) for i in x):
            return x
        return [{"value": _json_safe(i)} for i in x]
    if isinstance(x, dict):
        return [x]
    return [{"value": _json_safe(x)}]

def _len_safe(x: Any) -> int:
    try:
        return len(x)  # type: ignore
    except Exception:
        return 1

def _looks_like_run_record(d: Dict[str, Any]) -> bool:
    keys = set(map(lambda s: str(s).lower(), d.keys()))
    has_q = any(k in keys for k in ["query", "question", "prompt"])
    has_retr = any(k in keys for k in ["retrieved_docs", "docs", "contexts", "evidence", "citations", "sources", "topk", "ranking"])
    has_ans = any(k in keys for k in ["llm_answer", "answer", "response", "final_answer", "generated_answer"])
    return has_q and (has_retr or has_ans)

def _normalize_run_row(r: Dict[str, Any]) -> Dict[str, Any]:
    rr = dict(r)

    def pick(aliases: List[str]) -> Any:
        # direct
        for a in aliases:
            if a in rr and rr[a] is not None:
                return rr[a]
        # case-insensitive
        for a in aliases:
            for k in rr.keys():
                if str(k).lower() == a.lower():
                    return rr[k]
        return None

    return {
        "query": pick(["query", "question", "prompt"]),
        "retriever": pick(["retriever", "retriever_name", "retriever_type", "index_name"]),
        "retrieved_docs": pick(["retrieved_docs", "docs", "contexts", "evidence", "citations", "sources", "topk", "ranking"]),
        "llm_answer": pick(["llm_answer", "answer", "response", "final_answer", "generated_answer"]),
        "run_id": pick(["run_id", "run", "trial", "iteration"]),
        "timestamp": pick(["timestamp", "ts", "time"]),
        "_raw": rr
    }

def _score_var(name: str, obj: Any, want: str) -> float:
    score = 0.0
    n = name.lower()

    # penaliza vetores/índices
    if re.search(r"embed|faiss|index|vector|matrix|umap|hdbscan|cluster", n):
        score -= 5

    if want == "config":
        if re.search(r"config|cfg|setting|param|args", n): score += 3
        if isinstance(obj, dict): score += 4
        if isinstance(obj, dict): score += min(2, len(obj)/20)

    if want == "metrics":
        if re.search(r"metric|score|eval|report|summary", n): score += 3
        if isinstance(obj, pd.DataFrame): score += 5
        if isinstance(obj, (list, dict)): score += 1
        score += min(2, _len_safe(obj)/1000)

    if want == "outputs_agg":
        if re.search(r"result|final|summary|agg|outputs|answers", n): score += 2
        if isinstance(obj, pd.DataFrame): score += 3
        if isinstance(obj, list) and len(obj) > 0 and all(isinstance(i, dict) for i in obj[:5]): score += 4
        if isinstance(obj, dict): score += 1
        score += min(2, _len_safe(obj)/500)

    if want == "runs":
        if re.search(r"run|trace|turn|dialog|qa|query|prompt|retriev", n): score += 3
        if isinstance(obj, list) and len(obj) > 0 and all(isinstance(i, dict) for i in obj[:5]):
            if any(_looks_like_run_record(i) for i in obj[:10]): score += 6
            else: score += 2
        if isinstance(obj, pd.DataFrame) and len(obj) > 0:
            cols = set(map(str.lower, obj.columns))
            if ("query" in cols) or ("question" in cols) or ("prompt" in cols): score += 5
            if any(c in cols for c in ["retrieved_docs","docs","contexts","evidence","citations","sources","answer","response","llm_answer"]):
                score += 2
        score += min(2, _len_safe(obj)/200)

    if want == "logs":
        if re.search(r"log|trace|event|audit|telemetry|history", n): score += 4
        if isinstance(obj, list) and len(obj) > 0 and all(isinstance(i, dict) for i in obj[:5]): score += 4
        if isinstance(obj, pd.DataFrame) and len(obj) > 0: score += 4
        score += min(2, _len_safe(obj)/2000)

    return score

def _pick_best(globs: Dict[str, Any], want: str) -> Tuple[Optional[str], Any, float]:
    best_name, best_obj, best_sc = None, None, float("-inf")
    for k, v in list(globs.items()):  # ✅ list() pra congelar iteração
        if str(k).startswith("_"):
            continue
        sc = _score_var(str(k), v, want)
        if sc > best_sc:
            best_name, best_obj, best_sc = str(k), v, sc
    return best_name, best_obj, best_sc

def _df_from_any(obj: Any) -> Optional[pd.DataFrame]:
    if isinstance(obj, pd.DataFrame):
        return obj.copy()
    if isinstance(obj, list) and obj and all(isinstance(i, dict) for i in obj):
        return pd.DataFrame(obj)
    if isinstance(obj, dict):
        try:
            return pd.DataFrame([obj])
        except Exception:
            return None
    return None

# ============================================================
# 0) SNAPSHOT GLOBALS (FIX)
# ============================================================
G = dict(globals())  # ✅ snapshot imutável para varreduras
notes: List[str] = []

# ============================================================
# 1) AUTO-DISCOVERY
# ============================================================
cfg_name, cfg_obj, cfg_sc = _pick_best(G, "config")
met_name, met_obj, met_sc = _pick_best(G, "metrics")
agg_name, agg_obj, agg_sc = _pick_best(G, "outputs_agg")
runs_name, runs_obj, runs_sc = _pick_best(G, "runs")
logs_name, logs_obj, logs_sc = _pick_best(G, "logs")

source_vars = {
    "config": cfg_name,
    "metrics": met_name,
    "outputs_aggregated": agg_name,
    "runs": runs_name,
    "logs": logs_name
}
scores = {
    "config_score": cfg_sc,
    "metrics_score": met_sc,
    "outputs_agg_score": agg_sc,
    "runs_score": runs_sc,
    "logs_score": logs_sc
}

# ============================================================
# 2) EXPORT CONFIG (best-effort)
# ============================================================
config = None
if isinstance(cfg_obj, dict):
    config = cfg_obj
else:
    df_cfg = _df_from_any(cfg_obj)
    if df_cfg is not None and len(df_cfg) == 1:
        config = df_cfg.iloc[0].to_dict()

if config is None:
    config = {}
    for k in ["RUN_ID", "SEED", "MODEL", "LLM_MODEL", "TEMPERATURE", "TOP_K", "K", "N_RUNS", "PROMPTS", "RETRIEVERS"]:
        if k in G:
            config[k] = _json_safe(G[k])
    notes.append("Config não encontrada como dict explícito; gerada por best-effort de snapshot globals.")

CONFIG_PATH = ROOT / f"{PREFIX}_experiment_config.json"
_write_json(CONFIG_PATH, {"generated_at_utc": NOW_UTC, "source_var": cfg_name, "score": cfg_sc, "config": config})

# ============================================================
# 3) EXPORT METRICS
# ============================================================
metrics_df = _df_from_any(met_obj)
if metrics_df is None or len(metrics_df) == 0:
    metrics_df = _df_from_any(agg_obj)
    if metrics_df is None:
        metrics_df = pd.DataFrame([])
        notes.append("Não foi possível inferir metrics_df (vazio).")
    else:
        notes.append("metrics_df não encontrado; usei outputs_aggregated como fallback.")

METRICS_PATH = ROOT / f"{PREFIX}_agent_metrics.csv"
metrics_df.to_csv(METRICS_PATH, index=False)

# ============================================================
# 4) EXPORT OUTPUTS AGREGADOS
# ============================================================
agg_rows = _to_records(agg_obj)
AGG_OUT_PATH = ROOT / f"{PREFIX}_agent_outputs_aggregated.jsonl"
n_agg = _write_jsonl(AGG_OUT_PATH, [{"_raw": r} for r in agg_rows]) if agg_rows else 0
if n_agg == 0:
    notes.append("outputs_aggregated vazio (0 linhas).")

# ============================================================
# 5) FIND + EXPORT RUNS POR QUERY (melhor esforço)
# ============================================================
def _pick_best_runs_candidate(globs: Dict[str, Any]) -> Tuple[Optional[str], Any, float]:
    best = (None, None, float("-inf"))
    for k, v in list(globs.items()):
        if str(k).startswith("_"):
            continue
        # forte candidato: list[dict] com chaves de query
        if isinstance(v, list) and len(v) > 0 and all(isinstance(i, dict) for i in v[:5]):
            if any(_looks_like_run_record(i) for i in v[:10]):
                sc = _score_var(str(k), v, "runs")
                if sc > best[2]:
                    best = (str(k), v, sc)
        # candidato DF com coluna query/prompt
        if isinstance(v, pd.DataFrame) and len(v) > 0:
            cols = set(map(str.lower, v.columns))
            if ("query" in cols) or ("question" in cols) or ("prompt" in cols):
                sc = _score_var(str(k), v, "runs")
                if sc > best[2]:
                    best = (str(k), v, sc)
    return best  # type: ignore

# se o selecionado não tem cara de runs, tenta achar outro
better_runs = _pick_best_runs_candidate(G)
if better_runs[0] is not None:
    runs_name, runs_obj, runs_sc = better_runs
    source_vars["runs"] = runs_name
    scores["runs_score"] = runs_sc

runs_records = _to_records(runs_obj)
runs_norm = [_normalize_run_row(r) for r in runs_records if isinstance(r, dict)]
RUNS_PATH = ROOT / f"{PREFIX}_runs.jsonl"
n_runs = _write_jsonl(RUNS_PATH, runs_norm) if runs_norm else 0
if n_runs == 0:
    notes.append("Não encontrei runs por query (query + retrieved_docs/answer). Isso pode ser esperado se o PIPE 3 só gerou métricas agregadas.")

# ============================================================
# 6) EXPORT RANKINGS (a partir de runs_norm)
# ============================================================
rank_rows = []
if n_runs > 0:
    for rr in runs_norm:
        q = rr.get("query")
        retr = rr.get("retriever")
        rd = rr.get("retrieved_docs")
        if isinstance(rd, list) and len(rd) > 0:
            for i, doc in enumerate(rd):
                rank_rows.append({"query": q, "retriever": retr, "rank": i+1, "doc": _json_safe(doc)})
        elif isinstance(rd, dict) and len(rd) > 0:
            for i, (k, v) in enumerate(rd.items()):
                rank_rows.append({"query": q, "retriever": retr, "rank": i+1, "doc_id": k, "value": _json_safe(v)})

RANK_PATH = ROOT / f"{PREFIX}_retrieval_rankings.jsonl"
n_rank = _write_jsonl(RANK_PATH, rank_rows) if rank_rows else 0
if n_rank == 0:
    notes.append("Não consegui gerar retrieval_rankings (retrieved_docs ausente ou não estruturado).")

# ============================================================
# 7) FIND + EXPORT LOGS/TRACES
# ============================================================
def _pick_best_logs_candidate(globs: Dict[str, Any]) -> Tuple[Optional[str], Any, float]:
    best = (None, None, float("-inf"))
    for k, v in list(globs.items()):
        if str(k).startswith("_"):
            continue
        if re.search(r"log|trace|event|audit|telemetry|history", str(k), re.I):
            if _len_safe(v) > 0:
                sc = _score_var(str(k), v, "logs")
                if sc > best[2]:
                    best = (str(k), v, sc)
    return best  # type: ignore

better_logs = _pick_best_logs_candidate(G)
if better_logs[0] is not None:
    logs_name, logs_obj, logs_sc = better_logs
    source_vars["logs"] = logs_name
    scores["logs_score"] = logs_sc

log_rows = _to_records(logs_obj)
LOGS_PATH = ROOT / f"{PREFIX}_experiment_logs.jsonl"
n_logs = _write_jsonl(LOGS_PATH, log_rows) if log_rows else 0
if n_logs == 0:
    notes.append("Logs/traces não encontrados em memória (0 linhas). Se você não logou em variável, isso é esperado.")

# ============================================================
# 8) MANIFEST
# ============================================================
def _file_entry(p: Path) -> Dict[str, Any]:
    return {"path": str(p), "size_bytes": p.stat().st_size, "sha256": _sha256_file(p)}

manifest_files = []
for p in [CONFIG_PATH, METRICS_PATH, AGG_OUT_PATH]:
    if p.exists() and p.stat().st_size > 0:
        manifest_files.append(_file_entry(p))
for p in [RUNS_PATH, RANK_PATH, LOGS_PATH]:
    if p.exists() and p.stat().st_size > 0:
        manifest_files.append(_file_entry(p))

MANIFEST_PATH = ROOT / f"{PREFIX}_export_manifest.json"
manifest = {
    "generated_at_utc": NOW_UTC,
    "root_dir": str(ROOT),
    "source_vars": source_vars,
    "candidate_scores": scores,
    "counts": {
        "metrics_rows": int(len(metrics_df)) if metrics_df is not None else 0,
        "outputs_aggregated_rows": int(n_agg),
        "runs_rows": int(n_runs),
        "rank_rows": int(n_rank),
        "logs_rows": int(n_logs)
    },
    "files": manifest_files,
    "notes": notes
}
_write_json(MANIFEST_PATH, manifest)

# ============================================================
# 9) ZIP + DOWNLOAD
# ============================================================
ZIP_PATH = ROOT / f"{PREFIX}_export_pack.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in manifest_files:
        z.write(f["path"], arcname=os.path.basename(f["path"]))
    z.write(str(MANIFEST_PATH), arcname=MANIFEST_PATH.name)

# ============================================================
# 10) SUMMARY + DOWNLOAD
# ============================================================
print("✅ PIPE 3 — Export completo (na raiz) concluído.\n")
print("📌 Variáveis detectadas (source_vars):")
for k, v in source_vars.items():
    print(f" - {k}: {v}")

print("\n📦 Arquivos exportados:")
for f in manifest_files:
    print(f" - {f['path']}  ({f['size_bytes']} bytes)")

print("\n📊 Contagens:")
for k, v in manifest["counts"].items():
    print(f" - {k}: {v}")

if notes:
    print("\n⚠️ Observações:")
    for n in notes:
        print(f" - {n}")

print(f"\n🧾 Manifest: {MANIFEST_PATH}")
print(f"🗜️ ZIP: {ZIP_PATH}")

if IN_COLAB:
    print("\n⬇️ Iniciando download automático do ZIP…")
    colab_files.download(str(ZIP_PATH))
else:
    print("\nℹ️ Não detectei Google Colab. O ZIP foi criado na raiz; faça o download manual.")


In [ ]:
# ============================================================
# ✅ PIPE 3 — CÉLULA DEFINITIVA (RAIZ): EXPORT CORRETO + RUNS + LOGS + ZIP + DOWNLOAD
#
# ✅ Você vai substituir TODAS as células anteriores por esta.
# ✅ Não depende de /mnt/data, não cria pastas: TUDO NA RAIZ (.)
# ✅ Lê diretamente os artefatos reais do RUN_DIR (outputs/reports/logs/figures)
# ✅ Gera "runs" e "rankings" CORRETOS a partir dos agent_*.json (por query)
# ✅ Zipa e faz download automático no Colab
#
# Saídas (todas na raiz):
# - ./pipe3_experiment_config.json
# - ./pipe3_key_reports/...(copiados para raiz como prefixo "pipe3_")
# - ./pipe3_agent_metrics.csv                     (derivado do eval_metrics_v2...)
# - ./pipe3_agent_outputs_aggregated.jsonl        (agregado do eval_metrics_v2...)
# - ./pipe3_runs.jsonl                            (POR QUERY, extraído dos agent_*.json)
# - ./pipe3_retrieval_rankings.jsonl              (POR QUERY, se houver retrieved_docs/topk em agent_*.json)
# - ./pipe3_experiment_logs.jsonl                 (convertido de logs/run.log, se existir)
# - ./pipe3_export_manifest.json                  (com hashes/contagens)
# - ./pipe3_export_pack.zip                       (+ download)
# ============================================================

import os, re, json, zipfile, hashlib
from pathlib import Path
import pandas as pd
import datetime as dt
from typing import Any, Dict, List, Optional, Tuple

# ---------------------------
# Colab download
# ---------------------------
try:
    from google.colab import files as colab_files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

ROOT = Path(".").resolve()
PREFIX = "pipe3"
NOW_UTC = dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")

# ---------------------------
# Helpers
# ---------------------------
def _json_safe(obj: Any) -> Any:
    try:
        json.dumps(obj)
        return obj
    except Exception:
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict(orient="records")
        if isinstance(obj, (set, tuple)):
            return list(obj)
        return str(obj)

def _write_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(_json_safe(data), ensure_ascii=False, indent=2), encoding="utf-8")

def _write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> int:
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(_json_safe(r), ensure_ascii=False) + "\n")
            n += 1
    return n

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _file_entry(p: Path) -> Dict[str, Any]:
    return {"path": str(p), "size_bytes": p.stat().st_size, "sha256": _sha256_file(p)}

def _safe_read_json(path: Path) -> Any:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def _as_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def _pick(d: Dict[str, Any], aliases: List[str]) -> Any:
    # direct
    for a in aliases:
        if a in d and d[a] is not None:
            return d[a]
    # case-insensitive
    low = {str(k).lower(): k for k in d.keys()}
    for a in aliases:
        k = low.get(a.lower())
        if k is not None and d[k] is not None:
            return d[k]
    return None

def _normalize_run(rec: Dict[str, Any], fallback_retriever: Optional[str]=None, fallback_mode: Optional[str]=None) -> Dict[str, Any]:
    q = _pick(rec, ["query","question","prompt"])
    ans = _pick(rec, ["llm_answer","answer","response","final_answer","generated_answer"])
    retrieved = _pick(rec, ["retrieved_docs","docs","contexts","evidence","sources","topk","ranking","retrieved_topk"])
    cited = _pick(rec, ["cited_doc_ids","cited_docs","supporting_doc_ids","supporting_ids","used_doc_ids"])
    rid = _pick(rec, ["run_id","trial","iteration","idx"])
    ts = _pick(rec, ["timestamp","ts","time"])

    out = {
        "query": q,
        "retriever": _pick(rec, ["retriever","retriever_name","retriever_type"]) or fallback_retriever,
        "mode": _pick(rec, ["mode","condition","cond"]) or fallback_mode,
        "run_id": rid,
        "timestamp": ts,
        "retrieved_docs": retrieved,
        "cited_doc_ids": cited,
        "llm_answer": ans,
        "_raw": rec
    }
    return out

def _infer_retriever_mode_from_filename(fn: str) -> Tuple[Optional[str], Optional[str]]:
    # expected: agent_{retriever}_{mode}.json
    m = re.match(r"agent_(R\d+)_(.+)\.json$", fn)
    if m:
        return m.group(1), m.group(2)
    return None, None

# ---------------------------
# 0) Locate RUN_DIR
# ---------------------------
assert "RUN_DIR" in globals(), "RUN_DIR não encontrado no notebook. Rode as células do PIPE 3 antes desta."
RUN_DIR = Path(globals()["RUN_DIR"])
OUT_DIR = RUN_DIR / "outputs"
REP_DIR = RUN_DIR / "reports"
LOG_DIR = RUN_DIR / "logs"
FIG_DIR = RUN_DIR / "figures"

assert RUN_DIR.exists(), f"RUN_DIR não existe: {RUN_DIR}"
assert OUT_DIR.exists(), f"outputs não existe: {OUT_DIR}"
assert REP_DIR.exists(), f"reports não existe: {REP_DIR}"

notes: List[str] = []
exported: List[Path] = []

# ---------------------------
# 1) Export CONFIG (mais completa possível)
# ---------------------------
cfg = {}
# tenta aproveitar 'cfg' se existir
if "cfg" in globals() and isinstance(globals()["cfg"], dict):
    cfg = dict(globals()["cfg"])
else:
    # best-effort: coleta valores comuns
    for k in ["RUN_ID","SEED","MODEL","LLM_MODEL","TEMPERATURE","TOP_K","K","N_RUNS","TEST_QUERIES","DOMAIN_TERMS","DESC_CANDIDATES"]:
        if k in globals():
            cfg[k] = _json_safe(globals()[k])

cfg.update({
    "RUN_DIR": str(RUN_DIR),
    "outputs_dir": str(OUT_DIR),
    "reports_dir": str(REP_DIR),
    "logs_dir": str(LOG_DIR),
    "figures_dir": str(FIG_DIR),
})

CONFIG_PATH = ROOT / f"{PREFIX}_experiment_config.json"
_write_json(CONFIG_PATH, {"generated_at_utc": NOW_UTC, "config": cfg})
exported.append(CONFIG_PATH)

# ---------------------------
# 2) Copy KEY REPORT files from reports/ to root (prefix pipe3_)
# ---------------------------
# (estes são os arquivos que sustentam Results/Tables do artigo)
KEY_REPORTS = [
    "eval_metrics_v2_with_retrieved_topk.csv",
    "H1_deltas_v2.csv",
    "H3_deltas_v2.csv",
    "summary_metrics_v2.json",
    "H3_M1_CGSA.csv",
    "H3_M3_CEST.csv",
    "H3_semantic_summary.csv",
    # também copiamos v1 se existirem
    "eval_metrics.csv",
    "H1_deltas.csv",
    "H2_levels.csv",
    "H3_deltas.csv",
    "summary_metrics.json",
]
for name in KEY_REPORTS:
    src = REP_DIR / name
    if src.exists() and src.is_file():
        dst = ROOT / f"{PREFIX}_{name}"
        dst.write_bytes(src.read_bytes())
        exported.append(dst)

# ---------------------------
# 3) Build aggregated metrics + outputs_aggregated from eval_metrics_v2...
# ---------------------------
eval_v2_path = ROOT / f"{PREFIX}_eval_metrics_v2_with_retrieved_topk.csv"
if not eval_v2_path.exists():
    # fallback: usa reports direto
    src = REP_DIR / "eval_metrics_v2_with_retrieved_topk.csv"
    if src.exists():
        eval_v2_path.write_bytes(src.read_bytes())
        exported.append(eval_v2_path)

if eval_v2_path.exists():
    eval2 = pd.read_csv(eval_v2_path)
    # Tabela agregada principal (o que você já viu no notebook)
    keep_cols = [c for c in [
        "retriever","mode","gaps",
        "ICR","UCR",
        "domain_hit_rate_abstract_retrieved_topk",
        "domain_hit_rate_abstract_cited",
        "title_missing_rate_retrieved_topk",
        "abstract_missing_rate_retrieved_topk",
        "cited_total","claims_total"
    ] if c in eval2.columns]

    if not keep_cols:
        notes.append("eval_metrics_v2_with_retrieved_topk.csv encontrado, mas não contém colunas esperadas. Mantive como está.")
        metrics_df = eval2.copy()
    else:
        metrics_df = eval2[keep_cols].copy().sort_values(["retriever","mode"]).reset_index(drop=True)

    METRICS_CSV = ROOT / f"{PREFIX}_agent_metrics.csv"
    metrics_df.to_csv(METRICS_CSV, index=False)
    exported.append(METRICS_CSV)

    # outputs_aggregated.jsonl (mesmo conteúdo em JSONL)
    AGG_JSONL = ROOT / f"{PREFIX}_agent_outputs_aggregated.jsonl"
    n_agg = _write_jsonl(AGG_JSONL, [{"_raw": r} for r in metrics_df.to_dict(orient="records")])
    exported.append(AGG_JSONL)
else:
    notes.append("Não encontrei eval_metrics_v2_with_retrieved_topk.csv — sem isso, o PIPE 3 fica incompleto para Results.")

# ---------------------------
# 4) Build RUNS.jsonl (POR QUERY) a partir de outputs/agent_*.json
# ---------------------------
agent_files = sorted(list(OUT_DIR.glob("agent_*.json")))
if not agent_files:
    notes.append("Nenhum agent_*.json encontrado em outputs/. Sem isso, não dá para gerar runs por query.")
    RUNS_JSONL = ROOT / f"{PREFIX}_runs.jsonl"
    _write_jsonl(RUNS_JSONL, [])
    exported.append(RUNS_JSONL)
else:
    runs_rows: List[Dict[str, Any]] = []
    rank_rows: List[Dict[str, Any]] = []
    for af in agent_files:
        retr, mode = _infer_retriever_mode_from_filename(af.name)
        payload = _safe_read_json(af)
        if payload is None:
            notes.append(f"Falha ao ler JSON: {af.name}")
            continue

        # alguns formatos possíveis:
        # - list[dict] (um por query)
        # - dict com chave 'rows'/'results'/'items'
        items = None
        if isinstance(payload, list):
            items = payload
        elif isinstance(payload, dict):
            for k in ["rows","results","items","data","outputs","runs"]:
                if k in payload and isinstance(payload[k], list):
                    items = payload[k]
                    break
            if items is None:
                # se for dict "query -> {...}"
                if all(isinstance(v, dict) for v in payload.values()) and len(payload) > 0:
                    items = list(payload.values())

        if not items:
            # ainda assim guarda o payload bruto como um run único
            runs_rows.append(_normalize_run({"_payload": payload}, retr, mode))
            continue

        for idx, rec in enumerate(items):
            if not isinstance(rec, dict):
                rec = {"value": _json_safe(rec)}
            rnorm = _normalize_run(rec, retr, mode)
            if rnorm.get("run_id") is None:
                rnorm["run_id"] = idx
            runs_rows.append(rnorm)

            # tentar extrair rankings (se existir retrieved_docs/topk)
            rd = rnorm.get("retrieved_docs")
            if isinstance(rd, list):
                for i, doc in enumerate(rd):
                    rank_rows.append({
                        "retriever": retr,
                        "mode": mode,
                        "query": rnorm.get("query"),
                        "run_id": rnorm.get("run_id"),
                        "rank": i+1,
                        "doc": _json_safe(doc),
                    })
            elif isinstance(rd, dict):
                # dict doc_id->score
                for i, (k,v) in enumerate(rd.items()):
                    rank_rows.append({
                        "retriever": retr,
                        "mode": mode,
                        "query": rnorm.get("query"),
                        "run_id": rnorm.get("run_id"),
                        "rank": i+1,
                        "doc_id": k,
                        "value": _json_safe(v),
                    })

    RUNS_JSONL = ROOT / f"{PREFIX}_runs.jsonl"
    n_runs = _write_jsonl(RUNS_JSONL, runs_rows)
    exported.append(RUNS_JSONL)

    RANK_JSONL = ROOT / f"{PREFIX}_retrieval_rankings.jsonl"
    n_rank = _write_jsonl(RANK_JSONL, rank_rows) if rank_rows else 0
    exported.append(RANK_JSONL)

    if n_runs == 0:
        notes.append("runs.jsonl saiu vazio. Verifique o formato interno dos agent_*.json.")
    if n_rank == 0:
        notes.append("rankings.jsonl saiu vazio (retrieved_docs/topk não presentes nos agent_*.json).")

# ---------------------------
# 5) Export LOGS corretamente (logs/run.log -> logs.jsonl)
# ---------------------------
LOG_TXT = LOG_DIR / "run.log"
LOG_JSONL = ROOT / f"{PREFIX}_experiment_logs.jsonl"

log_rows = []
if LOG_TXT.exists():
    # cada linha vira um evento
    for ln in LOG_TXT.read_text(encoding="utf-8", errors="ignore").splitlines():
        # tenta extrair timestamp padrão: [ISO] msg
        m = re.match(r'^\[(.+?)\]\s+(.*)$', ln.strip())
        if m:
            log_rows.append({"ts": m.group(1), "msg": m.group(2), "_raw": ln})
        else:
            log_rows.append({"msg": ln, "_raw": ln})
    _write_jsonl(LOG_JSONL, log_rows)
    exported.append(LOG_JSONL)
else:
    # cria arquivo vazio explícito (não quebra zip)
    _write_jsonl(LOG_JSONL, [])
    exported.append(LOG_JSONL)
    notes.append("logs/run.log não encontrado; exportei logs.jsonl vazio.")

# ---------------------------
# 6) Manifest + ZIP + DOWNLOAD
# ---------------------------
MANIFEST = ROOT / f"{PREFIX}_export_manifest.json"

# só inclui arquivos que existem e não são absurdamente grandes desnecessários
# (ex.: NÃO inclui corpus de 108k docs por acidente)
final_files = []
for p in exported:
    if p.exists() and p.is_file():
        final_files.append(p)

manifest = {
    "generated_at_utc": NOW_UTC,
    "RUN_DIR": str(RUN_DIR),
    "files": [_file_entry(p) for p in final_files],
    "counts": {
        "agent_files": len(agent_files),
        "runs_rows": sum(1 for _ in open(ROOT / f"{PREFIX}_runs.jsonl", "r", encoding="utf-8")) if (ROOT / f"{PREFIX}_runs.jsonl").exists() else 0,
        "rank_rows": sum(1 for _ in open(ROOT / f"{PREFIX}_retrieval_rankings.jsonl", "r", encoding="utf-8")) if (ROOT / f"{PREFIX}_retrieval_rankings.jsonl").exists() else 0,
        "log_rows": len(log_rows),
    },
    "notes": notes
}
_write_json(MANIFEST, manifest)

ZIP_PATH = ROOT / f"{PREFIX}_export_pack.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in final_files:
        z.write(str(p), arcname=p.name)
    z.write(str(MANIFEST), arcname=MANIFEST.name)

# ---------------------------
# Print summary
# ---------------------------
print("✅ PIPE 3 — Export DEFINITIVO concluído (na raiz).")
print(f"RUN_DIR: {RUN_DIR}")
print("\n📦 Arquivos no pacote:")
for p in final_files:
    print(f" - {p.name} ({p.stat().st_size} bytes)")

print("\n📊 Contagens:")
for k,v in manifest["counts"].items():
    print(f" - {k}: {v}")

if notes:
    print("\n⚠️ Notas:")
    for n in notes:
        print(" -", n)

print(f"\n🗜️ ZIP: {ZIP_PATH}")

if IN_COLAB:
    print("\n⬇️ Iniciando download automático…")
    colab_files.download(str(ZIP_PATH))
else:
    print("\nℹ️ Não detectei Colab. Baixe manualmente o ZIP na raiz.")
